In [1]:
import numpy as np
import pandas as pd
import re
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
# from lightgbm.lgb import LGBMRegressor
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr

In [2]:
def train_and_test_predict(models, X_train, y_train, X_test, y_test):
    kf = KFold(n_splits=5, shuffle=True, random_state=101)
    results = {}
    predictions = []  

    for model in models:
        model_name = model.__class__.__name__
        predictions_train = []
        actual_y_train = []

        test_predictions_folds = []
        for train_index, val_index in kf.split(X_train):
            X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
            y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

            model.fit(X_train_fold, y_train_fold)

            y_pred_fold = model.predict(X_val_fold)
            y_pred_fold = np.clip(y_pred_fold, -10, -3.9)
            predictions_train.extend(y_pred_fold)
            actual_y_train.extend(y_val_fold)

            predictions_test_fold = model.predict(X_test)
            predictions_test_fold = np.clip(predictions_test_fold, -10, -3.9)
            test_predictions_folds.append(predictions_test_fold)


        mse_train = mean_squared_error(actual_y_train, predictions_train)
        mae_train = mean_absolute_error(actual_y_train, predictions_train)
        rmse_train = np.sqrt(mse_train)
        r2_train = r2_score(actual_y_train, predictions_train)
        pearson_train, _ = pearsonr(actual_y_train, predictions_train)
        spearman_train, _ = spearmanr(actual_y_train, predictions_train)


        predictions_test_mean = np.mean(test_predictions_folds, axis=0)
        predictions_test_std = np.std(test_predictions_folds, axis=0)

        mse_test = mean_squared_error(y_test, predictions_test_mean)
        mae_test = mean_absolute_error(y_test, predictions_test_mean)
        rmse_test = np.sqrt(mse_test)
        r2_test = r2_score(y_test, predictions_test_mean)
        print(r2_test)
        pearson_test, _ = pearsonr(y_test, predictions_test_mean)
        spearman_test, _ = spearmanr(y_test, predictions_test_mean)
        predictions.append({
            'Model': model_name,
            'Y Train pred': predictions_train,
            'Y Test actual': y_test,
            'Test prediction folds': test_predictions_folds,
            'Test Predictions Mean': predictions_test_mean,
            'Test Predictions Std': predictions_test_std,

        })

        results[model_name] = {
            'Train MSE (5 fold cv)': f"{mse_train:.4f}",
            'Train MAE (5 fold cv)': f"{mae_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train R2 (5 fold cv)': f"{r2_train:.4f}",
            'Train PCC (5 fold cv)': f"{pearson_train:.4f}",
            'Train SCC (5 fold cv)': f"{spearman_train:.4f}",
            'Test MSE': f"{mse_test:.4f}",
            'Test MAE': f"{mae_test:.4f}",
            'Test RMSE': f"{rmse_test:.4f}",
            'Test R2': f"{r2_test:.4f}",
            'Test Pearson Correlation': f"{pearson_test:.4f}",
            'Test Spearman Correlation': f"{spearman_test:.4f}",
        }

    results_df = pd.DataFrame(results).T
    predictions_df = pd.DataFrame(predictions)

    return results_df, predictions_df

In [3]:
#2d RDKit descriptors
df_train = pd.read_csv('features/Descriptors/Train_2d_RDKit_des.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_RDKit_des.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models_2drdkit = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models_2drdkit, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (5568, 217)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 217)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.021650 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 19594
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 156
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.021106 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 19618
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 158
[LightGBM] [Info] Start training from score -5.744959
[LightGBM] [Info] Auto-choosing col-wise multi-threading, 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2067,0.3340,0.4546,0.6681,0.8179,0.7871,0.2121,0.3322,0.4605,0.6664,0.8172,0.7972
DecisionTreeRegressor,0.3111,0.3859,0.5578,0.5003,0.7283,0.7110,0.2314,0.3418,0.4810,0.6360,0.7986,0.7789
RandomForestRegressor,0.2134,0.3406,0.4620,0.6572,0.8107,0.7810,0.2157,0.3327,0.4644,0.6607,0.8130,0.7931
GradientBoostingRegressor,0.2322,0.3591,0.4819,0.6271,0.7951,0.7573,0.2342,0.3563,0.4839,0.6317,0.7985,0.7737
AdaBoostRegressor,0.4149,0.5223,0.6441,0.3337,0.6219,0.5701,0.4023,0.5117,0.6342,0.3673,0.6473,0.6155
XGBRegressor,0.2141,0.3378,0.4627,0.6562,0.8107,0.7836,0.2074,0.3235,0.4554,0.6738,0.8211,0.8026
ExtraTreesRegressor,0.2062,0.3332,0.4541,0.6688,0.8178,0.7890,0.2137,0.3309,0.4622,0.6639,0.8150,0.7951
LinearRegression,0.3712,0.4354,0.6093,0.4038,0.6460,0.6722,0.3504,0.4280,0.5919,0.4489,0.6726,0.6902
KNeighborsRegressor,0.2847,0.3869,0.5336,0.5428,0.7429,0.7137,0.2862,0.3797,0.5350,0.5499,0.7462,0.7399
SVR,0.2469,0.3549,0.4969,0.6034,0.7804,0.7566,0.2515,0.3561,0.5015,0.6043,0.7810,0.7644


In [4]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.946673987951213, -6.876538862848576, -6.04...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.16054630977522, -6.139016132737373, -6.89...","[-7.177746557729078, -6.36758965089367, -6.918...","[0.07699340211630198, 0.160252333875179, 0.044..."
1,DecisionTreeRegressor,"[-7.0, -7.0, -6.244999999999999, -4.68, -5.15,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -4.89, -7.0, -7.0, -5.47, -5.68, -6.85...","[-7.0, -5.5200000000000005, -7.0, -6.286, -5.8...","[0.0, 0.6046155803483733, 0.0, 0.8892153844822..."
2,RandomForestRegressor,"[-6.89426941091, -6.8021600000000015, -6.03710...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.969200000000003, -6.005964051549585, -6.9...","[-6.9762400000000016, -6.173370611039321, -6.9...","[0.012928975210742216, 0.135491226173444, 0.06..."
3,GradientBoostingRegressor,"[-6.74480964262749, -7.007488359209473, -5.554...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.982061266075649, -6.262476550916484, -6.6...","[-7.193598658545653, -6.336573466574402, -6.61...","[0.14825564740271585, 0.20237938780268372, 0.0..."
4,AdaBoostRegressor,"[-5.652894644157253, -6.097628008129341, -5.55...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.143507462686569, -5.590606101139323, -5.5...","[-6.296760345073099, -5.667698216022634, -5.60...","[0.10357787778993623, 0.04308987802258041, 0.0..."
5,XGBRegressor,"[-7.222617, -6.3722773, -6.6379023, -5.1207943...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.180086, -6.507598, -6.92711, -7.2824736, ...","[-7.0340567, -6.3059316, -6.919819, -6.8459597...","[0.10285555, 0.29018506, 0.11030027, 0.2572470..."
6,ExtraTreesRegressor,"[-6.9170500000000015, -6.804599999999999, -6.0...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.9956000000000005, -6.495000000000001, -6....","[-6.994800000000001, -6.418440000000001, -6.95...","[0.004374471396637579, 0.1294908274743822, 0.0..."
7,LinearRegression,"[-8.915444782968027, -6.5657367670600415, -5.4...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-10.0, -6.216474713522705, -6.36512921318304...","[-10.0, -6.052751598166457, -6.257460170481865...","[0.0, 0.13721514598450196, 0.0887254207562553,..."
8,KNeighborsRegressor,"[-6.63, -6.986666666666667, -5.88, -4.89, -4.7...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.63, -7.0, -7.0, -5.383333333333333, -6.74...","[-6.7780000000000005, -6.949333333333334, -6.8...","[0.18126224096595522, 0.10133333333333318, 0.2..."
9,SVR,"[-6.175183663802253, -6.926558525196583, -6.08...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.175107164818723, -6.429851225805583, -6.9...","[-6.216625957531408, -6.502075212404745, -6.94...","[0.02230019151733686, 0.07401240206273532, 0.0..."


In [5]:
result_df.to_csv('Results/Descriptors/Results_2d_RDKit_desc.csv')
prediction_df.to_csv('Results/Descriptors/Prediction_data_2d_RDKit_desc.csv')

In [6]:
#2d Mordred descriptors
df_train = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = X_test.select_dtypes(include=['number'])
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models_2dM = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df , prediction_df= train_and_test_predict(models_2dM, X_train,y_train, X_test,  y_test)
result_df

/tmp/ipykernel_3612489/2480857366.py:2: DtypeWarning: Columns (1058,1060,1075,1081,1137,1139,1154,1160,1362,1363,1364,1366,1367,1378,1379,1380) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc.csv')


X_train shape:  (5568, 1426)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX


/tmp/ipykernel_3612489/2480857366.py:9: DtypeWarning: Columns (1058,1060,1081,1137,1139,1160,1362,1363,1364,1366,1367,1380) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc.csv')


X_test shape:  (1392, 1426)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.130696 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 265213
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 1187
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.129181 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 265241
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 1190
[LightGBM] [Info] Start training from score -5.744959
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.127066 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] T

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2052,0.3336,0.4530,0.6704,0.8193,0.7868,0.2066,0.3292,0.4545,0.6751,0.8229,0.8009
DecisionTreeRegressor,0.2942,0.3822,0.5424,0.5275,0.7445,0.7255,0.2340,0.3459,0.4837,0.6320,0.7962,0.7721
RandomForestRegressor,0.2143,0.3426,0.4630,0.6557,0.8101,0.7800,0.2141,0.3349,0.4627,0.6632,0.8150,0.7939
GradientBoostingRegressor,0.2233,0.3516,0.4725,0.6414,0.8038,0.7665,0.2191,0.3430,0.4681,0.6554,0.8144,0.7873
AdaBoostRegressor,0.3870,0.5032,0.6221,0.3784,0.6563,0.6097,0.3715,0.4884,0.6095,0.4156,0.6880,0.6703
XGBRegressor,0.2142,0.3406,0.4629,0.6559,0.8104,0.7787,0.2113,0.3318,0.4597,0.6676,0.8173,0.7947
ExtraTreesRegressor,0.2123,0.3401,0.4608,0.6590,0.8118,0.7832,0.2218,0.3389,0.4710,0.6511,0.8071,0.7877
LinearRegression,0.6392,0.4703,0.7995,-0.0266,0.5931,0.6936,0.3301,0.3791,0.5746,0.4807,0.7241,0.7489
KNeighborsRegressor,0.2737,0.3818,0.5231,0.5604,0.7558,0.7233,0.2629,0.3697,0.5127,0.5865,0.7695,0.7592
SVR,0.2251,0.3412,0.4745,0.6384,0.8024,0.7728,0.2373,0.3501,0.4872,0.6267,0.7963,0.7765


In [7]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.735305541486875, -6.9205211957429436, -6.3...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.095159381731123, -5.990806176817479, -6.7...","[-7.0656379422361955, -6.223382642199061, -6.6...","[0.049243797160678664, 0.20819955148271957, 0...."
1,DecisionTreeRegressor,"[-6.82, -6.93, -4.72, -4.74, -5.15, -4.74, -4....",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-4.7, -7.0, -6.92, -7.0, -6.24, -7.0, -6.85,...","[-6.531999999999999, -6.473999999999999, -6.71...","[0.916130995000169, 0.7336375126723005, 0.5168..."
2,RandomForestRegressor,"[-6.654516666666667, -6.821899999999999, -5.95...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.835900000000001, -6.290400000000002, -6.7...","[-6.879930000000002, -6.3542900000000015, -6.7...","[0.04330285902801241, 0.11125144673216666, 0.1..."
3,GradientBoostingRegressor,"[-7.189426609290867, -6.94670740996093, -5.724...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.781472482047078, -6.232044261154379, -6.6...","[-6.96447216267152, -6.204629935122231, -6.587...","[0.1269440868145078, 0.2824411602533746, 0.173..."
4,AdaBoostRegressor,"[-6.263708947213334, -6.099775603261716, -5.69...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.169385749385756, -5.662630481094572, -5.7...","[-6.129832443749557, -5.672400239377824, -5.67...","[0.22938904242239025, 0.04011960553512091, 0.0..."
5,XGBRegressor,"[-7.1114745, -7.200122, -5.9211245, -5.464409,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.981971, -6.1278973, -6.925021, -6.6805005...","[-7.159848, -6.274995, -6.6936846, -6.7125916,...","[0.14607872, 0.1749167, 0.26622304, 0.12617067..."
6,ExtraTreesRegressor,"[-6.751800000000002, -6.851999999999999, -6.20...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.9637, -6.613500000000001, -6.847599999999...","[-6.947040000000001, -6.543460000000001, -6.83...","[0.036522245275995055, 0.14411945878332955, 0...."
7,LinearRegression,"[-7.557044007836517, -10.0, -6.143251816763218...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.8035805630708595, -6.194793962428953, -8....","[-6.720716112614172, -6.318323524394141, -8.77...","[2.766276270378472, 1.0696032550140442, 0.0621..."
8,KNeighborsRegressor,"[-7.0, -7.0, -5.88, -4.88, -4.733333333333333,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -6.746666666666667, -6.746666666666667...","[-7.0, -6.898666666666666, -6.797333333333334,...","[0.0, 0.12410748030101418, 0.10133333333333318..."
9,SVR,"[-7.172611208081387, -7.019979819675282, -5.91...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.299919902210404, -6.7853018575183155, -7....","[-7.28606138592024, -6.7422261932980785, -7.00...","[0.04465894859693071, 0.05265462766631486, 0.0..."


In [8]:
result_df.to_csv('Results/Descriptors/Results_2d_Mordred_desc.csv')
prediction_df.to_csv('Results/Descriptors/Prediction_data_2d_Mordred_desc.csv')


In [9]:
#Removal of constant columns
def remove_constant_columns(df):
    constant_columns = [col for col in df.columns if df[col].nunique() <= 1]
    df_cleaned = df.drop(columns=constant_columns)
    return df_cleaned, constant_columns

In [10]:
#Low variance column removal
def remove_low_variance_columns(df, threshold=0.005):
    variances = df.var()
    low_variance_columns = variances[variances < threshold].index.tolist()
    df_cleaned = df.drop(columns=low_variance_columns)
    return df_cleaned, low_variance_columns

In [11]:
#2d RDKit descriptors const removal
df_train = pd.read_csv('features/Descriptors/Train_2d_RDKit_des.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train, const_col = remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_RDKit_des.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = X_test.drop(const_col,axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (5568, 183)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 183)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009292 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 19594
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 156
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005411 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 19618
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 158
[LightGBM] [Info] Start training from score -5.744959
[LightGBM] [Info] Auto-choosing col-wise multi-threading, 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2067,0.3340,0.4546,0.6681,0.8179,0.7871,0.2121,0.3322,0.4605,0.6664,0.8172,0.7972
DecisionTreeRegressor,0.3124,0.3871,0.5589,0.4982,0.7275,0.7098,0.2333,0.3428,0.4830,0.6330,0.7967,0.7763
RandomForestRegressor,0.2132,0.3404,0.4617,0.6576,0.8110,0.7815,0.2150,0.3323,0.4637,0.6618,0.8137,0.7940
GradientBoostingRegressor,0.2325,0.3595,0.4822,0.6266,0.7947,0.7568,0.2342,0.3564,0.4840,0.6316,0.7985,0.7739
AdaBoostRegressor,0.4098,0.5204,0.6401,0.3418,0.6298,0.5700,0.4009,0.5114,0.6332,0.3694,0.6507,0.6175
XGBRegressor,0.2141,0.3378,0.4627,0.6562,0.8107,0.7836,0.2074,0.3235,0.4554,0.6738,0.8211,0.8026
ExtraTreesRegressor,0.2060,0.3329,0.4538,0.6692,0.8181,0.7891,0.2137,0.3306,0.4623,0.6639,0.8150,0.7950
LinearRegression,0.3712,0.4354,0.6093,0.4038,0.6460,0.6723,0.3504,0.4280,0.5919,0.4489,0.6726,0.6902
KNeighborsRegressor,0.2851,0.3869,0.5339,0.5421,0.7427,0.7138,0.2868,0.3799,0.5355,0.5490,0.7459,0.7395
SVR,0.2469,0.3549,0.4969,0.6034,0.7804,0.7566,0.2515,0.3561,0.5015,0.6043,0.7810,0.7644


In [12]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.946673987951213, -6.876538862848576, -6.04...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.16054630977522, -6.139016132737373, -6.89...","[-7.177746557729078, -6.36758965089367, -6.918...","[0.07699340211630198, 0.160252333875179, 0.044..."
1,DecisionTreeRegressor,"[-7.0, -7.0, -6.244999999999999, -4.66, -5.15,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -4.89, -7.0, -7.0, -5.47, -7.0, -6.85,...","[-6.992, -5.409999999999999, -7.0, -5.96200000...","[0.016000000000000014, 0.6697163578710021, 0.0..."
2,RandomForestRegressor,"[-6.92, -6.8033, -5.978608333333333, -5.446700...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.962200000000001, -5.996745784910002, -6.9...","[-6.97702, -6.142934056847143, -6.909792, -6.4...","[0.009147327478558787, 0.1290219924737224, 0.0..."
3,GradientBoostingRegressor,"[-6.876390513166994, -7.007488359209473, -5.55...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.982061266075649, -6.262476550916485, -6.6...","[-7.193598658545653, -6.336573466574402, -6.61...","[0.14825564740271557, 0.20237938780268333, 0.0..."
4,AdaBoostRegressor,"[-5.7551806123565115, -5.789599756949643, -5.4...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.1350288069610235, -5.575545977011509, -5....","[-6.28931510950671, -5.679736332908967, -5.598...","[0.11421124820673118, 0.060087524383770435, 0...."
5,XGBRegressor,"[-7.222617, -6.3722773, -6.6379023, -5.1207943...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.180086, -6.507598, -6.92711, -7.2824736, ...","[-7.0340567, -6.3059316, -6.919819, -6.8459597...","[0.10285555, 0.29018506, 0.11030027, 0.2572470..."
6,ExtraTreesRegressor,"[-6.917100000000001, -6.785699999999999, -6.08...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.994900000000002, -6.423500000000003, -6.9...","[-6.989880000000001, -6.464280000000002, -6.95...","[0.006406684009688747, 0.10584187073176653, 0...."
7,LinearRegression,"[-8.915444779816623, -6.565736767049641, -5.49...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-10.0, -6.216474713238361, -6.36512921309384...","[-10.0, -6.05275159840851, -6.257460170592668,...","[0.0, 0.13721514597450124, 0.08872542079344065..."
8,KNeighborsRegressor,"[-6.63, -6.986666666666667, -5.88, -4.89, -4.7...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.63, -7.0, -7.0, -5.383333333333333, -6.74...","[-6.7780000000000005, -6.949333333333334, -6.8...","[0.18126224096595522, 0.10133333333333318, 0.2..."
9,SVR,"[-6.1751343035325945, -6.926225657607661, -6.0...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.175057799882703, -6.429847081386513, -6.9...","[-6.216586780451111, -6.502132590926554, -6.94...","[0.02231498254671676, 0.0739980451521487, 0.02..."


In [13]:
result_df.to_csv('Results/Descriptors/Results_2d_rdkit_const_rem.csv')
prediction_df.to_csv('Results/Descriptors/Prediction_data_2d_rdkit_const_rem.csv')


In [14]:
#2d Mordred descriptors const removal
df_train = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train, const_col = remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = X_test.select_dtypes(include=['number'])
X_test = X_test.drop(const_col,axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models_2dM = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models_2dM, X_train,y_train, X_test,  y_test)
result_df

/tmp/ipykernel_3612489/1139105683.py:2: DtypeWarning: Columns (1058,1060,1075,1081,1137,1139,1154,1160,1362,1363,1364,1366,1367,1378,1379,1380) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc.csv')


X_train shape:  (5568, 1227)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX


/tmp/ipykernel_3612489/1139105683.py:10: DtypeWarning: Columns (1058,1060,1081,1137,1139,1160,1362,1363,1364,1366,1367,1380) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc.csv')


X_test shape:  (1392, 1227)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.124697 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 265213
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 1187
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.123912 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 265241
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 1190
[LightGBM] [Info] Start training from score -5.744959
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.123465 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] T

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2052,0.3336,0.4530,0.6704,0.8193,0.7868,0.2066,0.3292,0.4545,0.6751,0.8229,0.8009
DecisionTreeRegressor,0.2942,0.3817,0.5424,0.5275,0.7438,0.7238,0.2376,0.3471,0.4875,0.6263,0.7928,0.7741
RandomForestRegressor,0.2149,0.3431,0.4636,0.6549,0.8095,0.7793,0.2138,0.3349,0.4624,0.6637,0.8154,0.7944
GradientBoostingRegressor,0.2235,0.3517,0.4727,0.6411,0.8036,0.7666,0.2195,0.3433,0.4685,0.6548,0.8141,0.7865
AdaBoostRegressor,0.3857,0.5017,0.6210,0.3806,0.6565,0.6106,0.3682,0.4844,0.6068,0.4208,0.6895,0.6705
XGBRegressor,0.2142,0.3406,0.4629,0.6559,0.8104,0.7787,0.2113,0.3318,0.4597,0.6676,0.8173,0.7947
ExtraTreesRegressor,0.2141,0.3411,0.4628,0.6561,0.8100,0.7831,0.2213,0.3389,0.4705,0.6518,0.8075,0.7876
LinearRegression,0.6392,0.4703,0.7995,-0.0266,0.5931,0.6936,0.3301,0.3791,0.5746,0.4807,0.7241,0.7489
KNeighborsRegressor,0.2736,0.3820,0.5231,0.5606,0.7558,0.7233,0.2623,0.3698,0.5122,0.5874,0.7702,0.7602
SVR,0.2252,0.3412,0.4745,0.6384,0.8024,0.7728,0.2373,0.3501,0.4872,0.6267,0.7963,0.7765


In [15]:
result_df

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2052,0.3336,0.4530,0.6704,0.8193,0.7868,0.2066,0.3292,0.4545,0.6751,0.8229,0.8009
DecisionTreeRegressor,0.2942,0.3817,0.5424,0.5275,0.7438,0.7238,0.2376,0.3471,0.4875,0.6263,0.7928,0.7741
RandomForestRegressor,0.2149,0.3431,0.4636,0.6549,0.8095,0.7793,0.2138,0.3349,0.4624,0.6637,0.8154,0.7944
GradientBoostingRegressor,0.2235,0.3517,0.4727,0.6411,0.8036,0.7666,0.2195,0.3433,0.4685,0.6548,0.8141,0.7865
AdaBoostRegressor,0.3857,0.5017,0.6210,0.3806,0.6565,0.6106,0.3682,0.4844,0.6068,0.4208,0.6895,0.6705
XGBRegressor,0.2142,0.3406,0.4629,0.6559,0.8104,0.7787,0.2113,0.3318,0.4597,0.6676,0.8173,0.7947
ExtraTreesRegressor,0.2141,0.3411,0.4628,0.6561,0.8100,0.7831,0.2213,0.3389,0.4705,0.6518,0.8075,0.7876
LinearRegression,0.6392,0.4703,0.7995,-0.0266,0.5931,0.6936,0.3301,0.3791,0.5746,0.4807,0.7241,0.7489
KNeighborsRegressor,0.2736,0.3820,0.5231,0.5606,0.7558,0.7233,0.2623,0.3698,0.5122,0.5874,0.7702,0.7602
SVR,0.2252,0.3412,0.4745,0.6384,0.8024,0.7728,0.2373,0.3501,0.4872,0.6267,0.7963,0.7765


In [16]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.735305541486875, -6.9205211957429436, -6.3...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.095159381731123, -5.990806176817479, -6.7...","[-7.0656379422361955, -6.223382642199061, -6.6...","[0.049243797160678664, 0.20819955148271957, 0...."
1,DecisionTreeRegressor,"[-7.0, -7.0, -4.68, -4.68, -5.15, -4.68, -4.68...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-4.26, -6.244999999999999, -6.96, -7.0, -6.2...","[-6.444, -6.491, -6.728, -6.922, -5.892, -6.65...","[1.0921098845812176, 0.6303522824579918, 0.524..."
2,RandomForestRegressor,"[-6.604723076923077, -6.7789, -6.0228266666666...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.827425000000002, -6.144400000000002, -6.7...","[-6.881725, -6.340030000000001, -6.72093761597...","[0.05499682263549351, 0.1332165590307748, 0.14..."
3,GradientBoostingRegressor,"[-7.189426609290868, -6.946707409960931, -5.72...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.781472482047079, -6.232044261154381, -6.6...","[-6.96447216267152, -6.20462993512223, -6.5874...","[0.12694408681450794, 0.2824411602533745, 0.17..."
4,AdaBoostRegressor,"[-6.083163362536862, -6.304680851063831, -5.46...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.315775075987846, -5.72387980283165, -5.67...","[-6.17345854934281, -5.686763891885435, -5.674...","[0.09624540040691523, 0.02828573103474178, 0.0..."
5,XGBRegressor,"[-7.1114745, -7.200122, -5.9211245, -5.464409,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.981971, -6.1278973, -6.925021, -6.6805005...","[-7.159848, -6.274995, -6.6936846, -6.7125916,...","[0.14607872, 0.1749167, 0.26622304, 0.12617067..."
6,ExtraTreesRegressor,"[-6.5864, -6.862900000000001, -6.27255, -5.360...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.9831, -6.5390500000000005, -6.82761290322...","[-6.94212, -6.540626948800001, -6.848882434154...","[0.029767861864769193, 0.1564071842625883, 0.0..."
7,LinearRegression,"[-7.557044872476467, -10.0, -6.143252049966463...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.803588096911767, -6.194795048723677, -8.7...","[-6.720717619382353, -6.31832355734184, -8.779...","[2.766275770823795, 1.0696034058149013, 0.0621..."
8,KNeighborsRegressor,"[-7.0, -7.0, -5.88, -4.88, -4.733333333333333,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -6.746666666666667, -6.746666666666667...","[-7.0, -6.898666666666666, -6.797333333333334,...","[0.0, 0.12410748030101418, 0.10133333333333318..."
9,SVR,"[-7.172501753188275, -7.019828948900898, -5.91...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.299626736962334, -6.785176922695333, -7.0...","[-7.286013435872765, -6.742356343932397, -7.00...","[0.04474706247529814, 0.05262738427231812, 0.0..."


In [17]:
result_df.to_csv('Results/Descriptors/Results_2d_Mordred_const_rem.csv')
prediction_df.to_csv('Results/Descriptors/Prediction_df_2d_Mordred_const_rem.csv')


In [18]:
#2d RDKit descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_2d_RDKit_des.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train, const_col = remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_RDKit_des.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = X_test.drop(const_col,axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models_LVR_rdkit = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models_LVR_rdkit, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (5568, 149)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 149)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.010898 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 16888
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 145
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006512 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 16918
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 147
[LightGBM] [Info] Start training from score -5.744959
[LightGBM] [Info] Auto-choosing col-wise multi-threading, 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2081,0.3350,0.4562,0.6658,0.8165,0.7860,0.2127,0.3319,0.4612,0.6654,0.8165,0.7963
DecisionTreeRegressor,0.3002,0.3826,0.5479,0.5178,0.7386,0.7197,0.2406,0.3450,0.4905,0.6216,0.7905,0.7686
RandomForestRegressor,0.2137,0.3399,0.4623,0.6567,0.8104,0.7799,0.2140,0.3313,0.4626,0.6634,0.8147,0.7949
GradientBoostingRegressor,0.2343,0.3599,0.4840,0.6237,0.7934,0.7527,0.2354,0.3575,0.4852,0.6297,0.7979,0.7684
AdaBoostRegressor,0.4143,0.5214,0.6436,0.3347,0.6245,0.5801,0.4018,0.5117,0.6338,0.3681,0.6506,0.6240
XGBRegressor,0.2108,0.3351,0.4592,0.6613,0.8138,0.7864,0.2052,0.3226,0.4530,0.6773,0.8231,0.8038
ExtraTreesRegressor,0.2074,0.3338,0.4554,0.6669,0.8167,0.7879,0.2144,0.3320,0.4630,0.6627,0.8143,0.7942
LinearRegression,0.3818,0.4423,0.6179,0.3868,0.6326,0.6569,0.3556,0.4337,0.5963,0.4407,0.6652,0.6717
KNeighborsRegressor,0.2819,0.3852,0.5310,0.5472,0.7464,0.7176,0.2808,0.3757,0.5299,0.5583,0.7514,0.7487
SVR,0.2469,0.3561,0.4969,0.6035,0.7800,0.7546,0.2535,0.3581,0.5035,0.6012,0.7787,0.7616


In [19]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.778957437376595, -6.827088317456864, -6.00...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.989564961842148, -6.1368929981822555, -6....","[-7.121944937536517, -6.237705277453703, -6.90...","[0.07509825988765652, 0.07363273887258097, 0.1..."
1,DecisionTreeRegressor,"[-7.0, -7.0, -6.244999999999999, -5.05, -5.15,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -4.89, -7.0, -7.0, -6.27, -7.0, -6.85,...","[-6.992, -5.720000000000001, -7.0, -6.698, -6....","[0.016000000000000014, 0.7863586967790209, 0.0..."
2,RandomForestRegressor,"[-6.842, -6.9244, -5.850108333333331, -5.38200...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.988400000000002, -5.977795784910001, -6.9...","[-6.975500000000001, -6.193093927478154, -6.95...","[0.0169064484738812, 0.1641023701435488, 0.033..."
3,GradientBoostingRegressor,"[-6.854939176320365, -6.921942576114342, -5.48...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.905455446838034, -6.146337829946822, -6.5...","[-7.065130124146715, -6.352390672263323, -6.56...","[0.15710357223888058, 0.20288519074098627, 0.0..."
4,AdaBoostRegressor,"[-5.920937500000001, -5.85057618749055, -5.546...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.061298076923079, -5.7103846153846165, -5....","[-6.251961186380965, -5.7727853590584015, -5.6...","[0.12972122934788904, 0.0921802133101486, 0.04..."
5,XGBRegressor,"[-6.882107, -6.7906647, -6.597628, -5.24423, -...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.905106, -6.44469, -6.908705, -6.6967864, ...","[-7.1257257, -6.422535, -7.0225286, -6.6813354...","[0.25522998, 0.1519141, 0.15859328, 0.18693231..."
6,ExtraTreesRegressor,"[-6.899849999999999, -6.863800000000001, -6.10...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.989000000000001, -6.444100000000002, -6.9...","[-6.98362, -6.459100000000002, -6.94136, -6.51...","[0.006456748407674181, 0.20768522335496045, 0...."
7,LinearRegression,"[-9.617622714262655, -6.617079416739121, -5.38...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-10.0, -6.252216290915951, -6.22558331086572...","[-10.0, -6.03949232689128, -6.075192193209634,...","[0.0, 0.14346996553443409, 0.11107660006094226..."
8,KNeighborsRegressor,"[-6.63, -6.986666666666667, -5.88, -4.89, -4.7...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.63, -7.0, -7.0, -5.180000000000001, -6.74...","[-6.7780000000000005, -6.949333333333334, -6.8...","[0.18126224096595522, 0.10133333333333318, 0.2..."
9,SVR,"[-6.205303148131038, -6.906765732846281, -6.05...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.205293468301002, -6.552598193608387, -6.8...","[-6.239451653224063, -6.604271180065403, -6.97...","[0.01815178859569605, 0.07986619026140122, 0.0..."


In [20]:
result_df.to_csv('Results/Descriptors/Results_2d_rdkit_LVR.csv')
prediction_df.to_csv('Results/Descriptors/Prediction_data_2d_rdkit_LVR.csv')


In [21]:
#2d Mordred descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train, const_col = remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = X_test.select_dtypes(include=['number'])
X_test = X_test.drop(const_col,axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
results_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
results_df

/tmp/ipykernel_3612489/1788122206.py:2: DtypeWarning: Columns (1058,1060,1075,1081,1137,1139,1154,1160,1362,1363,1364,1366,1367,1378,1379,1380) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc.csv')


X_train shape:  (5568, 821)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX


/tmp/ipykernel_3612489/1788122206.py:10: DtypeWarning: Columns (1058,1060,1081,1137,1139,1160,1362,1363,1364,1366,1367,1380) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc.csv')


X_test shape:  (1392, 821)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.085575 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 170429
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 808
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.085823 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 170473
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 811
[LightGBM] [Info] Start training from score -5.744959
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.085964 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Tota

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2082,0.3357,0.4563,0.6656,0.8163,0.7854,0.2071,0.3281,0.4550,0.6743,0.8223,0.8033
DecisionTreeRegressor,0.2980,0.3814,0.5459,0.5214,0.7398,0.7212,0.2353,0.3468,0.4851,0.6299,0.7948,0.7710
RandomForestRegressor,0.2149,0.3425,0.4636,0.6548,0.8094,0.7796,0.2157,0.3357,0.4644,0.6608,0.8134,0.7933
GradientBoostingRegressor,0.2266,0.3547,0.4761,0.6360,0.8008,0.7647,0.2246,0.3476,0.4739,0.6468,0.8090,0.7796
AdaBoostRegressor,0.3914,0.5044,0.6256,0.3713,0.6485,0.6075,0.3752,0.4905,0.6125,0.4098,0.6805,0.6625
XGBRegressor,0.2174,0.3410,0.4662,0.6509,0.8073,0.7761,0.2137,0.3320,0.4623,0.6638,0.8149,0.7910
ExtraTreesRegressor,0.2102,0.3387,0.4585,0.6624,0.8139,0.7849,0.2210,0.3379,0.4701,0.6524,0.8079,0.7894
LinearRegression,0.4257,0.4059,0.6524,0.3163,0.6725,0.7367,0.3084,0.3724,0.5554,0.5149,0.7372,0.7625
KNeighborsRegressor,0.2755,0.3820,0.5249,0.5575,0.7534,0.7228,0.2675,0.3710,0.5172,0.5792,0.7641,0.7568
SVR,0.2361,0.3484,0.4859,0.6207,0.7912,0.7621,0.2464,0.3546,0.4964,0.6124,0.7872,0.7682


In [22]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.779166709714935, -6.990389408711763, -6.34...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.272376944821139, -6.020476662588773, -6.7...","[-7.0852794995891895, -6.265442599980678, -6.6...","[0.12700062440838517, 0.16959163451144194, 0.1..."
1,DecisionTreeRegressor,"[-7.0, -7.0, -6.03, -5.2, -6.244999999999999, ...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -5.74, -7.0, -7.0, -5.57, -6.51, -6.85...","[-6.992, -6.206, -6.731999999999999, -6.698, -...","[0.016000000000000014, 0.6244229335954917, 0.5..."
2,RandomForestRegressor,"[-6.5874, -6.838499999999999, -6.0951816666666...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.824200000000001, -6.227425000000002, -6.7...","[-6.902939999999999, -6.387695000000001, -6.73...","[0.05048780446800969, 0.13577022353962473, 0.1..."
3,GradientBoostingRegressor,"[-6.741378048160809, -6.953489973327972, -5.79...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.2592906519527345, -6.008690362381369, -6....","[-7.147868016333987, -6.226032767248003, -6.63...","[0.08977104476172272, 0.2651446860854107, 0.09..."
4,AdaBoostRegressor,"[-6.118562069713626, -6.2907094965872465, -5.3...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.266660245185529, -5.615490551355289, -5.7...","[-6.07567042174192, -5.682555866097489, -5.676...","[0.1280692378739982, 0.06489852637272152, 0.02..."
5,XGBRegressor,"[-7.0040116, -6.9146557, -6.336396, -5.062, -4...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.9632425, -6.2396154, -7.2159743, -6.89189...","[-7.0798173, -6.4098015, -6.848582, -6.9309783...","[0.10771529, 0.23953909, 0.23422658, 0.1693177..."
6,ExtraTreesRegressor,"[-6.8066, -6.956899999999998, -6.2260500000000...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.972, -6.633000000000001, -6.9744, -6.5555...","[-6.957800000000001, -6.5981200000000015, -6.9...","[0.0369838883839972, 0.07208371244601593, 0.03..."
7,LinearRegression,"[-7.527671684015331, -7.2071348544841385, -6.2...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.57391535161787, -7.386611547756729, -7.46...","[-7.251487885802698, -7.033285869285891, -7.79...","[1.5744785402789796, 0.5875458154500403, 0.301..."
8,KNeighborsRegressor,"[-7.0, -7.0, -5.88, -4.853333333333333, -4.733...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -6.746666666666667, -6.746666666666667...","[-7.0, -6.898666666666666, -6.797333333333334,...","[0.0, 0.12410748030101418, 0.10133333333333318..."
9,SVR,"[-7.11074742873326, -7.008255790008885, -5.934...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.11881157321091, -6.790641808424154, -7.01...","[-7.116296752472833, -6.767235916417836, -7.01...","[0.03794918390120279, 0.0399489754387168, 0.07..."


In [23]:
results_df.to_csv('Results/Descriptors/Results_2d_Mordred_LVR.csv')
prediction_df.to_csv('Results/Descriptors/Prediction_data_2d_Mordred_LVR.csv')


In [24]:
#2d Padel descriptors
df_train = pd.read_csv('features/Descriptors/Train_2d_padel.csv')
df_train['ID'] = df_train['Name'].str.extract(r'_(\d+)$')
df_train['ID'] = df_train['ID'].astype(int)
df_train = df_train.drop('Name',axis=1)
df_train = df_train.fillna(0)
df_train



,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,nAtom,nHeavyAtom,nH,...,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb,ID
0,0,-4.2390,17.969121,111.3714,69.282962,0,0,66,32,34,...,61.494697,1.921709,33.025324,14.973326,18.051998,2732.0,56.0,0.188,156.0,1020
1,0,-4.2390,17.969121,111.3714,69.282962,0,0,66,32,34,...,61.494634,1.921707,33.024381,14.973124,18.051258,2735.0,56.0,0.188,156.0,1022
2,0,-4.2390,17.969121,111.3714,69.282962,0,0,66,32,34,...,61.494634,1.921707,33.024381,14.973124,18.051258,2735.0,56.0,0.188,156.0,1021
3,0,-4.0344,16.276383,122.7648,75.470134,0,0,72,34,38,...,65.139553,1.915869,33.439372,14.950444,18.488928,3129.0,64.0,-0.386,168.0,1023
4,0,0.5074,0.257455,217.1772,135.903096,0,0,128,56,72,...,109.216914,1.950302,36.902385,17.809033,19.093351,11361.0,95.0,6.902,276.0,1029
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5563,0,-4.5553,20.750758,443.6195,272.993848,0,0,253,117,136,...,230.754758,1.972263,82.512962,38.029136,44.483827,89262.0,201.0,8.428,580.0,922
5564,0,-5.9605,35.527560,440.7941,272.895469,0,0,252,119,133,...,235.781272,1.981355,87.859311,40.619051,47.240260,94051.0,199.0,8.707,594.0,914
5565,0,-5.5847,31.188874,437.8295,270.079883,0,0,248,117,131,...,231.630909,1.979751,85.100101,38.091980,44.482545,90275.0,203.0,8.341,588.0,904
5566,0,-3.1676,10.033690,466.9254,281.726676,0,0,255,123,132,...,244.597932,1.988601,87.647449,38.186244,44.409996,106182.0,208.0,10.250,620.0,888


In [25]:
df = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc.csv')
df 


/tmp/ipykernel_3612489/589299306.py:1: DtypeWarning: Columns (1058,1060,1075,1081,1137,1139,1154,1160,1362,1363,1364,1366,1367,1378,1379,1380) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc.csv')


,ID,SMILES,Permeability,ABCIndex,ABCGGIndex,AcidicGroupCount,BasicGroupCount,AdjacencyMatrix,AdjacencyMatrix.1,AdjacencyMatrix.2,...,WalkCount.19,WalkCount.20,Weight,Weight.1,WienerIndex,WienerIndex.1,ZagrebIndex,ZagrebIndex.1,ZagrebIndex.2,ZagrebIndex.3
0,915,CC[C@H](C)[C@H](NC(=O)[C@@H]1CC(=O)N[C@@H](Cc2...,-7.0,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,155.712529,2.416852,4.832818,...,11.593980,169.458441,1772.125588,6.420745,107741,214,630.0,725.0,55.173611,28.291667
1,888,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1cccc(Cl)c1)N(C)...,-7.0,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,152.500384,2.445965,4.858291,...,11.590155,178.997365,1742.937365,6.835048,106182,208,620.0,719.0,49.861111,27.277778
2,593,C/N=C(\NC)NCCC[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C...,-7.0,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,3,154.777488,2.415089,4.830179,...,11.523579,167.253791,1732.084392,6.511595,102214,204,616.0,705.0,50.027778,28.083333
3,916,CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@@H]1CC(=...,-7.0,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,149.378369,2.419912,4.839823,...,11.562315,165.286866,1724.125588,6.338697,101212,208,608.0,699.0,55.673611,27.319444
4,900,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)N(C)C(=O...,-7.0,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,150.261697,2.429936,4.859872,...,11.591302,165.339859,1722.146324,6.285206,107844,215,608.0,706.0,56.194444,27.555556
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5563,2469,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-4.7,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,35.896500,2.304816,4.609632,...,9.859065,63.474076,402.263091,6.385128,2286,42,138.0,153.0,10.638889,6.583333
5564,2467,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-5.6,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,33.357653,2.311467,4.622934,...,9.832367,61.242657,374.231791,6.565470,1880,40,130.0,145.0,10.138889,6.083333
5565,2512,CC(C)C[C@H]1NC(=O)[C@@H](C)NCCCCCCNC(=O)[C@H](...,-5.5,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,31.502840,2.295993,4.591987,...,9.633842,59.559229,370.258006,6.170967,1648,37,118.0,129.0,10.777778,6.083333
5566,2511,CC(C)[C@@H]1NC(=O)[C@@H](CO)NC(=O)[C@@H](C)NCC...,-6.3,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,30.506102,2.303541,4.607082,...,9.648660,58.482085,356.242356,6.249866,1472,37,114.0,126.0,10.527778,5.861111


In [26]:
merged_df = df_train.merge(df[['ID', 'SMILES', 'Permeability']], on='ID', how='left')
merged_df = merged_df[['ID', 'SMILES', 'Permeability'] + [col for col in merged_df.columns if col not in ['ID', 'SMILES', 'Permeability']]]
merged_df

,ID,SMILES,Permeability,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,...,AMW,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb
0,1020,C[C@H]1C(=O)N[C@H](C)C(=O)N[C@H](C)C(=O)N[C@H]...,-8.20,0,-4.2390,17.969121,111.3714,69.282962,0,0,...,6.882636,61.494697,1.921709,33.025324,14.973326,18.051998,2732.0,56.0,0.188,156.0
1,1022,C[C@H]1C(=O)N[C@H](C)C(=O)N[C@H](C)C(=O)N[C@H]...,-8.30,0,-4.2390,17.969121,111.3714,69.282962,0,0,...,6.882636,61.494634,1.921707,33.024381,14.973124,18.051258,2735.0,56.0,0.188,156.0
2,1021,C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@@H](C)N(C)C(=O)...,-8.20,0,-4.2390,17.969121,111.3714,69.282962,0,0,...,6.882636,61.494634,1.921707,33.024381,14.973124,18.051258,2735.0,56.0,0.188,156.0
3,1023,C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](C)N(C)C(...,-7.00,0,-4.0344,16.276383,122.7648,75.470134,0,0,...,6.698407,65.139553,1.915869,33.439372,14.950444,18.488928,3129.0,64.0,-0.386,168.0
4,1029,CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[C@@H...,-7.00,0,0.5074,0.257455,217.1772,135.903096,0,0,...,6.129268,109.216914,1.950302,36.902385,17.809033,19.093351,11361.0,95.0,6.902,276.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5563,922,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)N(C)C(=O...,-6.24,0,-4.5553,20.750758,443.6195,272.993848,0,0,...,6.438858,230.754758,1.972263,82.512962,38.029136,44.483827,89262.0,201.0,8.428,580.0
5564,914,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C...,-5.36,0,-5.9605,35.527560,440.7941,272.895469,0,0,...,6.571450,235.781272,1.981355,87.859311,40.619051,47.240260,94051.0,199.0,8.707,594.0
5565,904,CC[C@H](C)[C@H](NC(=O)[C@H](C)N(C)C(=O)[C@@H]1...,-7.00,0,-5.5847,31.188874,437.8295,270.079883,0,0,...,6.640971,231.630909,1.979751,85.100101,38.091980,44.482545,90275.0,203.0,8.341,588.0
5566,888,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1cccc(Cl)c1)N(C)...,-7.00,0,-3.1676,10.033690,466.9254,281.726676,0,0,...,6.835048,244.597932,1.988601,87.647449,38.186244,44.409996,106182.0,208.0,10.250,620.0


In [27]:
df_ordered = merged_df.merge(df[['ID']], on='ID', how='right')
df_ordered = df_ordered.reindex(df.index)
df_ordered

,ID,SMILES,Permeability,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,...,AMW,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb
0,915,CC[C@H](C)[C@H](NC(=O)[C@@H]1CC(=O)N[C@@H](Cc2...,-7.0,0,-4.4613,19.903198,477.8936,296.686157,0,0,...,6.420745,250.045897,1.968865,90.654326,43.537099,47.117227,107741.0,214.0,11.490,630.0
1,888,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1cccc(Cl)c1)N(C)...,-7.0,0,-3.1676,10.033690,466.9254,281.726676,0,0,...,6.835048,244.597932,1.988601,87.647449,38.186244,44.409996,106182.0,208.0,10.250,620.0
2,593,C/N=C(\NC)NCCC[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C...,-7.0,0,-3.0574,9.347695,476.2546,289.385813,0,0,...,6.511595,248.034411,1.984275,88.350956,35.707994,52.642962,102214.0,204.0,10.960,616.0
3,916,CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@@H]1CC(=...,-7.0,0,-4.2279,17.875138,460.2191,289.646157,0,0,...,6.338697,240.980957,1.959195,90.761838,43.488529,47.273309,101212.0,208.0,9.243,608.0
4,900,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)N(C)C(=O...,-7.0,0,-5.0166,25.166276,461.8415,291.937743,0,0,...,6.285206,240.843065,1.958074,87.967406,40.534498,47.432908,107844.0,215.0,10.212,608.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5563,2469,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-4.7,0,-1.1989,1.437361,110.0599,68.196962,0,0,...,6.385128,57.707840,1.989926,19.684197,7.618398,12.065799,2286.0,42.0,3.395,138.0
5564,2467,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-5.6,0,-0.6229,0.388004,104.2367,62.009790,0,0,...,6.565470,53.707230,1.989157,19.692329,7.619967,12.072362,1880.0,40.0,2.679,130.0
5565,2512,CC(C)C[C@H]1NC(=O)[C@@H](C)NCCCCCCNC(=O)[C@H](...,-5.5,0,-3.4938,12.206638,88.5588,61.958962,0,0,...,6.170967,50.804604,1.954023,22.049967,10.061855,11.988112,1648.0,37.0,2.588,118.0
5566,2511,CC(C)[C@@H]1NC(=O)[C@@H](CO)NC(=O)[C@@H](C)NCC...,-6.3,0,-3.7494,14.058000,84.1070,58.865376,0,0,...,6.249866,48.797056,1.951882,22.005013,10.045259,11.959755,1472.0,37.0,1.808,114.0


In [28]:
df_ordered.to_csv('features/Descriptors/Train_2d_padel_curated.csv', index=False)

In [29]:
#2d test padel descriptors
df_test = pd.read_csv('features/Descriptors/Test_2d_padel.csv')
df_test['ID'] = df_test['Name'].str.extract(r'_(\d+)$')
df_test['ID'] = df_test['ID'].astype(int)
df_test = df_test.drop('Name',axis=1)
df_test = df_test.fillna(0)
df_test


,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,nAtom,nHeavyAtom,nH,...,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb,ID
0,0,-4.0344,16.276383,122.7648,75.470134,0,0,72,34,38,...,65.139553,1.915869,33.439372,14.950444,18.488928,3129.0,64.0,-0.386,168.0,1024
1,0,-4.2390,17.969121,111.3714,69.282962,0,0,66,32,34,...,61.494697,1.921709,33.025324,14.973326,18.051998,2732.0,56.0,0.188,156.0,1019
2,0,-1.0901,1.188318,197.5996,125.288752,0,0,117,53,64,...,104.585334,1.973308,36.997868,17.883760,19.114109,9990.0,88.0,5.837,266.0,1063
3,0,-2.5034,6.267012,171.7864,101.392892,0,0,91,47,44,...,94.457952,2.009744,36.417884,17.842938,18.574946,7863.0,74.0,2.829,238.0,1083
4,0,-2.5853,6.683776,183.7187,117.767994,0,0,109,51,58,...,101.771380,1.995517,37.261449,17.936517,19.324931,9053.0,85.0,4.485,262.0,1062
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1387,0,-5.2403,27.460744,475.3685,294.303778,0,0,272,126,146,...,247.936262,1.967748,90.383811,40.574663,47.283631,111779.0,220.0,10.264,626.0,908
1388,0,-3.5373,12.512491,439.2397,261.472332,0,0,239,115,124,...,227.956848,1.982233,82.160615,37.988792,44.171822,84553.0,198.0,7.104,576.0,514
1389,0,-2.4343,5.925816,326.1493,184.096268,0,0,161,85,76,...,172.565284,2.030180,57.110921,25.550678,31.560243,36826.0,140.0,6.007,438.0,7333
1390,0,-4.2478,18.043805,425.6011,254.229574,0,0,230,112,118,...,223.189715,1.992765,79.540550,35.551907,41.463277,85015.0,194.0,7.233,568.0,880


In [30]:
df = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc.csv')
df

/tmp/ipykernel_3612489/354123813.py:1: DtypeWarning: Columns (1058,1060,1081,1137,1139,1160,1362,1363,1364,1366,1367,1380) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc.csv')


,ID,SMILES,Permeability,ABCIndex,ABCGGIndex,AcidicGroupCount,BasicGroupCount,AdjacencyMatrix,AdjacencyMatrix.1,AdjacencyMatrix.2,...,WalkCount.19,WalkCount.20,Weight,Weight.1,WienerIndex,WienerIndex.1,ZagrebIndex,ZagrebIndex.1,ZagrebIndex.2,ZagrebIndex.3
0,908,CC[C@H](C)[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[...,-7.00,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,155.498239,2.424408,4.847496,...,11.617204,168.482860,1776.076051,6.529691,111779,220,626.0,728.0,55.444444,28.277778
1,923,CC[C@H](C)[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](...,-7.00,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,150.138684,2.427131,4.854261,...,11.589822,165.351205,1724.125588,6.338697,99910,213,610.0,706.0,56.284722,27.402778
2,897,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.00,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,150.676716,2.420176,4.840351,...,11.578086,164.308833,1700.068073,6.464137,103927,212,606.0,704.0,53.222222,27.333333
3,587,CC(C)C[C@@H]1NC(=O)[C@H](Cc2c[nH]cn2)NC(=O)[C@...,-6.74,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,151.698924,2.415398,4.830623,...,11.527095,177.329996,1685.010893,6.633901,95572,199,610.0,701.0,47.777778,27.083333
4,921,CC[C@H](C)[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[...,-5.54,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,145.614560,2.422204,4.844100,...,11.547955,161.156358,1668.062988,6.415627,93670,207,588.0,682.0,53.972222,26.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1387,2481,CC(C)C[C@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2ccccc...,-4.50,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,38.436032,2.302525,4.605049,...,9.885069,65.694305,430.294391,6.236151,2750,44,146.0,161.0,11.138889,7.083333
1388,2485,CC(C)C[C@H]1NC(=O)[C@H](C)NC(=O)[C@@H](Cc2cccc...,-4.80,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,38.436032,2.302525,4.605049,...,9.885069,65.694305,430.294391,6.236151,2750,44,146.0,161.0,11.138889,7.083333
1389,5604,CC(C)CN1CC(=O)N[C@@H](Cc2ccccc2)C(=O)NCCCCC(=O...,-6.38,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,37.855097,2.317338,4.629714,...,9.940542,65.841858,430.258006,6.619354,2644,45,148.0,164.0,11.750000,7.000000
1390,2513,C[C@H]1NCCCCCCNC(=O)[C@H](Cc2ccc(O)cc2)NC(=O)[...,-7.80,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,39.745462,2.403441,4.738314,...,10.079414,79.819128,430.258006,6.619354,2650,47,154.0,176.0,10.250000,6.972222


In [31]:
merged_df = df_test.merge(df[['ID', 'SMILES', 'Permeability']], on='ID', how='left')
merged_df = merged_df[['ID', 'SMILES', 'Permeability'] + [col for col in merged_df.columns if col not in ['ID', 'SMILES', 'Permeability']]]
merged_df

,ID,SMILES,Permeability,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,...,AMW,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb
0,1024,C[C@H]1C(=O)N(C)[C@H](C)C(=O)N[C@H](C)C(=O)N(C...,-7.10,0,-4.0344,16.276383,122.7648,75.470134,0,0,...,6.698407,65.139553,1.915869,33.439372,14.950444,18.488928,3129.0,64.0,-0.386,168.0
1,1019,C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@@H](C)N(C)C(=O)...,-8.30,0,-4.2390,17.969121,111.3714,69.282962,0,0,...,6.882636,61.494697,1.921709,33.025324,14.973326,18.051998,2732.0,56.0,0.188,156.0
2,1063,CC(C)C[C@@H]1NC(=O)[C@@H](CC(C)C)N(C)C(=O)[C@@...,-6.17,0,-1.0901,1.188318,197.5996,125.288752,0,0,...,6.328920,104.585334,1.973308,36.997868,17.883760,19.114109,9990.0,88.0,5.837,266.0
3,1083,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)CNC(=...,-7.52,0,-2.5034,6.267012,171.7864,101.392892,0,0,...,7.124474,94.457952,2.009744,36.417884,17.842938,18.574946,7863.0,74.0,2.829,238.0
4,1062,CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[C@H]...,-6.54,0,-2.5853,6.683776,183.7187,117.767994,0,0,...,6.517768,101.771380,1.995517,37.261449,17.936517,19.324931,9053.0,85.0,4.485,262.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1387,908,CC[C@H](C)[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[...,-7.00,0,-5.2403,27.460744,475.3685,294.303778,0,0,...,6.529691,247.936262,1.967748,90.383811,40.574663,47.283631,111779.0,220.0,10.264,626.0
1388,514,CC(C)C[C@H]1C(=O)N[C@@H]([C@@H](C)O)C(=O)N(C)[...,-6.70,0,-3.5373,12.512491,439.2397,261.472332,0,0,...,6.665009,227.956848,1.982233,82.160615,37.988792,44.171822,84553.0,198.0,7.104,576.0
1389,7333,C[C@H]1NC(=O)[C@@H]2CCCN2C(=O)[C@@H](Cc2ccccc2...,-5.37,0,-2.4343,5.925816,326.1493,184.096268,0,0,...,7.183693,172.565284,2.030180,57.110921,25.550678,31.560243,36826.0,140.0,6.007,438.0
1390,880,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)N(C)C(=O...,-7.00,0,-4.2478,18.043805,425.6011,254.229574,0,0,...,6.816787,223.189715,1.992765,79.540550,35.551907,41.463277,85015.0,194.0,7.233,568.0


In [32]:
df_ordered = merged_df.merge(df[['ID']], on='ID', how='right')
df_ordered = df_ordered.reindex(df.index)
df_ordered

,ID,SMILES,Permeability,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,...,AMW,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb
0,908,CC[C@H](C)[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[...,-7.00,0,-5.2403,27.460744,475.3685,294.303778,0,0,...,6.529691,247.936262,1.967748,90.383811,40.574663,47.283631,111779.0,220.0,10.264,626.0
1,923,CC[C@H](C)[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](...,-7.00,0,-5.4756,29.982195,460.5846,289.646157,0,0,...,6.338697,240.768109,1.957464,90.840338,43.433503,47.406835,99910.0,213.0,9.667,610.0
2,897,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.00,0,-5.2907,27.991506,457.4260,283.509813,0,0,...,6.464137,240.251303,1.969273,87.601926,40.568304,47.033622,103927.0,212.0,9.469,606.0
3,587,CC(C)C[C@@H]1NC(=O)[C@H](Cc2c[nH]cn2)NC(=O)[C@...,-6.74,0,-2.6817,7.191515,466.7971,278.764676,0,0,...,6.633901,242.969389,1.991552,85.904257,35.712474,50.191783,95572.0,199.0,10.338,610.0
4,921,CC[C@H](C)[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[...,-5.54,0,-5.8836,34.616749,443.4036,277.271813,0,0,...,6.415627,233.157008,1.959303,90.305554,42.950496,47.355058,93670.0,207.0,9.346,588.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1387,2481,CC(C)C[C@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2ccccc...,-4.50,0,-1.7749,3.150270,115.8831,74.384134,0,0,...,6.236151,61.708009,1.990581,19.682164,7.618006,12.064158,2750.0,44.0,4.111,146.0
1388,2485,CC(C)C[C@H]1NC(=O)[C@H](C)NC(=O)[C@@H](Cc2cccc...,-4.80,0,-1.7749,3.150270,115.8831,74.384134,0,0,...,6.236151,61.708009,1.990581,19.682164,7.618006,12.064158,2750.0,44.0,4.111,146.0
1389,5604,CC(C)CN1CC(=O)N[C@@H](Cc2ccccc2)C(=O)NCCCCC(=O...,-6.38,0,-1.6788,2.818369,115.5380,70.758962,0,0,...,6.619354,61.526924,1.984739,22.541021,10.153913,12.387108,2644.0,45.0,1.941,148.0
1390,2513,C[C@H]1NCCCCCCNC(=O)[C@H](Cc2ccc(O)cc2)NC(=O)[...,-7.80,0,-3.7309,13.919615,109.3044,70.758962,0,0,...,6.619354,62.566820,2.018285,22.680885,10.197129,12.483757,2650.0,47.0,1.336,154.0


In [33]:
df_ordered.to_csv('features/Descriptors/Test_2d_padel_curated.csv', index=False)

In [34]:
#3d Train descriptors
df_train = pd.read_csv('features/Descriptors/Train_3d_padel.csv')
df_train['ID'] = df_train['Name'].str.extract(r'_(\d+)$')
df_train['ID'] = df_train['ID'].astype(int)
df_train = df_train.drop('Name',axis=1)
df_train = df_train.fillna(0)
df_train

,TDB1u,TDB2u,TDB3u,TDB4u,TDB5u,TDB6u,TDB7u,TDB8u,TDB9u,TDB10u,...,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds,ID
0,1.257037,2.171619,3.051367,3.768084,4.446747,5.310072,6.172854,6.961696,7.677743,8.107739,...,0.388688,0.487362,0.445466,0.469314,19.415071,107.583470,262.967432,0.361921,1.402141,1023
1,1.266092,2.199094,3.012133,3.781758,4.621822,5.416989,6.202800,6.899676,7.482442,8.243258,...,0.331150,0.417938,0.481153,0.396502,28.419085,241.498648,859.174369,0.279497,1.295592,1046
2,1.264416,2.188026,3.016937,3.776957,4.616965,5.340868,6.103287,6.911743,7.625871,8.367343,...,0.366755,0.490281,0.465703,0.365969,31.758290,269.313599,709.586063,0.408889,1.321953,1031
3,1.255556,2.171136,3.007365,3.711201,4.563689,5.339879,5.969794,6.683112,7.503840,8.116940,...,0.357198,0.407435,0.467579,0.338714,30.366844,277.039576,1015.801134,0.288218,1.213728,1029
4,1.257578,2.180149,3.028882,3.768428,4.556887,5.342206,5.972700,6.683522,7.302166,7.795733,...,0.321993,0.439392,0.455970,0.433690,17.809613,87.051863,206.975001,0.372184,1.329052,1020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5563,1.279234,2.215212,3.033501,3.814042,4.605440,5.495401,6.233342,6.800465,7.560795,8.316338,...,0.373198,0.411570,0.401501,0.408370,29.089414,239.831499,723.669225,0.360980,1.221441,989
5564,1.254987,2.162249,2.993825,3.715075,4.530517,5.289092,6.009551,6.818365,7.556336,8.205864,...,0.351940,0.568952,0.508951,0.409094,34.349408,323.103730,1010.235458,0.379012,1.486996,977
5565,1.255969,2.167574,2.988664,3.678783,4.543610,5.358258,6.109109,6.897022,7.616435,8.240070,...,0.344552,0.572864,0.493883,0.348709,33.027623,330.814234,1325.054201,0.268087,1.415456,979
5566,1.255808,2.169016,2.992356,3.684378,4.521964,5.353863,6.093558,6.914970,7.689858,8.451555,...,0.371876,0.512823,0.473309,0.453661,37.535778,368.845567,961.107234,0.426983,1.439793,978


In [35]:
df = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc.csv')
df 

/tmp/ipykernel_3612489/3415914565.py:1: DtypeWarning: Columns (1058,1060,1075,1081,1137,1139,1154,1160,1362,1363,1364,1366,1367,1378,1379,1380) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc.csv')


,ID,SMILES,Permeability,ABCIndex,ABCGGIndex,AcidicGroupCount,BasicGroupCount,AdjacencyMatrix,AdjacencyMatrix.1,AdjacencyMatrix.2,...,WalkCount.19,WalkCount.20,Weight,Weight.1,WienerIndex,WienerIndex.1,ZagrebIndex,ZagrebIndex.1,ZagrebIndex.2,ZagrebIndex.3
0,915,CC[C@H](C)[C@H](NC(=O)[C@@H]1CC(=O)N[C@@H](Cc2...,-7.0,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,155.712529,2.416852,4.832818,...,11.593980,169.458441,1772.125588,6.420745,107741,214,630.0,725.0,55.173611,28.291667
1,888,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1cccc(Cl)c1)N(C)...,-7.0,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,152.500384,2.445965,4.858291,...,11.590155,178.997365,1742.937365,6.835048,106182,208,620.0,719.0,49.861111,27.277778
2,593,C/N=C(\NC)NCCC[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C...,-7.0,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,3,154.777488,2.415089,4.830179,...,11.523579,167.253791,1732.084392,6.511595,102214,204,616.0,705.0,50.027778,28.083333
3,916,CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@@H]1CC(=...,-7.0,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,149.378369,2.419912,4.839823,...,11.562315,165.286866,1724.125588,6.338697,101212,208,608.0,699.0,55.673611,27.319444
4,900,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)N(C)C(=O...,-7.0,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,150.261697,2.429936,4.859872,...,11.591302,165.339859,1722.146324,6.285206,107844,215,608.0,706.0,56.194444,27.555556
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5563,2469,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-4.7,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,35.896500,2.304816,4.609632,...,9.859065,63.474076,402.263091,6.385128,2286,42,138.0,153.0,10.638889,6.583333
5564,2467,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-5.6,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,33.357653,2.311467,4.622934,...,9.832367,61.242657,374.231791,6.565470,1880,40,130.0,145.0,10.138889,6.083333
5565,2512,CC(C)C[C@H]1NC(=O)[C@@H](C)NCCCCCCNC(=O)[C@H](...,-5.5,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,31.502840,2.295993,4.591987,...,9.633842,59.559229,370.258006,6.170967,1648,37,118.0,129.0,10.777778,6.083333
5566,2511,CC(C)[C@@H]1NC(=O)[C@@H](CO)NC(=O)[C@@H](C)NCC...,-6.3,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,30.506102,2.303541,4.607082,...,9.648660,58.482085,356.242356,6.249866,1472,37,114.0,126.0,10.527778,5.861111


In [36]:
merged_df = df_train.merge(df[['ID', 'SMILES', 'Permeability']], on='ID', how='left')
merged_df = merged_df[['ID', 'SMILES', 'Permeability'] + [col for col in merged_df.columns if col not in ['ID', 'SMILES', 'Permeability']]]
merged_df

,ID,SMILES,Permeability,TDB1u,TDB2u,TDB3u,TDB4u,TDB5u,TDB6u,TDB7u,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,1023,C[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@@H](C)N(C)C(...,-7.00,1.257037,2.171619,3.051367,3.768084,4.446747,5.310072,6.172854,...,0.519259,0.388688,0.487362,0.445466,0.469314,19.415071,107.583470,262.967432,0.361921,1.402141
1,1046,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)NC(=O)[C...,-7.05,1.266092,2.199094,3.012133,3.781758,4.621822,5.416989,6.202800,...,0.519664,0.331150,0.417938,0.481153,0.396502,28.419085,241.498648,859.174369,0.279497,1.295592
2,1031,CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[C@H]...,-5.17,1.264416,2.188026,3.016937,3.776957,4.616965,5.340868,6.103287,...,0.572504,0.366755,0.490281,0.465703,0.365969,31.758290,269.313599,709.586063,0.408889,1.321953
3,1029,CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[C@@H...,-7.00,1.255556,2.171136,3.007365,3.711201,4.563689,5.339879,5.969794,...,0.501614,0.357198,0.407435,0.467579,0.338714,30.366844,277.039576,1015.801134,0.288218,1.213728
4,1020,C[C@H]1C(=O)N[C@H](C)C(=O)N[C@H](C)C(=O)N[C@H]...,-8.20,1.257578,2.180149,3.028882,3.768428,4.556887,5.342206,5.972700,...,0.581456,0.321993,0.439392,0.455970,0.433690,17.809613,87.051863,206.975001,0.372184,1.329052
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5563,989,CC[C@H](C)[C@@H]1NC(=O)[C@@H]2CCCN2C(=O)[C@H](...,-6.30,1.279234,2.215212,3.033501,3.814042,4.605440,5.495401,6.233342,...,0.534122,0.373198,0.411570,0.401501,0.408370,29.089414,239.831499,723.669225,0.360980,1.221441
5564,977,C/C1=C\[C@@H](CC(C)C)NC(=O)[C@@H]2CCCN2C(=O)[C...,-4.96,1.254987,2.162249,2.993825,3.715075,4.530517,5.289092,6.009551,...,0.567401,0.351940,0.568952,0.508951,0.409094,34.349408,323.103730,1010.235458,0.379012,1.486996
5565,979,C/C1=C\[C@@H](CC(C)C)NC(=O)[C@@H]2CCCN2C(=O)[C...,-5.00,1.255969,2.167574,2.988664,3.678783,4.543610,5.358258,6.109109,...,0.500839,0.344552,0.572864,0.493883,0.348709,33.027623,330.814234,1325.054201,0.268087,1.415456
5566,978,C/C1=C\[C@H](CC(C)C)NC(=O)[C@@H]2CCCN2C(=O)[C@...,-5.52,1.255808,2.169016,2.992356,3.684378,4.521964,5.353863,6.093558,...,0.579446,0.371876,0.512823,0.473309,0.453661,37.535778,368.845567,961.107234,0.426983,1.439793


In [37]:
df_ordered = merged_df.merge(df[['ID']], on='ID', how='right')
df_ordered = df_ordered.reindex(df.index)
df_ordered

,ID,SMILES,Permeability,TDB1u,TDB2u,TDB3u,TDB4u,TDB5u,TDB6u,TDB7u,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,915,CC[C@H](C)[C@H](NC(=O)[C@@H]1CC(=O)N[C@@H](Cc2...,-7.0,1.260283,2.183915,3.015985,3.746559,4.532172,5.359537,6.122190,...,0.645217,0.280139,0.482152,0.434807,0.328596,85.400757,1822.029585,10310.940180,0.467826,1.245555
1,888,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1cccc(Cl)c1)N(C)...,-7.0,1.270866,2.203014,3.036826,3.799846,4.589789,5.362487,6.069947,...,0.823799,0.133805,0.505465,0.357149,0.373757,96.526908,1405.322971,5704.929467,0.735698,1.236370
2,593,C/N=C(\NC)NCCC[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C...,-7.0,1.261305,2.191209,3.025323,3.768154,4.579206,5.368434,6.128057,...,0.638375,0.270605,0.448344,0.374973,0.359780,77.329248,1527.741116,8875.871077,0.457562,1.183097
3,916,CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@@H]1CC(=...,-7.0,1.257879,2.178317,3.009955,3.720143,4.525405,5.336925,6.073236,...,0.565329,0.374752,0.554262,0.543760,0.297042,87.856411,2070.065223,10766.402505,0.410122,1.395064
4,900,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)N(C)C(=O...,-7.0,1.259460,2.177883,3.011654,3.739545,4.533945,5.328030,5.973578,...,0.834311,0.113966,0.516432,0.367614,0.451429,108.294919,1690.332479,8044.745507,0.751467,1.335474
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5563,2469,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-4.7,1.256703,2.185632,2.985814,3.747393,4.545386,5.327188,5.936460,...,0.676563,0.210855,0.446073,0.409627,0.423449,19.367569,90.986445,227.031496,0.514844,1.279149
5564,2467,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-5.6,1.258144,2.200670,2.994074,3.736591,4.603888,5.393495,6.116882,...,0.787624,0.163507,0.454126,0.429936,0.446975,20.666779,74.857439,151.076798,0.681436,1.331037
5565,2512,CC(C)C[C@H]1NC(=O)[C@@H](C)NCCCCCCNC(=O)[C@H](...,-5.5,1.252175,2.169126,2.939989,3.654827,4.533258,5.425237,6.114574,...,0.585877,0.334994,0.455378,0.451219,0.447788,17.819583,85.459813,191.155916,0.381306,1.354384
5566,2511,CC(C)[C@@H]1NC(=O)[C@@H](CO)NC(=O)[C@@H](C)NCC...,-6.3,1.248475,2.173751,2.942921,3.651044,4.489940,5.315549,5.913909,...,0.458760,0.383626,0.465648,0.486191,0.476767,14.563300,65.485651,165.726434,0.263580,1.428605


In [38]:
df_ordered.to_csv('features/Descriptors/Train_3d_padel_curated.csv', index=False)

In [39]:
#3d test padel descriptors
df_test = pd.read_csv('features/Descriptors/Test_3d_padel.csv')
df_test['ID'] = df_test['Name'].str.extract(r'_(\d+)$')
df_test['ID'] = df_test['ID'].astype(int)
df_test = df_test.drop('Name',axis=1)
df_test = df_test.fillna(0)
df_test

,TDB1u,TDB2u,TDB3u,TDB4u,TDB5u,TDB6u,TDB7u,TDB8u,TDB9u,TDB10u,...,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds,ID
0,1.256465,2.171115,3.043943,3.760309,4.464387,5.360304,6.219546,7.167544,7.765691,8.151008,...,0.376568,0.495379,0.498255,0.389493,19.616949,109.504153,270.665477,0.358914,1.383127,1024
1,1.265921,2.192010,3.014118,3.767711,4.629879,5.383726,6.098741,6.841896,7.628198,8.337366,...,0.383523,0.483017,0.456354,0.359474,30.083409,263.624132,866.546679,0.338206,1.298845,1058
2,1.264041,2.193906,3.011775,3.780729,4.637621,5.404075,6.155364,7.001784,7.676756,8.464960,...,0.319468,0.467477,0.469673,0.370640,30.915888,261.015244,817.548113,0.378132,1.307790,1062
3,1.261741,2.193148,3.012591,3.756058,4.654545,5.421353,6.162123,6.812761,7.378946,7.913688,...,0.335013,0.418961,0.461067,0.360849,27.589955,241.499845,933.836601,0.191420,1.240877,1071
4,1.261087,2.180244,3.015841,3.750559,4.616401,5.379107,6.027330,6.841333,7.588789,8.276007,...,0.403953,0.476127,0.437525,0.359561,29.703394,273.107532,1030.822024,0.265360,1.273213,1059
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1387,1.261538,2.191765,3.000769,3.732591,4.617266,5.391543,6.138224,6.794698,7.310541,7.855419,...,0.344405,0.473503,0.513797,0.330815,26.279190,221.520662,841.442417,0.175834,1.318114,8487
1388,1.262673,2.188731,3.013727,3.752963,4.564081,5.340549,6.100493,6.728522,7.285962,7.809849,...,0.329969,0.498479,0.449435,0.381066,26.447825,217.920793,796.536481,0.223900,1.328980,8475
1389,1.256855,2.158995,2.997735,3.724414,4.521534,5.337197,6.108157,6.853174,7.521999,8.523547,...,0.427552,0.506159,0.482429,0.417011,29.241594,233.292447,562.749041,0.418720,1.405599,947
1390,1.261310,2.176234,3.004790,3.747631,4.622944,5.438962,6.127827,6.864541,7.591081,8.401877,...,0.405306,0.462813,0.495558,0.414682,42.948201,535.160216,2154.556991,0.351409,1.373053,7054


In [40]:
df = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc.csv')
df

/tmp/ipykernel_3612489/354123813.py:1: DtypeWarning: Columns (1058,1060,1081,1137,1139,1160,1362,1363,1364,1366,1367,1380) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc.csv')


,ID,SMILES,Permeability,ABCIndex,ABCGGIndex,AcidicGroupCount,BasicGroupCount,AdjacencyMatrix,AdjacencyMatrix.1,AdjacencyMatrix.2,...,WalkCount.19,WalkCount.20,Weight,Weight.1,WienerIndex,WienerIndex.1,ZagrebIndex,ZagrebIndex.1,ZagrebIndex.2,ZagrebIndex.3
0,908,CC[C@H](C)[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[...,-7.00,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,155.498239,2.424408,4.847496,...,11.617204,168.482860,1776.076051,6.529691,111779,220,626.0,728.0,55.444444,28.277778
1,923,CC[C@H](C)[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](...,-7.00,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,150.138684,2.427131,4.854261,...,11.589822,165.351205,1724.125588,6.338697,99910,213,610.0,706.0,56.284722,27.402778
2,897,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.00,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,150.676716,2.420176,4.840351,...,11.578086,164.308833,1700.068073,6.464137,103927,212,606.0,704.0,53.222222,27.333333
3,587,CC(C)C[C@@H]1NC(=O)[C@H](Cc2c[nH]cn2)NC(=O)[C@...,-6.74,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,151.698924,2.415398,4.830623,...,11.527095,177.329996,1685.010893,6.633901,95572,199,610.0,701.0,47.777778,27.083333
4,921,CC[C@H](C)[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[...,-5.54,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,145.614560,2.422204,4.844100,...,11.547955,161.156358,1668.062988,6.415627,93670,207,588.0,682.0,53.972222,26.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1387,2481,CC(C)C[C@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2ccccc...,-4.50,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,38.436032,2.302525,4.605049,...,9.885069,65.694305,430.294391,6.236151,2750,44,146.0,161.0,11.138889,7.083333
1388,2485,CC(C)C[C@H]1NC(=O)[C@H](C)NC(=O)[C@@H](Cc2cccc...,-4.80,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,38.436032,2.302525,4.605049,...,9.885069,65.694305,430.294391,6.236151,2750,44,146.0,161.0,11.138889,7.083333
1389,5604,CC(C)CN1CC(=O)N[C@@H](Cc2ccccc2)C(=O)NCCCCC(=O...,-6.38,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,37.855097,2.317338,4.629714,...,9.940542,65.841858,430.258006,6.619354,2644,45,148.0,164.0,11.750000,7.000000
1390,2513,C[C@H]1NCCCCCCNC(=O)[C@H](Cc2ccc(O)cc2)NC(=O)[...,-7.80,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,39.745462,2.403441,4.738314,...,10.079414,79.819128,430.258006,6.619354,2650,47,154.0,176.0,10.250000,6.972222


In [41]:
merged_df = df_test.merge(df[['ID', 'SMILES', 'Permeability']], on='ID', how='left')
merged_df = merged_df[['ID', 'SMILES', 'Permeability'] + [col for col in merged_df.columns if col not in ['ID', 'SMILES', 'Permeability']]]
merged_df

,ID,SMILES,Permeability,TDB1u,TDB2u,TDB3u,TDB4u,TDB5u,TDB6u,TDB7u,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,1024,C[C@H]1C(=O)N(C)[C@H](C)C(=O)N[C@H](C)C(=O)N(C...,-7.100,1.256465,2.171115,3.043943,3.760309,4.464387,5.360304,6.219546,...,0.529375,0.376568,0.495379,0.498255,0.389493,19.616949,109.504153,270.665477,0.358914,1.383127
1,1058,CC(C)C[C@H]1C(=O)N(C)[C@@H](CC(C)C)C(=O)N[C@H]...,-5.950,1.265921,2.192010,3.014118,3.767711,4.629879,5.383726,6.098741,...,0.508614,0.383523,0.483017,0.456354,0.359474,30.083409,263.624132,866.546679,0.338206,1.298845
2,1062,CC(C)C[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[C@H]...,-6.540,1.264041,2.193906,3.011775,3.780729,4.637621,5.404075,6.155364,...,0.585422,0.319468,0.467477,0.469673,0.370640,30.915888,261.015244,817.548113,0.378132,1.307790
3,1071,CC(C)C[C@@H]1NC(=O)[C@H](Cc2cc3ccccc3[nH]2)NC(...,-5.780,1.261741,2.193148,3.012591,3.756058,4.654545,5.421353,6.162123,...,0.459267,0.335013,0.418961,0.461067,0.360849,27.589955,241.499845,933.836601,0.191420,1.240877
4,1059,CC(C)C[C@H]1C(=O)N2CCC[C@H]2C(=O)N[C@@H](Cc2cc...,-5.730,1.261087,2.180244,3.015841,3.750559,4.616401,5.379107,6.027330,...,0.439620,0.403953,0.476127,0.437525,0.359561,29.703394,273.107532,1030.822024,0.265360,1.273213
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1387,8487,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccc(O)cc2)NC(=O)[C...,-6.495,1.261538,2.191765,3.000769,3.732591,4.617266,5.391543,6.138224,...,0.439484,0.344405,0.473503,0.513797,0.330815,26.279190,221.520662,841.442417,0.175834,1.318114
1388,8475,CC(C)C[C@@H]1NC(=O)[C@@H](CC(C)C)N(C)C(=O)[C@H...,-5.960,1.262673,2.188731,3.013727,3.752963,4.564081,5.340549,6.100493,...,0.482600,0.329969,0.498479,0.449435,0.381066,26.447825,217.920793,796.536481,0.223900,1.328980
1389,947,CC[C@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(=O)[C@...,-5.340,1.256855,2.158995,2.997735,3.724414,4.521534,5.337197,6.108157,...,0.518262,0.427552,0.506159,0.482429,0.417011,29.241594,233.292447,562.749041,0.418720,1.405599
1390,7054,CC(=O)N[C@@H]1C(=O)N2CCC[C@H]2C(=O)N(C)[C@H](C...,-5.320,1.261310,2.176234,3.004790,3.747631,4.622944,5.438962,6.127827,...,0.495633,0.405306,0.462813,0.495558,0.414682,42.948201,535.160216,2154.556991,0.351409,1.373053


In [42]:
df_ordered = merged_df.merge(df[['ID']], on='ID', how='right')
df_ordered = df_ordered.reindex(df.index)
df_ordered

,ID,SMILES,Permeability,TDB1u,TDB2u,TDB3u,TDB4u,TDB5u,TDB6u,TDB7u,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,908,CC[C@H](C)[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[...,-7.00,1.263617,2.188920,3.018525,3.748227,4.527683,5.360596,6.040785,...,0.767875,0.179312,0.502200,0.441173,0.380319,98.965167,1838.483001,8985.839378,0.651812,1.323692
1,923,CC[C@H](C)[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](...,-7.00,1.258121,2.178721,3.009466,3.712720,4.485822,5.322303,5.985906,...,0.585474,0.346546,0.520768,0.527965,0.365485,81.328539,1761.081455,9261.968931,0.398030,1.414218
2,897,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,587,CC(C)C[C@@H]1NC(=O)[C@H](Cc2c[nH]cn2)NC(=O)[C@...,-6.74,1.263921,2.197874,3.031545,3.773258,4.584405,5.393252,6.127764,...,0.591414,0.326538,0.433409,0.441889,0.423103,80.003796,1718.149538,9911.992077,0.387121,1.298401
4,921,CC[C@H](C)[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[...,-5.54,1.257514,2.180824,3.007405,3.726490,4.515325,5.362363,6.114876,...,0.611991,0.343470,0.568474,0.538717,0.354893,86.672490,1898.733289,8081.078328,0.433191,1.462084
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1387,2481,CC(C)C[C@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2ccccc...,-4.50,1.253144,2.184557,2.967642,3.700838,4.464691,5.222626,5.860924,...,0.719948,0.221983,0.461292,0.478911,0.410452,21.264177,96.995284,207.488727,0.579923,1.350655
1388,2485,CC(C)C[C@H]1NC(=O)[C@H](C)NC(=O)[C@@H](Cc2cccc...,-4.80,1.253560,2.186175,2.966364,3.703335,4.517522,5.290943,6.059905,...,0.712577,0.226527,0.474584,0.463075,0.429658,22.925205,114.891434,256.251692,0.568866,1.367317
1389,5604,CC(C)CN1CC(=O)N[C@@H](Cc2ccccc2)C(=O)NCCCCC(=O...,-6.38,1.256889,2.195213,2.985593,3.709740,4.648743,5.383893,6.093295,...,0.544506,0.389352,0.501813,0.459884,0.479810,20.788934,118.318494,265.091453,0.400788,1.441507
1390,2513,C[C@H]1NCCCCCCNC(=O)[C@H](Cc2ccc(O)cc2)NC(=O)[...,-7.80,1.265159,2.203653,2.993652,3.786291,4.648162,5.437906,6.132583,...,0.554492,0.350152,0.404760,0.473007,0.444278,19.365724,105.166185,258.994679,0.356966,1.322045


In [43]:
df_ordered.to_csv('features/Descriptors/Test_3d_padel_curated.csv', index=False)

In [44]:
#2d Padel descriptors
df_train = pd.read_csv('features/Descriptors/Train_2d_padel_curated.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_padel_curated.csv')
df_test = df_test.dropna()
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (5568, 1444)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 1444)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.109659 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 229913
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 1037
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.108495 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 229911
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 1037
[LightGBM] [Info] Start training from score -5.744959
[LightGBM] [Info] Auto-choosing col-wise multi-threa

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2065,0.3338,0.4544,0.6684,0.8180,0.7853,0.2091,0.3294,0.4573,0.6711,0.8201,0.8016
DecisionTreeRegressor,0.3057,0.3879,0.5529,0.5090,0.7328,0.7129,0.2516,0.3538,0.5016,0.6042,0.7797,0.7640
RandomForestRegressor,0.2164,0.3430,0.4651,0.6525,0.8079,0.7750,0.2228,0.3409,0.4720,0.6496,0.8063,0.7861
GradientBoostingRegressor,0.2207,0.3497,0.4698,0.6455,0.8059,0.7657,0.2252,0.3453,0.4746,0.6458,0.8072,0.7824
AdaBoostRegressor,0.3797,0.4946,0.6162,0.3901,0.6579,0.6210,0.3695,0.4836,0.6079,0.4188,0.6834,0.6753
XGBRegressor,0.2184,0.3430,0.4674,0.6492,0.8062,0.7720,0.2127,0.3312,0.4612,0.6654,0.8159,0.7961
ExtraTreesRegressor,0.2120,0.3403,0.4604,0.6596,0.8121,0.7813,0.2229,0.3398,0.4722,0.6493,0.8060,0.7854
LinearRegression,0.4501,0.4210,0.6709,0.2771,0.6640,0.7256,0.3887,0.3904,0.6234,0.3887,0.6932,0.7576
KNeighborsRegressor,0.2721,0.3805,0.5217,0.5629,0.7570,0.7200,0.2708,0.3744,0.5203,0.5741,0.7615,0.7468
SVR,0.2245,0.3398,0.4738,0.6394,0.8030,0.7735,0.2357,0.3442,0.4855,0.6293,0.7974,0.7778


In [45]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.748635769695707, -7.011964823583876, -6.10...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.954111188135858, -6.367386692896654, -6.9...","[-6.956350399856309, -6.506628545345231, -6.79...","[0.08406140027189811, 0.10822893239959114, 0.1..."
1,DecisionTreeRegressor,"[-7.0, -6.96, -5.89, -4.77, -5.09, -5.85, -5.7...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.14, -7.0, -6.85, -7.14, -5.62, -5.15, -6....","[-6.892, -6.601999999999999, -6.77000000000000...","[0.19620397549489138, 0.5154570787175203, 0.24..."
2,RandomForestRegressor,"[-6.68751698384667, -6.862200000000001, -5.876...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.858300000000001, -6.324199999999999, -6.7...","[-6.8408000000000015, -6.434270468620001, -6.7...","[0.028751347794494276, 0.10010356977581633, 0...."
3,GradientBoostingRegressor,"[-7.049422858919388, -7.147010884658419, -6.02...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.951227068886026, -6.032477857197904, -6.8...","[-6.981714209497238, -6.276449892893166, -6.74...","[0.10107874725030874, 0.24112945669168437, 0.0..."
4,AdaBoostRegressor,"[-5.7551578947368425, -5.872286852091592, -5.7...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.872286852091592, -5.6988704693241266, -5....","[-5.79603272411477, -5.641744415968907, -5.679...","[0.05831065145361535, 0.05609407832570005, 0.0..."
5,XGBRegressor,"[-6.7539024, -6.8032804, -6.6862526, -5.050490...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.2862964, -6.2903624, -7.0353165, -6.62834...","[-7.1721354, -6.479604, -6.917823, -6.5554595,...","[0.1477116, 0.12707025, 0.22393642, 0.22948918..."
6,ExtraTreesRegressor,"[-6.684034245075001, -6.779600000000001, -6.18...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.943200000000001, -6.506450000000002, -6.8...","[-6.93241, -6.498234218461455, -6.87662, -6.43...","[0.02072940906055988, 0.09884156122215218, 0.0..."
7,LinearRegression,"[-7.153058696611124, -7.07600376993666, -5.857...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-8.58458272217119, -5.106589204850179, -7.33...","[-8.040778367860366, -5.93491623070322, -7.513...","[0.5454599056812761, 0.6034196920331906, 0.292..."
8,KNeighborsRegressor,"[-6.156666666666666, -6.986666666666667, -5.88...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.986666666666667, -6.503333333333333, -7.0...","[-6.989333333333333, -6.469333333333333, -6.82...","[0.005333333333333102, 0.4431373000173502, 0.2..."
9,SVR,"[-6.934465611188575, -6.889854669845263, -5.70...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.4013267862398555, -6.655377641323549, -7....","[-7.35947626872543, -6.698621922978845, -7.039...","[0.09555579260429996, 0.11422767098606637, 0.0..."


In [46]:
result_df.to_csv('Results/Descriptors/Results_2D_padel.csv')
prediction_df.to_csv('Results/Descriptors/Prediction_data_2D_padel.csv')


In [47]:
#2d padel descriptors const removal
df_train = pd.read_csv('features/Descriptors/Train_2d_padel_curated.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train, const_col = remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_padel_curated.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = X_test.drop(const_col,axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (5568, 1094)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 1094)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.118399 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 229913
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 1037
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.116712 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 229911
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 1037
[LightGBM] [Info] Start training from score -5.744959
[LightGBM] [Info] Auto-choosing col-wise multi-threa

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2065,0.3338,0.4544,0.6684,0.8180,0.7853,0.2091,0.3294,0.4573,0.6711,0.8201,0.8016
DecisionTreeRegressor,0.2999,0.3850,0.5476,0.5183,0.7379,0.7166,0.2417,0.3484,0.4916,0.6198,0.7886,0.7672
RandomForestRegressor,0.2166,0.3431,0.4654,0.6521,0.8076,0.7747,0.2230,0.3411,0.4722,0.6493,0.8061,0.7858
GradientBoostingRegressor,0.2207,0.3498,0.4697,0.6456,0.8059,0.7654,0.2253,0.3454,0.4747,0.6456,0.8071,0.7822
AdaBoostRegressor,0.3839,0.4972,0.6196,0.3834,0.6540,0.6160,0.3696,0.4841,0.6079,0.4187,0.6835,0.6700
XGBRegressor,0.2184,0.3430,0.4674,0.6492,0.8062,0.7720,0.2127,0.3312,0.4612,0.6654,0.8159,0.7961
ExtraTreesRegressor,0.2144,0.3415,0.4631,0.6556,0.8097,0.7799,0.2232,0.3400,0.4724,0.6489,0.8057,0.7853
LinearRegression,0.4501,0.4210,0.6709,0.2771,0.6640,0.7256,0.3887,0.3904,0.6234,0.3887,0.6932,0.7576
KNeighborsRegressor,0.2717,0.3805,0.5212,0.5637,0.7574,0.7204,0.2727,0.3767,0.5222,0.5710,0.7595,0.7421
SVR,0.2245,0.3398,0.4738,0.6395,0.8030,0.7734,0.2357,0.3442,0.4855,0.6293,0.7974,0.7778


In [48]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.748635769695707, -7.011964823583876, -6.10...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.954111188135858, -6.367386692896654, -6.9...","[-6.956350399856309, -6.506628545345231, -6.79...","[0.08406140027189811, 0.10822893239959114, 0.1..."
1,DecisionTreeRegressor,"[-6.24, -7.0, -6.244999999999999, -5.05, -5.66...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.14, -6.24, -7.0, -7.0, -4.72, -5.05, -6.8...","[-6.944, -6.4719999999999995, -6.478, -6.698, ...","[0.15869467539901885, 0.43595412602703976, 0.7..."
2,RandomForestRegressor,"[-6.650800000000003, -6.841400000000002, -5.87...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.873900000000003, -6.335900000000001, -6.7...","[-6.822980000000001, -6.439100000000001, -6.70...","[0.045854875422358894, 0.11401727939220462, 0...."
3,GradientBoostingRegressor,"[-7.049422858919388, -7.147010884658418, -6.02...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.951227068886026, -6.032477857197904, -6.8...","[-6.981714209497238, -6.277971173161731, -6.72...","[0.10107874725030862, 0.24386791317603904, 0.0..."
4,AdaBoostRegressor,"[-5.6282765451801104, -5.755157894736841, -5.6...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.755157894736841, -5.570006479709387, -5.7...","[-5.784487357410014, -5.620410412023963, -5.69...","[0.07788979162441326, 0.030163314719644056, 0...."
5,XGBRegressor,"[-6.7539024, -6.8032804, -6.6862526, -5.050490...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.2862964, -6.2903624, -7.0353165, -6.62834...","[-7.1721354, -6.479604, -6.917823, -6.5554595,...","[0.1477116, 0.12707025, 0.22393642, 0.22948918..."
6,ExtraTreesRegressor,"[-6.731500000000002, -6.920400000000001, -6.08...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.9544500000000005, -6.527900000000003, -6....","[-6.9190900000000015, -6.46036, -6.87019000000...","[0.028923492181961132, 0.15551453436897866, 0...."
7,LinearRegression,"[-7.153058698565211, -7.07600377233458, -5.857...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-8.584582722584592, -5.106589208863362, -7.3...","[-8.040778364661062, -5.934916230268508, -7.51...","[0.5454599078839653, 0.603419691139284, 0.2921..."
8,KNeighborsRegressor,"[-6.156666666666666, -6.986666666666667, -5.88...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.986666666666667, -6.503333333333333, -7.0...","[-6.989333333333333, -6.469333333333333, -6.82...","[0.005333333333333102, 0.4431373000173502, 0.2..."
9,SVR,"[-6.934631320750119, -6.890011323868332, -5.70...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.401519844122166, -6.655425649497315, -7.1...","[-7.359586718950572, -6.698786596614786, -7.03...","[0.09564655076261316, 0.1140427502123015, 0.04..."


In [49]:
result_df.to_csv('Results/Descriptors/Results_2D_padel_const_rem.csv')
prediction_df.to_csv('Results/Descriptors/Prediction_data_2D_padel_const_rem.csv')

In [50]:
#2d padel descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_2d_padel_curated.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train, const_col = remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_padel_curated.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = X_test.drop(const_col,axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
results_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)


X_train shape:  (5568, 726)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 726)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.078160 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 151170
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 715
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.081625 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 151151
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 715
[LightGBM] [Info] Start training from score -5.744959
[LightGBM] [Info] Auto-choosing col-wise multi-threading

In [51]:
results_df

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2076,0.3359,0.4556,0.6666,0.8168,0.7833,0.2088,0.3298,0.4570,0.6716,0.8205,0.8029
DecisionTreeRegressor,0.3069,0.3873,0.5540,0.5071,0.7355,0.7166,0.2456,0.3532,0.4955,0.6138,0.7850,0.7654
RandomForestRegressor,0.2167,0.3438,0.4655,0.6519,0.8075,0.7747,0.2224,0.3406,0.4716,0.6502,0.8067,0.7872
GradientBoostingRegressor,0.2260,0.3521,0.4754,0.6370,0.8008,0.7617,0.2254,0.3465,0.4748,0.6454,0.8074,0.7795
AdaBoostRegressor,0.3876,0.5012,0.6226,0.3775,0.6524,0.6102,0.3691,0.4862,0.6075,0.4195,0.6890,0.6772
XGBRegressor,0.2157,0.3420,0.4645,0.6535,0.8089,0.7756,0.2127,0.3307,0.4612,0.6654,0.8159,0.7934
ExtraTreesRegressor,0.2105,0.3394,0.4588,0.6619,0.8136,0.7824,0.2201,0.3378,0.4691,0.6538,0.8087,0.7890
LinearRegression,0.3250,0.3862,0.5701,0.4780,0.7271,0.7434,0.3238,0.3791,0.5690,0.4907,0.7288,0.7690
KNeighborsRegressor,0.2743,0.3829,0.5237,0.5595,0.7556,0.7213,0.2718,0.3762,0.5213,0.5725,0.7607,0.7450
SVR,0.2318,0.3445,0.4814,0.6277,0.7959,0.7672,0.2399,0.3472,0.4898,0.6227,0.7938,0.7738


In [52]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.877540262011381, -6.9401193384424955, -6.0...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.89882107802736, -6.380349875715229, -6.89...","[-6.9418341999909305, -6.437458134968256, -6.7...","[0.10929102757570498, 0.07674694355792538, 0.1..."
1,DecisionTreeRegressor,"[-7.0, -6.96, -6.244999999999999, -5.15, -5.92...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.13, -7.0, -6.89, -7.13, -5.04, -5.05, -6....","[-6.888, -6.572, -6.846000000000001, -6.786, -...","[0.21646246787838308, 0.5596570378365665, 0.25..."
2,RandomForestRegressor,"[-6.684883333333333, -6.727100000000004, -5.84...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.883100000000002, -6.34215, -6.77030000000...","[-6.818088000000003, -6.41619, -6.760375000000...","[0.04595719504060266, 0.07991127830287756, 0.0..."
3,GradientBoostingRegressor,"[-6.8840723700689095, -6.826898840146733, -6.1...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.81425556403836, -5.927446593915589, -6.83...","[-6.968253819354184, -6.152169898735246, -6.72...","[0.18235359132380244, 0.23292608279291382, 0.1..."
4,AdaBoostRegressor,"[-5.811559714224488, -6.172577599897636, -5.69...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.830068994817486, -5.580872452564586, -5.7...","[-5.928738064150819, -5.649077743033617, -5.78...","[0.07946684589357647, 0.08194540878370628, 0.0..."
5,XGBRegressor,"[-6.568617, -7.142002, -6.170573, -5.8462305, ...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.33181, -6.3768473, -7.142535, -6.7683806,...","[-7.047418, -6.2930164, -7.012419, -6.402696, ...","[0.20126063, 0.40105987, 0.087599926, 0.376910..."
6,ExtraTreesRegressor,"[-6.781360000000002, -6.835616057605002, -5.97...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.969000000000002, -6.510800000000002, -6.8...","[-6.931480000000002, -6.539850000000001, -6.89...","[0.019725759807926756, 0.06977602740196588, 0...."
7,LinearRegression,"[-7.701398342181804, -7.565047633404632, -5.95...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-8.845920650802967, -7.132296085575675, -7.5...","[-8.600805287011992, -6.6329682804337455, -7.5...","[0.23982720389071688, 0.30739036057718994, 0.1..."
8,KNeighborsRegressor,"[-6.156666666666666, -6.986666666666667, -5.88...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.986666666666667, -6.503333333333333, -7.0...","[-6.989333333333333, -6.624, -6.91200000000000...","[0.005333333333333102, 0.15548347536349721, 0...."
9,SVR,"[-6.926286875544857, -6.880064167856939, -5.66...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.358312561014705, -6.673857585291174, -7.1...","[-7.29355867130121, -6.7397447292470005, -7.05...","[0.10222043090732864, 0.09112631382221116, 0.0..."


In [53]:
results_df.to_csv('Results/Descriptors/Results_2D_padel_LVR.csv')
prediction_df.to_csv('Results/Descriptors/Prediction_data_2D_padel_const_LVR.csv')

In [54]:
#2d All descriptors
df_train_padel = pd.read_csv('features/Descriptors/Train_2d_padel_curated.csv')
df_train_rdkit = pd.read_csv('features/Descriptors/Train_2d_RDKit_des.csv')
df_train_mordred = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc.csv')

df_2d_train = df_train_rdkit.merge(df_train_mordred, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_train_padel, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_2d_train

/tmp/ipykernel_3612489/2724054456.py:4: DtypeWarning: Columns (1058,1060,1075,1081,1137,1139,1154,1160,1362,1363,1364,1366,1367,1378,1379,1380) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train_mordred = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc.csv')


,ID,SMILES,Permeability,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,AMW,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb
0,915,CC[C@H](C)[C@H](NC(=O)[C@@H]1CC(=O)N[C@@H](Cc2...,-7.0,15.738544,15.738544,0.010382,-1.908222,0.047997,24.622047,1773.325,...,6.420745,250.045897,1.968865,90.654326,43.537099,47.117227,107741.0,214.0,11.490,630.0
1,888,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1cccc(Cl)c1)N(C)...,-7.0,15.975705,15.975705,0.027671,-1.943827,0.026511,21.983740,1745.057,...,6.835048,244.597932,1.988601,87.647449,38.186244,44.409996,106182.0,208.0,10.250,620.0
2,593,C/N=C(\NC)NCCC[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C...,-7.0,15.828473,15.828473,0.049834,-1.862294,0.021075,22.464000,1733.267,...,6.511595,248.034411,1.984275,88.350956,35.707994,52.642962,102214.0,204.0,10.960,616.0
3,916,CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@@H]1CC(=...,-7.0,15.595105,15.595105,0.004954,-1.811756,0.069603,24.479675,1725.281,...,6.338697,240.980957,1.959195,90.761838,43.488529,47.273309,101212.0,208.0,9.243,608.0
4,900,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)N(C)C(=O...,-7.0,15.867592,15.867592,0.029192,-1.876686,0.046796,23.089431,1723.309,...,6.285206,240.843065,1.958074,87.967406,40.534498,47.432908,107844.0,215.0,10.212,608.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5563,2469,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-4.7,12.835172,12.835172,0.168936,-0.728726,0.606745,24.965517,402.539,...,6.385128,57.707840,1.989926,19.684197,7.618398,12.065799,2286.0,42.0,3.395,138.0
5564,2467,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-5.6,12.673271,12.673271,0.196704,-0.722720,0.611202,25.037037,374.485,...,6.565470,53.707230,1.989157,19.692329,7.619967,12.072362,1880.0,40.0,2.679,130.0
5565,2512,CC(C)C[C@H]1NC(=O)[C@@H](C)NCCCCCCNC(=O)[C@H](...,-5.5,12.572226,12.572226,0.181773,-1.016186,0.465702,27.576923,370.494,...,6.170967,50.804604,1.954023,22.049967,10.061855,11.988112,1648.0,37.0,2.588,118.0
5566,2511,CC(C)[C@@H]1NC(=O)[C@@H](CO)NC(=O)[C@@H](C)NCC...,-6.3,12.364448,12.364448,0.112529,-1.084532,0.446971,28.200000,356.467,...,6.249866,48.797056,1.951882,22.005013,10.045259,11.959755,1472.0,37.0,1.808,114.0


In [55]:
df_2d_train.to_csv('features/Descriptors/Train_2d_all_descriptors.csv', index=False)

In [56]:
df_test_padel = pd.read_csv('features/Descriptors/Test_2d_padel_curated.csv')
df_test_rdkit = pd.read_csv('features/Descriptors/Test_2d_RDKit_des.csv')
df_test_mordred = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc.csv')

df_2d_test = df_test_rdkit.merge(df_test_mordred, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_test_padel, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_2d_test

/tmp/ipykernel_3612489/1932829997.py:3: DtypeWarning: Columns (1058,1060,1081,1137,1139,1160,1362,1363,1364,1366,1367,1380) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test_mordred = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc.csv')


,ID,SMILES,Permeability,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,AMW,WTPT-1,WTPT-2,WTPT-3,WTPT-4,WTPT-5,WPATH,WPOL,XLogP,Zagreb
0,908,CC[C@H](C)[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[...,-7.00,15.806942,15.806942,0.022547,-1.946094,0.037676,23.507937,1777.744,...,6.529691,247.936262,1.967748,90.383811,40.574663,47.283631,111779.0,220.0,10.264,626.0
1,923,CC[C@H](C)[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](...,-7.00,15.512854,15.512854,0.072929,-1.849230,0.082004,25.902439,1725.281,...,6.338697,240.768109,1.957464,90.840338,43.433503,47.406835,99910.0,213.0,9.667,610.0
2,897,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.00,16.029525,16.029525,0.042154,-1.949385,0.046040,22.827869,1701.218,...,6.464137,240.251303,1.969273,87.601926,40.568304,47.033622,103927.0,212.0,9.469,606.0
3,587,CC(C)C[C@@H]1NC(=O)[C@H](Cc2c[nH]cn2)NC(=O)[C@...,-6.74,15.776936,15.776936,0.046352,-1.865645,0.035370,22.622951,1686.166,...,6.633901,242.969389,1.991552,85.904257,35.712474,50.191783,95572.0,199.0,10.338,610.0
4,921,CC[C@H](C)[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[...,-5.54,15.432735,15.432735,0.040020,-1.845687,0.100132,26.663866,1669.173,...,6.415627,233.157008,1.959303,90.305554,42.950496,47.355058,93670.0,207.0,9.346,588.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1387,2481,CC(C)C[C@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2ccccc...,-4.50,12.958074,12.958074,0.156212,-0.733388,0.586385,24.903226,430.593,...,6.236151,61.708009,1.990581,19.682164,7.618006,12.064158,2750.0,44.0,4.111,146.0
1388,2485,CC(C)C[C@H]1NC(=O)[C@H](C)NC(=O)[C@@H](Cc2cccc...,-4.80,12.958074,12.958074,0.156212,-0.733388,0.586385,24.903226,430.593,...,6.236151,61.708009,1.990581,19.682164,7.618006,12.064158,2750.0,44.0,4.111,146.0
1389,5604,CC(C)CN1CC(=O)N[C@@H](Cc2ccccc2)C(=O)NCCCCC(=O...,-6.38,12.898424,12.898424,0.143657,-0.742398,0.662387,22.258065,430.549,...,6.619354,61.526924,1.984739,22.541021,10.153913,12.387108,2644.0,45.0,1.941,148.0
1390,2513,C[C@H]1NCCCCCCNC(=O)[C@H](Cc2ccc(O)cc2)NC(=O)[...,-7.80,13.095577,13.095577,0.065481,-0.737917,0.561202,26.806452,430.549,...,6.619354,62.566820,2.018285,22.680885,10.197129,12.483757,2650.0,47.0,1.336,154.0


In [57]:
df_2d_test.to_csv('features/Descriptors/Test_2d_all_descriptors.csv', index=False)

In [58]:
#2d All descriptors
df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_all_descriptors.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
# X_test = X_test.select_dtypes(include=['number'])
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

/tmp/ipykernel_3612489/1766064782.py:2: DtypeWarning: Columns (1275,1277,1280,1285,1292,1298,1304,1354,1356,1359,1364,1371,1377,1383,1579,1580,1581,1583,1584,1590,1595,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors.csv')


X_train shape:  (5568, 3087)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX


/tmp/ipykernel_3612489/1766064782.py:10: DtypeWarning: Columns (1275,1277,1292,1298,1354,1356,1371,1377,1579,1580,1581,1583,1584,1595,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test = pd.read_csv('features/Descriptors/Test_2d_all_descriptors.csv')


X_test shape:  (1392, 3087)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.257359 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 514720
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 2380
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.278556 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 514769
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 2385
[LightGBM] [Info] Start training from score -5.744959
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.249411 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] T

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2035,0.3322,0.4512,0.6731,0.8208,0.7880,0.2067,0.3275,0.4547,0.6748,0.8223,0.8024
DecisionTreeRegressor,0.2969,0.3825,0.5449,0.5231,0.7420,0.7207,0.2331,0.3451,0.4828,0.6334,0.7967,0.7757
RandomForestRegressor,0.2144,0.3424,0.4631,0.6556,0.8098,0.7771,0.2199,0.3389,0.4689,0.6542,0.8092,0.7889
GradientBoostingRegressor,0.2192,0.3485,0.4682,0.6479,0.8074,0.7677,0.2209,0.3438,0.4700,0.6525,0.8115,0.7822
AdaBoostRegressor,0.3758,0.4938,0.6130,0.3965,0.6651,0.6201,0.3643,0.4819,0.6036,0.4270,0.6902,0.6734
XGBRegressor,0.2158,0.3413,0.4646,0.6534,0.8089,0.7763,0.2110,0.3290,0.4593,0.6681,0.8176,0.7959
ExtraTreesRegressor,0.2087,0.3381,0.4568,0.6648,0.8154,0.7838,0.2181,0.3359,0.4670,0.6569,0.8107,0.7898
LinearRegression,1.1639,0.6263,1.0789,-0.8695,0.4285,0.5712,0.5250,0.4697,0.7245,0.1743,0.6071,0.6696
KNeighborsRegressor,0.2728,0.3812,0.5223,0.5619,0.7561,0.7210,0.2708,0.3761,0.5204,0.5740,0.7610,0.7465
SVR,0.2226,0.3385,0.4718,0.6425,0.8048,0.7764,0.2331,0.3444,0.4828,0.6334,0.8001,0.7791


In [59]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.539461088078157, -6.803531350334017, -6.09...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.97358933898271, -6.067327614470404, -7.01...","[-6.959196594392329, -6.502676262743206, -6.82...","[0.09860569343482871, 0.22735345425253795, 0.1..."
1,DecisionTreeRegressor,"[-7.0, -7.0, -6.244999999999999, -5.05, -6.244...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.13, -7.0, -7.0, -7.0, -4.74, -5.89, -6.85...","[-6.773999999999999, -7.0, -6.676, -6.698, -5....","[0.481730214954387, 0.0, 0.6087889617921797, 0..."
2,RandomForestRegressor,"[-6.64648935393, -6.856800000000001, -5.949225...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.795799999999999, -6.303139353930002, -6.6...","[-6.763413333333334, -6.483087870785999, -6.70...","[0.05407051532746641, 0.09816706514747413, 0.0..."
3,GradientBoostingRegressor,"[-6.9540434903844535, -6.907221734458648, -5.6...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.012626489538526, -6.152884125714732, -6.8...","[-7.1588226729921285, -6.341101212156845, -6.7...","[0.11496616723171289, 0.19093286239462887, 0.1..."
4,AdaBoostRegressor,"[-6.260360360360363, -5.8487187267137655, -5.6...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.8487187267137655, -5.657363611479575, -5....","[-5.9847411657106235, -5.698061968565483, -5.7...","[0.14376578681722224, 0.057848383438823224, 0...."
5,XGBRegressor,"[-6.817078, -6.047762, -6.0193253, -5.507966, ...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.2019434, -6.598266, -6.849673, -6.5706487...","[-7.1478257, -6.6694136, -6.930481, -6.629074,...","[0.10844771, 0.35088056, 0.16293056, 0.3154879..."
6,ExtraTreesRegressor,"[-6.654799999999999, -6.845, -6.04850000000000...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.976500000000001, -6.479100000000001, -6.9...","[-6.96313, -6.335000000000001, -6.935300000000...","[0.023421477323175253, 0.14870335234956872, 0...."
7,LinearRegression,"[-3.9, -10.0, -6.614688931127603, -7.100615030...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-3.9, -7.6880218668156886, -10.0, -5.9123612...","[-4.2535550562631395, -6.829081188032021, -9.2...","[0.7071101125262795, 2.501768568607798, 0.7458..."
8,KNeighborsRegressor,"[-6.63, -6.986666666666667, -5.88, -4.85333333...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.63, -6.746666666666667, -7.0, -5.90333333...","[-6.7780000000000005, -6.698, -6.9120000000000...","[0.18126224096595522, 0.09733333333333362, 0.1..."
9,SVR,"[-6.318796628581415, -6.948208138960834, -5.81...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.0390969526590785, -6.672015594595891, -7....","[-6.099283434420012, -6.744563780571371, -7.05...","[0.033440920838260633, 0.12952266577688742, 0...."


In [60]:
result_df.to_csv('Results/Descriptors/Results_2D_All_desc.csv')
prediction_df.to_csv('Results/Descriptors/Prediction_data_2D_All_desc.csv')

In [61]:
#2d All descriptors const rem
df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train, const_col =  remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_all_descriptors.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

/tmp/ipykernel_3612489/1936886144.py:2: DtypeWarning: Columns (1275,1277,1280,1285,1292,1298,1304,1354,1356,1359,1364,1371,1377,1383,1579,1580,1581,1583,1584,1590,1595,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors.csv')


X_train shape:  (5568, 2504)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX


/tmp/ipykernel_3612489/1936886144.py:11: DtypeWarning: Columns (1275,1277,1292,1298,1354,1356,1371,1377,1579,1580,1581,1583,1584,1595,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test = pd.read_csv('features/Descriptors/Test_2d_all_descriptors.csv')


X_test shape:  (1392, 2504)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.249497 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 514720
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 2380
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.248721 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 514769
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 2385
[LightGBM] [Info] Start training from score -5.744959
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.246558 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] T

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2035,0.3322,0.4512,0.6731,0.8208,0.7880,0.2067,0.3275,0.4547,0.6748,0.8223,0.8024
DecisionTreeRegressor,0.2903,0.3799,0.5388,0.5337,0.7474,0.7197,0.2386,0.3491,0.4884,0.6248,0.7915,0.7689
RandomForestRegressor,0.2140,0.3422,0.4626,0.6562,0.8102,0.7777,0.2200,0.3390,0.4690,0.6540,0.8091,0.7893
GradientBoostingRegressor,0.2191,0.3485,0.4681,0.6480,0.8076,0.7682,0.2210,0.3438,0.4701,0.6524,0.8114,0.7823
AdaBoostRegressor,0.3817,0.4970,0.6178,0.3869,0.6558,0.6114,0.3666,0.4830,0.6055,0.4233,0.6864,0.6648
XGBRegressor,0.2158,0.3413,0.4646,0.6534,0.8089,0.7763,0.2110,0.3290,0.4593,0.6681,0.8176,0.7959
ExtraTreesRegressor,0.2088,0.3376,0.4570,0.6646,0.8152,0.7840,0.2173,0.3356,0.4661,0.6583,0.8115,0.7905
LinearRegression,1.1691,0.6272,1.0812,-0.8777,0.4267,0.5703,0.5238,0.4688,0.7237,0.1761,0.6087,0.6699
KNeighborsRegressor,0.2726,0.3814,0.5221,0.5622,0.7562,0.7205,0.2702,0.3750,0.5198,0.5749,0.7616,0.7482
SVR,0.2226,0.3385,0.4718,0.6425,0.8048,0.7764,0.2331,0.3444,0.4828,0.6334,0.8001,0.7792


In [62]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.539461088078157, -6.803531350334017, -6.09...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.97358933898271, -6.067327614470404, -7.01...","[-6.959196594392329, -6.502676262743206, -6.82...","[0.09860569343482871, 0.22735345425253795, 0.1..."
1,DecisionTreeRegressor,"[-7.0, -6.96, -4.39, -4.77, -4.39, -4.77, -5.9...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -6.51, -5.68, -7.0, -4.43, -5.89, -6.8...","[-6.715999999999999, -6.901999999999999, -6.33...","[0.528908309634099, 0.19600000000000006, 0.594..."
2,RandomForestRegressor,"[-6.581066666666668, -6.858353846153847, -5.88...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.827600000000001, -6.259516666666669, -6.7...","[-6.782415000000002, -6.4753033333333345, -6.7...","[0.06713178010450702, 0.12043543240167187, 0.0..."
3,GradientBoostingRegressor,"[-7.039285841113938, -6.9072217344586475, -5.6...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.012626489538524, -6.152884125714731, -6.9...","[-7.170121917525461, -6.335603421086503, -6.72...","[0.13165719334922252, 0.18569180054635284, 0.1..."
4,AdaBoostRegressor,"[-6.112876026848745, -6.304680851063826, -5.62...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.304680851063826, -5.583267048192531, -5.6...","[-6.003769741342643, -5.668356588435022, -5.67...","[0.18368840609164222, 0.07677160349906306, 0.0..."
5,XGBRegressor,"[-6.817078, -6.047762, -6.0193253, -5.507966, ...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.2019434, -6.598266, -6.849673, -6.5706487...","[-7.1478257, -6.6694136, -6.930481, -6.629074,...","[0.10844771, 0.35088056, 0.16293056, 0.3154879..."
6,ExtraTreesRegressor,"[-6.804700000000001, -6.8997, -6.2081000000000...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.9938, -6.31775, -6.9558, -6.4112000000000...","[-6.96216, -6.319710000000001, -6.935460000000...","[0.0175473758721923, 0.13596906780588006, 0.03..."
7,LinearRegression,"[-3.9, -10.0, -6.61468835380947, -7.1006156212...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-3.9, -7.688019640474522, -10.0, -5.91234901...","[-4.253611991073848, -6.829080279669435, -9.24...","[0.7072239821476979, 2.5017680773012656, 0.745..."
8,KNeighborsRegressor,"[-6.63, -6.986666666666667, -5.88, -4.85333333...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.63, -6.746666666666667, -7.0, -5.90333333...","[-6.7780000000000005, -6.698, -6.9120000000000...","[0.18126224096595522, 0.09733333333333362, 0.1..."
9,SVR,"[-6.318918864486116, -6.948290956005209, -5.81...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.039152901703593, -6.672225664802366, -7.1...","[-6.09929826060578, -6.7445454792236275, -7.05...","[0.033439875133216126, 0.1295012653674868, 0.0..."


In [63]:
result_df.to_csv('Results/Descriptors/Results_2D_All_desc_const_rem.csv')
prediction_df.to_csv('Results/Descriptors/Prediction_data_2D_All_desc_const_rem.csv')

In [64]:
#2d All descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train, const_col = remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_all_descriptors.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

/tmp/ipykernel_3612489/4119621428.py:2: DtypeWarning: Columns (1275,1277,1280,1285,1292,1298,1304,1354,1356,1359,1364,1371,1377,1383,1579,1580,1581,1583,1584,1590,1595,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors.csv')


X_train shape:  (5568, 1696)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX


/tmp/ipykernel_3612489/4119621428.py:11: DtypeWarning: Columns (1275,1277,1292,1298,1354,1356,1371,1377,1579,1580,1581,1583,1584,1595,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test = pd.read_csv('features/Descriptors/Test_2d_all_descriptors.csv')


X_test shape:  (1392, 1696)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.194483 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 338487
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 1668
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.173908 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 338541
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 1673
[LightGBM] [Info] Start training from score -5.744959
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.170874 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] T

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2071,0.3344,0.4551,0.6674,0.8173,0.7851,0.2079,0.3287,0.4559,0.6730,0.8211,0.8021
DecisionTreeRegressor,0.3005,0.3825,0.5482,0.5174,0.7392,0.7212,0.2352,0.3458,0.4850,0.6300,0.7947,0.7727
RandomForestRegressor,0.2148,0.3428,0.4635,0.6549,0.8094,0.7765,0.2191,0.3390,0.4680,0.6554,0.8100,0.7895
GradientBoostingRegressor,0.2222,0.3502,0.4714,0.6430,0.8045,0.7648,0.2219,0.3448,0.4711,0.6509,0.8106,0.7823
AdaBoostRegressor,0.3757,0.4928,0.6129,0.3966,0.6614,0.6280,0.3649,0.4819,0.6041,0.4261,0.6862,0.6822
XGBRegressor,0.2145,0.3415,0.4631,0.6555,0.8104,0.7776,0.2105,0.3286,0.4588,0.6689,0.8180,0.7961
ExtraTreesRegressor,0.2071,0.3367,0.4551,0.6673,0.8169,0.7852,0.2165,0.3349,0.4653,0.6595,0.8123,0.7919
LinearRegression,0.5854,0.4566,0.7651,0.0597,0.6037,0.6970,0.3775,0.3955,0.6144,0.4063,0.6924,0.7466
KNeighborsRegressor,0.2760,0.3833,0.5254,0.5567,0.7529,0.7173,0.2691,0.3741,0.5187,0.5768,0.7624,0.7508
SVR,0.2300,0.3441,0.4795,0.6306,0.7974,0.7685,0.2380,0.3470,0.4879,0.6256,0.7953,0.7751


In [65]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.56996058153446, -6.847107573283172, -5.960...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.983420775491325, -6.1530939335646115, -6....","[-6.94071121542805, -6.498439043361185, -6.820...","[0.10019625674994523, 0.17782249089105537, 0.1..."
1,DecisionTreeRegressor,"[-6.85, -7.0, -5.66, -5.05, -4.39, -4.77, -4.8...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -6.24, -5.49, -7.0, -4.43, -5.89, -6.8...","[-6.478, -6.848000000000001, -6.40600000000000...","[0.6520245394155038, 0.30399999999999994, 0.68..."
2,RandomForestRegressor,"[-6.692200000000001, -6.8001, -5.7679283333333...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.730166666666667, -6.349183333333334, -6.7...","[-6.7423453333333345, -6.527794666666668, -6.7...","[0.07658406244411699, 0.12152607092215963, 0.1..."
3,GradientBoostingRegressor,"[-6.761586589784095, -6.902089045647594, -5.52...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.227563300898761, -5.95526043418461, -6.76...","[-7.086896011867054, -6.329231015407993, -6.69...","[0.19395700101665453, 0.23218489564606024, 0.0..."
4,AdaBoostRegressor,"[-5.865655737704917, -6.396654777873103, -5.69...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.192133615363256, -5.619955262929486, -5.8...","[-6.218160455223533, -5.810947647143365, -5.75...","[0.06135520608349629, 0.14665506876100695, 0.0..."
5,XGBRegressor,"[-6.8515882, -7.058459, -6.1940455, -5.5840745...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.1235733, -6.6881285, -7.051892, -6.791096...","[-6.88164, -6.549446, -7.004785, -6.7434325, -...","[0.1611492, 0.16963719, 0.16040054, 0.06859808..."
6,ExtraTreesRegressor,"[-6.770500000000001, -6.9204, -6.1692, -5.2986...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.9895000000000005, -6.3099, -6.9772, -6.46...","[-6.96054, -6.304240000000001, -6.956320000000...","[0.018901174566677597, 0.1455230751461783, 0.0..."
7,LinearRegression,"[-3.9, -7.995614386008469, -6.357520137361803,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-3.9, -5.530105422124848, -7.037881288289441...","[-3.9, -6.72726512708329, -7.213515256349699, ...","[0.0, 0.6961542107986322, 0.2706151903980963, ..."
8,KNeighborsRegressor,"[-6.63, -7.0, -5.88, -4.8533333333333335, -4.7...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.63, -6.746666666666667, -7.0, -6.12666666...","[-6.7780000000000005, -6.696000000000001, -7.0...","[0.18126224096595522, 0.10133333333333354, 0.0..."
9,SVR,"[-6.259507575245289, -6.921333664641216, -5.80...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.102795876702296, -6.678565301973631, -7.0...","[-6.142469460916427, -6.749782262829788, -7.05...","[0.022738568939848627, 0.12831351278460568, 0...."


In [66]:
result_df.to_csv('Results/Descriptors/Results_2D_All_desc_LVR.csv')
prediction_df.to_csv('Results/Descriptors/Prediction_data_2D_All_desc_LVR.csv')

In [67]:
def features(df, target_column='Permeability', threshold=0.9):
    correlation_matrix = df.corr()
    
    features_to_drop = set()
    
    for feature in correlation_matrix.columns:
        if feature == target_column:
            continue 
        target_corr = correlation_matrix[target_column][feature]
        
        for other_feature in correlation_matrix.columns:
            if other_feature == feature or other_feature == target_column:
                continue
            
            if abs(correlation_matrix[feature][other_feature]) > threshold:
                other_target_corr = correlation_matrix[target_column][other_feature]

                if abs(other_target_corr) < abs(target_corr):
                    features_to_drop.add(other_feature)
                else:
                    features_to_drop.add(feature)
    selected_features = [col for col in df.columns if col not in features_to_drop and col != target_column]
    
    return selected_features

In [68]:
def remove_low_variance_columns(df, threshold=0.005):
    variances = df.var()
    low_variance_columns = variances[variances < threshold].index.tolist()
    df_cleaned = df.drop(columns=low_variance_columns)
    return df_cleaned, low_variance_columns

In [69]:
df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors.csv')
df_train =df_train.dropna()
train = df_train.drop(['ID','SMILES'],axis=1)
train = train.select_dtypes(include=['number'])
train, _ = remove_low_variance_columns(train)
selected_features = features(train, "Permeability")
X_train = df_train[selected_features] 
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_all_descriptors.csv')
df_test =df_test.dropna()
X_test =  df_test[X_train.columns]
y_test =  df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df



/tmp/ipykernel_3612489/2653650005.py:1: DtypeWarning: Columns (1275,1277,1280,1285,1292,1298,1304,1354,1356,1359,1364,1371,1377,1383,1579,1580,1581,1583,1584,1590,1595,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors.csv')


X_train shape:  (5568, 236)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX


/tmp/ipykernel_3612489/2653650005.py:12: DtypeWarning: Columns (1275,1277,1292,1298,1354,1356,1371,1377,1579,1580,1581,1583,1584,1595,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test = pd.read_csv('features/Descriptors/Test_2d_all_descriptors.csv')


X_test shape:  (1392, 236)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009626 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 48659
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 227
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007273 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 48679
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 229
[LightGBM] [Info] Start training from score -5.744959
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006615 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2104,0.3378,0.4587,0.6621,0.8142,0.7836,0.2077,0.3263,0.4557,0.6734,0.8218,0.8020
DecisionTreeRegressor,0.3131,0.3910,0.5596,0.4971,0.7270,0.7015,0.2419,0.3500,0.4918,0.6196,0.7881,0.7654
RandomForestRegressor,0.2197,0.3461,0.4687,0.6471,0.8045,0.7724,0.2226,0.3392,0.4718,0.6499,0.8065,0.7842
GradientBoostingRegressor,0.2313,0.3570,0.4809,0.6286,0.7955,0.7599,0.2282,0.3462,0.4777,0.6410,0.8047,0.7787
AdaBoostRegressor,0.3985,0.5087,0.6313,0.3599,0.6337,0.5863,0.3683,0.4880,0.6069,0.4207,0.6854,0.6577
XGBRegressor,0.2191,0.3431,0.4681,0.6481,0.8057,0.7720,0.2176,0.3324,0.4665,0.6578,0.8112,0.7923
ExtraTreesRegressor,0.2095,0.3387,0.4578,0.6634,0.8145,0.7837,0.2169,0.3343,0.4657,0.6588,0.8118,0.7915
LinearRegression,0.2964,0.3963,0.5445,0.5239,0.7295,0.7142,0.3248,0.4000,0.5699,0.4891,0.7081,0.7306
KNeighborsRegressor,0.2771,0.3831,0.5264,0.5549,0.7511,0.7192,0.2827,0.3792,0.5317,0.5553,0.7492,0.7448
SVR,0.2196,0.3367,0.4686,0.6473,0.8077,0.7816,0.2255,0.3374,0.4749,0.6453,0.8077,0.7937


In [70]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.43389801602978, -6.645840340609075, -5.998...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.063523565849772, -6.496724101287346, -7.0...","[-6.996391869217476, -6.49396000024232, -6.837...","[0.09600160564912816, 0.2385217128862351, 0.16..."
1,DecisionTreeRegressor,"[-7.0, -7.0, -4.92, -4.37, -5.07, -5.05, -5.85...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.49, -7.0, -5.92, -5.89, -4.85, -7.0, -6.8...","[-6.012, -6.934, -6.744, -6.531999999999999, -...","[0.6201096677201542, 0.08138795979750321, 0.41..."
2,RandomForestRegressor,"[-6.68048509493, -6.633934245075002, -5.908555...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.872800000000001, -6.397245047067501, -6.6...","[-6.734223333333334, -6.5023472409465715, -6.6...","[0.13854804365273454, 0.08490239995112182, 0.0..."
3,GradientBoostingRegressor,"[-6.897212141286649, -6.901244226824872, -6.24...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.905659037277905, -5.9998094238473945, -6....","[-6.888557357390316, -6.072970537703846, -6.75...","[0.1443984793732857, 0.1152560176130878, 0.096..."
4,AdaBoostRegressor,"[-5.879759200430288, -6.156630475313308, -5.34...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.156630475313308, -5.378317482467481, -5.7...","[-5.932191923839826, -5.607144878097397, -5.74...","[0.11449008013648437, 0.12367248878060925, 0.0..."
5,XGBRegressor,"[-6.877865, -6.606186, -6.1412106, -5.305226, ...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.502764, -6.7294416, -7.370799, -6.402763,...","[-7.1435637, -6.8550797, -7.0570626, -6.581530...","[0.20008358, 0.16473135, 0.3034836, 0.1441388,..."
6,ExtraTreesRegressor,"[-6.704000000000002, -6.839400000000001, -6.18...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.9573, -6.442250000000001, -6.923900000000...","[-6.965260000000001, -6.502170000000001, -6.90...","[0.0137964633149224, 0.10933912200123007, 0.01..."
7,LinearRegression,"[-9.337617300557223, -6.357415389592783, -5.78...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-10.0, -6.79104936510267, -6.980659296684613...","[-10.0, -6.427130404060739, -6.769842210709814...","[0.0, 0.18330229327024974, 0.11332819460762222..."
8,KNeighborsRegressor,"[-6.63, -6.986666666666667, -5.88, -5.16, -4.7...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.63, -6.003333333333333, -7.0, -5.86666666...","[-6.7780000000000005, -5.8693333333333335, -6....","[0.18126224096595522, 0.5429565155496138, 0.38..."
9,SVR,"[-6.014664250773488, -6.750791448747776, -5.68...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.014134369610947, -6.509366391495711, -6.8...","[-6.038053870168538, -6.573030179574597, -6.87...","[0.016803507557831138, 0.1846304667504601, 0.0..."


In [71]:
result_df.to_csv('Results/Descriptors/Results_2D_All_desc_LVR_remove_corr_features.csv')
prediction_df.to_csv('Results/Descriptors/Prediction_data_2D_All_desc_LVRremove_corr_features.csv')

In [72]:
#3d RDKit descriptors
df_train = pd.read_csv('features/Descriptors/Train_3d_RDKit_desc.csv')
df_train = df_train.fillna(0)
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_3d_RDKit_desc.csv')
df_test = df_test.fillna(0)
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (5568, 11)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 11)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015896 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2805
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 11
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001468 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2805
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 11
[LightGBM] [Info] Start training from score -5.744959


,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.5281,0.5465,0.7267,0.1517,0.3910,0.3672,0.5146,0.5420,0.7174,0.1905,0.4396,0.3997
DecisionTreeRegressor,1.0368,0.7546,1.0182,-0.6653,0.1814,0.1934,0.6440,0.6004,0.8025,-0.0129,0.3122,0.2977
RandomForestRegressor,0.5372,0.5520,0.7329,0.1372,0.3867,0.3493,0.5251,0.5451,0.7246,0.1741,0.4191,0.3823
GradientBoostingRegressor,0.5315,0.5499,0.7291,0.1463,0.3824,0.3429,0.5355,0.5533,0.7318,0.1578,0.3982,0.3624
AdaBoostRegressor,0.6713,0.6705,0.8193,-0.0781,0.2821,0.2395,0.6582,0.6681,0.8113,-0.0353,0.3336,0.2801
XGBRegressor,0.5887,0.5803,0.7673,0.0544,0.3364,0.3062,0.5295,0.5446,0.7277,0.1671,0.4164,0.3879
ExtraTreesRegressor,0.5592,0.5634,0.7478,0.1018,0.3602,0.3227,0.5382,0.5518,0.7336,0.1535,0.4006,0.3599
LinearRegression,0.5920,0.5851,0.7694,0.0492,0.2221,0.2716,0.6024,0.5909,0.7762,0.0524,0.2296,0.2748
KNeighborsRegressor,0.6819,0.6218,0.8258,-0.0953,0.2461,0.2299,0.6266,0.5942,0.7916,0.0145,0.2907,0.2584
SVR,0.5471,0.5421,0.7396,0.1213,0.3788,0.3632,0.5474,0.5389,0.7399,0.1390,0.4060,0.3898


In [73]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.16037417061462, -6.017202329904702, -4.798...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.431941868915263, -6.1801052764328475, -6....","[-6.379295330903075, -6.199440755216278, -6.74...","[0.03234296066056556, 0.10636997735576485, 0.0..."
1,DecisionTreeRegressor,"[-6.89, -6.89, -4.8, -4.38, -5.85, -4.62, -6.2...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -5.2, -6.725555555555555, -5.2, -5.89,...","[-6.956, -6.05, -6.717702380952382, -5.976, -5...","[0.05388877434123007, 0.800149985940136, 0.070..."
2,RandomForestRegressor,"[-6.4338, -6.5843, -4.672200000000003, -4.9358...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.844099999999998, -6.242199999999997, -6.7...","[-6.845559999999997, -6.264879999999999, -6.73...","[0.031541502817717476, 0.06476342795127564, 0...."
3,GradientBoostingRegressor,"[-6.535414502618415, -6.014249101441637, -4.90...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.774726165160332, -5.821003043122007, -6.7...","[-6.841574331065407, -6.060764959896899, -6.78...","[0.12259079244926353, 0.20216883404201386, 0.0..."
4,AdaBoostRegressor,"[-6.124690265486725, -6.012491188334828, -5.49...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.1191739712996585, -6.012491188334828, -6....","[-6.141382245008641, -5.961518362849903, -6.51...","[0.10221437744485708, 0.08423022377772915, 0.1..."
5,XGBRegressor,"[-6.0496516, -6.2466526, -5.176728, -5.1222095...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.092831, -6.593839, -6.723245, -5.978617, ...","[-6.964574, -6.631154, -6.7248125, -6.4045897,...","[0.08983253, 0.3484255, 0.07123882, 0.49929735..."
6,ExtraTreesRegressor,"[-6.344100000000002, -6.413699999999997, -4.71...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.820599999999996, -6.487600000000001, -6.7...","[-6.820159999999996, -6.495699999999999, -6.71...","[0.03315398015321746, 0.10033687258430947, 0.0..."
7,LinearRegression,"[-4.174588592553809, -5.258284139168037, -4.99...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-4.76476320888328, -4.1970783372572, -6.6922...","[-4.8684108978649565, -4.208129426507948, -6.6...","[0.08380600685927832, 0.08147441069554712, 0.0..."
8,KNeighborsRegressor,"[-6.44, -6.603333333333333, -4.746666666666666...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.603333333333334, -6.146666666666666, -6.9...","[-6.754666666666668, -6.255333333333333, -6.78...","[0.17090088095475411, 0.11887621947031955, 0.1..."
9,SVR,"[-6.186026126616809, -6.486234448915026, -4.86...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.064051695942463, -6.2357427840166615, -6....","[-7.0522595517577145, -6.574865930891666, -6.8...","[0.17468706052071326, 0.22760017551812506, 0.0..."


In [74]:
result_df.to_csv('Results/Descriptors/Results_3D_RDKit_desc.csv')
prediction_df.to_csv('Results/Descriptors/Prediction_data_3D_RDKit_desc.csv')

In [75]:
#3d Padel descriptors
df_train = pd.read_csv('features/Descriptors/Train_3d_padel_curated.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_3d_padel_curated.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (5568, 431)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 431)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.072284 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 109905
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 431
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.067659 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 109905
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 431
[LightGBM] [Info] Start training from score -5.744959
[LightGBM] [Info] Auto-choosing col-wise multi-threading

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3006,0.4035,0.5483,0.5172,0.7209,0.6974,0.2860,0.3913,0.5348,0.5501,0.7469,0.7269
DecisionTreeRegressor,0.6253,0.5636,0.7907,-0.0043,0.5119,0.5086,0.3545,0.4337,0.5954,0.4424,0.6716,0.6522
RandomForestRegressor,0.3062,0.4078,0.5534,0.5081,0.7165,0.6958,0.2948,0.3985,0.5430,0.5363,0.7382,0.7281
GradientBoostingRegressor,0.3265,0.4193,0.5714,0.4755,0.6929,0.6709,0.3193,0.4132,0.5651,0.4978,0.7117,0.6981
AdaBoostRegressor,0.4845,0.5638,0.6961,0.2218,0.5460,0.5215,0.4644,0.5533,0.6814,0.2696,0.5916,0.5695
XGBRegressor,0.3339,0.4245,0.5779,0.4637,0.6841,0.6573,0.2832,0.3896,0.5321,0.5546,0.7458,0.7200
ExtraTreesRegressor,0.2968,0.4027,0.5448,0.5233,0.7288,0.7070,0.2882,0.3950,0.5369,0.5466,0.7468,0.7308
LinearRegression,0.3948,0.4646,0.6284,0.3658,0.6159,0.6201,0.3942,0.4591,0.6278,0.3800,0.6215,0.6357
KNeighborsRegressor,0.4510,0.4896,0.6716,0.2756,0.5536,0.5313,0.4402,0.4772,0.6635,0.3076,0.5658,0.5627
SVR,0.3822,0.4302,0.6182,0.3862,0.6294,0.6401,0.3944,0.4289,0.6280,0.3796,0.6276,0.6482


In [76]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.4896889542588205, -6.686931476361777, -5.6...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.19385525551183, -6.175850047879983, -6.45...","[-6.997626658053943, -6.501255877978977, -6.60...","[0.1414507241823429, 0.2126730162660031, 0.100..."
1,DecisionTreeRegressor,"[-7.0, -4.52, -5.77, -5.49, -7.0, -4.77, -4.74...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -7.0, -6.473999999999999, -5.36, -6.85...","[-6.7780000000000005, -6.696000000000001, -6.6...","[0.4440000000000001, 0.37232244090304295, 0.09..."
2,RandomForestRegressor,"[-6.369900000000004, -6.116700000000001, -5.78...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.7044000000000015, -6.312200000000002, -6....","[-6.755860000000001, -6.387120000000001, -6.65...","[0.052235719579613364, 0.08821767169904152, 0...."
3,GradientBoostingRegressor,"[-6.5315544531496785, -6.363054568149864, -5.4...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.779612058370822, -6.562858789939575, -6.3...","[-6.833426694599207, -6.464498209380642, -6.60...","[0.08034215738978995, 0.13845525501031264, 0.1..."
4,AdaBoostRegressor,"[-5.585609827919191, -5.675411640887905, -5.69...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.675411640887905, -5.585609827919191, -5.9...","[-5.668536070975354, -5.565296554046377, -5.96...","[0.06205372502762775, 0.048855387394222426, 0...."
5,XGBRegressor,"[-5.866906, -5.924335, -6.0000567, -5.353, -5....",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.7839274, -6.1093287, -6.483452, -6.163554...","[-6.932872, -6.533258, -6.6213865, -6.203944, ...","[0.3078478, 0.24963082, 0.08498923, 0.14996995..."
6,ExtraTreesRegressor,"[-6.540299999999999, -6.163700000000001, -5.71...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.872700000000001, -6.6315, -6.474000000000...","[-6.87072, -6.5911, -6.622490476190483, -6.302...","[0.05303423045543345, 0.08013970301916488, 0.0..."
7,LinearRegression,"[-6.904507361574728, -5.530670537752909, -5.52...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.575464817009358, -6.786363805095542, -6.4...","[-6.785368086808984, -6.806725954561488, -6.62...","[0.2739254340325793, 0.22137635488942836, 0.09..."
8,KNeighborsRegressor,"[-6.63, -6.053333333333334, -5.37, -5.35666666...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -6.63, -6.493333333333333, -6.12666666...","[-6.935333333333334, -6.6033333333333335, -6.6...","[0.11615315559878517, 0.05333333333333314, 0.1..."
9,SVR,"[-6.926963905307199, -6.169507286726842, -5.64...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.140369868762438, -6.81811538627642, -6.69...","[-7.149275245410413, -6.766917078199967, -6.86...","[0.043224320861689175, 0.16555912085187943, 0...."


In [77]:
result_df.to_csv('Results/Descriptors/Results_3D_padel_desc.csv')
prediction_df.to_csv('Results/Descriptors/Prediction_data_3D_padel_desc.csv')

In [78]:
df_train_rdkit = pd.read_csv('features/Descriptors/Train_3d_RDKit_desc.csv')
df_train_rdkit = df_train_rdkit.fillna(0)
df_train_padel = pd.read_csv('features/Descriptors/Train_3d_padel_curated.csv')

df_3d_descriptors = df_train_rdkit.merge(df_train_padel, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_3d_descriptors

,ID,SMILES,Permeability,3d_rdkit_1,3d_rdkit_2,3d_rdkit_3,3d_rdkit_4,3d_rdkit_5,3d_rdkit_6,3d_rdkit_7,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,915,CC[C@H](C)[C@H](NC(=O)[C@@H]1CC(=O)N[C@@H](Cc2...,-7.0,55956.786985,99960.858242,143887.632605,0.388892,0.694715,9.194126,0.000012,...,0.645217,0.280139,0.482152,0.434807,0.328596,85.400757,1822.029585,10310.940180,0.467826,1.245555
1,888,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1cccc(Cl)c1)N(C)...,-7.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.823799,0.133805,0.505465,0.357149,0.373757,96.526908,1405.322971,5704.929467,0.735698,1.236370
2,593,C/N=C(\NC)NCCC[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C...,-7.0,58796.959784,81694.256927,121554.991217,0.483707,0.672077,8.694432,0.000011,...,0.638375,0.270605,0.448344,0.374973,0.359780,77.329248,1527.741116,8875.871077,0.457562,1.183097
3,916,CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@@H]1CC(=...,-7.0,67853.021592,84551.096230,143909.838389,0.471497,0.587528,9.266829,0.000009,...,0.565329,0.374752,0.554262,0.543760,0.297042,87.856411,2070.065223,10766.402505,0.410122,1.395064
4,900,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)N(C)C(=O...,-7.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.834311,0.113966,0.516432,0.367614,0.451429,108.294919,1690.332479,8044.745507,0.751467,1.335474
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5563,2469,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-4.7,1743.977405,5966.057232,7181.572782,0.242841,0.830745,4.300825,0.000476,...,0.676563,0.210855,0.446073,0.409627,0.423449,19.367569,90.986445,227.031496,0.514844,1.279149
5564,2467,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-5.6,1098.802374,5640.564299,6362.780182,0.172692,0.886494,4.182530,0.000807,...,0.787624,0.163507,0.454126,0.429936,0.446975,20.666779,74.857439,151.076798,0.681436,1.331037
5565,2512,CC(C)C[C@H]1NC(=O)[C@@H](C)NCCCCCCNC(=O)[C@H](...,-5.5,2206.160044,3101.366929,4907.135825,0.449582,0.632012,3.712842,0.000286,...,0.585877,0.334994,0.455378,0.451219,0.447788,17.819583,85.459813,191.155916,0.381306,1.354384
5566,2511,CC(C)[C@@H]1NC(=O)[C@@H](CO)NC(=O)[C@@H](C)NCC...,-6.3,2030.228104,2614.705158,4174.247246,0.486370,0.626390,3.517138,0.000309,...,0.458760,0.383626,0.465648,0.486191,0.476767,14.563300,65.485651,165.726434,0.263580,1.428605


In [79]:

nan_rows = df_3d_descriptors[df_3d_descriptors.isna().any(axis=1)]
nan_rows

,ID,SMILES,Permeability,3d_rdkit_1,3d_rdkit_2,3d_rdkit_3,3d_rdkit_4,3d_rdkit_5,3d_rdkit_6,3d_rdkit_7,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds


In [80]:
df_3d_descriptors.to_csv('features/Descriptors/Train_3d_all_descriptors.csv', index=False)

In [81]:
df_test_rdkit = pd.read_csv('features/Descriptors/Test_3d_RDKit_desc.csv')
df_test_rdkit = df_test_rdkit.fillna(0)
df_test_padel = pd.read_csv('features/Descriptors/Test_3d_padel_curated.csv')

df_3d_descriptors = df_test_rdkit.merge(df_test_padel, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_3d_descriptors

,ID,SMILES,Permeability,3d_rdkit_1,3d_rdkit_2,3d_rdkit_3,3d_rdkit_4,3d_rdkit_5,3d_rdkit_6,3d_rdkit_7,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,908,CC[C@H](C)[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[...,-7.00,33760.208016,138153.247483,155908.190288,0.216539,0.886119,9.602166,0.000026,...,0.767875,0.179312,0.502200,0.441173,0.380319,98.965167,1838.483001,8985.839378,0.651812,1.323692
1,923,CC[C@H](C)[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](...,-7.00,56404.755742,90641.865119,134352.716685,0.419826,0.674656,9.030600,0.000012,...,0.585474,0.346546,0.520768,0.527965,0.365485,81.328539,1761.081455,9261.968931,0.398030,1.414218
2,897,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,587,CC(C)C[C@@H]1NC(=O)[C@H](Cc2c[nH]cn2)NC(=O)[C@...,-6.74,48070.001754,80006.497656,115096.002236,0.417651,0.695128,8.491650,0.000014,...,0.591414,0.326538,0.433409,0.441889,0.423103,80.003796,1718.149538,9911.992077,0.387121,1.298401
4,921,CC[C@H](C)[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[...,-5.54,65151.355592,73494.985367,128961.900675,0.505198,0.569897,8.953319,0.000009,...,0.611991,0.343470,0.568474,0.538717,0.354893,86.672490,1898.733289,8081.078328,0.433191,1.462084
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1387,2481,CC(C)C[C@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2ccccc...,-4.50,2415.488872,5133.998799,6699.482959,0.360549,0.766328,4.067647,0.000317,...,0.719948,0.221983,0.461292,0.478911,0.410452,21.264177,96.995284,207.488727,0.579923,1.350655
1388,2485,CC(C)C[C@H]1NC(=O)[C@H](C)NC(=O)[C@@H](Cc2cccc...,-4.80,2626.126299,7156.499036,7502.735704,0.350022,0.953852,4.480131,0.000363,...,0.712577,0.226527,0.474584,0.463075,0.429658,22.925205,114.891434,256.251692,0.568866,1.367317
1389,5604,CC(C)CN1CC(=O)N[C@@H](Cc2ccccc2)C(=O)NCCCCC(=O...,-6.38,2992.608278,4917.452764,5722.031906,0.522997,0.859389,3.978826,0.000287,...,0.544506,0.389352,0.501813,0.459884,0.479810,20.788934,118.318494,265.091453,0.400788,1.441507
1390,2513,C[C@H]1NCCCCCCNC(=O)[C@H](Cc2ccc(O)cc2)NC(=O)[...,-7.80,2522.551028,5008.984905,7154.625746,0.352576,0.700104,4.129789,0.000278,...,0.554492,0.350152,0.404760,0.473007,0.444278,19.365724,105.166185,258.994679,0.356966,1.322045


In [82]:
nan_rows = df_3d_descriptors[df_3d_descriptors.isna().any(axis=1)]
nan_rows

,ID,SMILES,Permeability,3d_rdkit_1,3d_rdkit_2,3d_rdkit_3,3d_rdkit_4,3d_rdkit_5,3d_rdkit_6,3d_rdkit_7,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds


In [83]:
df_3d_descriptors.to_csv('features/Descriptors/Test_3d_all_descriptors.csv', index=False)

In [84]:
#3d All descriptors
df_train = pd.read_csv('features/Descriptors/Train_3d_all_descriptors.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_3d_all_descriptors.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models_3dall = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models_3dall, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (5568, 442)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 442)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.050470 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 112710
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 442
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.048418 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 112710
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 442
[LightGBM] [Info] Start training from score -5.744959
[LightGBM] [Info] Auto-choosing col-wise multi-threading

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3005,0.4050,0.5482,0.5173,0.7208,0.6961,0.2861,0.3924,0.5349,0.5500,0.7468,0.7296
DecisionTreeRegressor,0.6533,0.5714,0.8083,-0.0494,0.4907,0.4974,0.3569,0.4341,0.5974,0.4387,0.6705,0.6505
RandomForestRegressor,0.3069,0.4080,0.5540,0.5070,0.7159,0.6953,0.2951,0.3985,0.5432,0.5358,0.7381,0.7278
GradientBoostingRegressor,0.3280,0.4201,0.5727,0.4732,0.6910,0.6709,0.3188,0.4135,0.5646,0.4986,0.7122,0.6983
AdaBoostRegressor,0.4864,0.5651,0.6974,0.2188,0.5392,0.5233,0.4653,0.5539,0.6822,0.2681,0.5878,0.5684
XGBRegressor,0.3270,0.4213,0.5718,0.4748,0.6910,0.6642,0.2820,0.3895,0.5310,0.5565,0.7475,0.7240
ExtraTreesRegressor,0.2977,0.4038,0.5456,0.5219,0.7276,0.7054,0.2876,0.3945,0.5363,0.5476,0.7474,0.7330
LinearRegression,0.3950,0.4651,0.6285,0.3655,0.6156,0.6191,0.3963,0.4598,0.6295,0.3767,0.6195,0.6349
KNeighborsRegressor,0.4539,0.4920,0.6737,0.2710,0.5498,0.5236,0.4413,0.4808,0.6643,0.3059,0.5644,0.5568
SVR,0.3817,0.4306,0.6178,0.3869,0.6300,0.6401,0.3945,0.4307,0.6281,0.3795,0.6277,0.6464


In [85]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.565762637649602, -6.527817519819895, -5.72...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.103340804310352, -6.167817775718607, -6.3...","[-6.9251632190260555, -6.51195614876837, -6.50...","[0.09894455977684198, 0.18243091405709977, 0.1..."
1,DecisionTreeRegressor,"[-7.0, -4.85, -4.68, -5.2, -7.0, -4.68, -4.6, ...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -7.0, -6.19, -5.28, -6.96, -6.96, -6.8...","[-6.984, -6.206, -6.160000000000001, -6.202, -...","[0.03200000000000003, 0.8950441329901002, 0.14..."
2,RandomForestRegressor,"[-6.4053, -6.0840000000000005, -5.7785, -5.329...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.711100000000001, -6.3378000000000005, -6....","[-6.73588, -6.4361999999999995, -6.31034033333...","[0.06603500283940324, 0.10541227632491461, 0.0..."
3,GradientBoostingRegressor,"[-6.595400983191616, -6.345790781614015, -5.48...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.733049059683465, -6.61150878924153, -6.31...","[-6.821859710374719, -6.4590853460695525, -6.4...","[0.11025460184816774, 0.0982782003202941, 0.10..."
4,AdaBoostRegressor,"[-5.648979471064177, -5.5743157126678184, -5.6...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.648979471064177, -5.5743157126678184, -5....","[-5.677954113428067, -5.625811431750975, -5.91...","[0.029637102162635928, 0.026351799212944494, 0..."
5,XGBRegressor,"[-6.343426, -6.3267446, -5.6914873, -5.518805,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.688993, -6.2686763, -6.197953, -6.344839,...","[-6.9019265, -6.5324974, -6.1695504, -6.167803...","[0.24570036, 0.19626938, 0.14036265, 0.3200777..."
6,ExtraTreesRegressor,"[-6.573600000000002, -6.238900000000002, -5.80...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.874200000000001, -6.670800000000002, -6.1...","[-6.85272, -6.63424, -6.160000000000002, -6.25...","[0.060926099497670146, 0.09950516770499906, 0...."
7,LinearRegression,"[-6.896546834649548, -5.521741287321499, -5.39...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.7055047298359, -6.773576617369379, -6.449...","[-6.8921567662710075, -6.800036091405687, -6.5...","[0.2748486671397165, 0.23235366867251422, 0.12..."
8,KNeighborsRegressor,"[-6.63, -6.45, -6.093333333333334, -5.35666666...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -6.63, -6.19, -6.126666666666668, -6.3...","[-6.940666666666667, -6.6033333333333335, -6.3...","[0.11866666666666674, 0.05333333333333314, 0.1..."
9,SVR,"[-6.951769937986526, -6.338963708102585, -5.66...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.163373464534208, -6.82873605659044, -5.99...","[-7.154422812340369, -6.776573351937932, -6.04...","[0.07359164973885451, 0.16510108192847875, 0.1..."


In [86]:
result_df.to_csv('Results/Descriptors/Results_3D_All_desc.csv')
prediction_df.to_csv('Results/Descriptors/Prediction_data_3D_All_desc.csv')

In [87]:
#3d All descriptors const rem
df_train = pd.read_csv('features/Descriptors/Train_3d_all_descriptors.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train,  const_col =  remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_3d_all_descriptors.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")


X_train shape:  (5568, 442)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 442)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX


In [88]:
#3d All descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_3d_all_descriptors.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train,  const_col =  remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_3d_all_descriptors.csv')
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

X_train shape:  (5568, 391)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
X_test shape:  (1392, 391)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.046371 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 99705
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 391
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.041273 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 99705
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 391
[LightGBM] [Info] Start training from score -5.744959
[LightGBM] [Info] Auto-choosing col-wise multi-threading, 

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.3059,0.4069,0.5531,0.5087,0.7151,0.6911,0.2928,0.3961,0.5411,0.5395,0.7395,0.7243
DecisionTreeRegressor,0.6645,0.5783,0.8152,-0.0673,0.4829,0.4878,0.3814,0.4454,0.6176,0.4001,0.6426,0.6239
RandomForestRegressor,0.3141,0.4122,0.5605,0.4955,0.7070,0.6876,0.3047,0.4037,0.5520,0.5207,0.7274,0.7218
GradientBoostingRegressor,0.3351,0.4251,0.5789,0.4618,0.6825,0.6625,0.3315,0.4199,0.5757,0.4786,0.6973,0.6868
AdaBoostRegressor,0.4930,0.5681,0.7022,0.2081,0.5275,0.5077,0.4669,0.5561,0.6833,0.2657,0.5843,0.5601
XGBRegressor,0.3369,0.4287,0.5804,0.4589,0.6810,0.6537,0.2947,0.3962,0.5428,0.5365,0.7332,0.7125
ExtraTreesRegressor,0.3028,0.4063,0.5502,0.5137,0.7220,0.7036,0.2938,0.3987,0.5420,0.5379,0.7412,0.7244
LinearRegression,0.4019,0.4666,0.6340,0.3545,0.6051,0.6125,0.3945,0.4567,0.6281,0.3795,0.6199,0.6363
KNeighborsRegressor,0.4552,0.4871,0.6747,0.2689,0.5530,0.5364,0.4153,0.4641,0.6445,0.3467,0.5963,0.5897
SVR,0.3837,0.4309,0.6194,0.3838,0.6280,0.6400,0.3972,0.4314,0.6302,0.3753,0.6243,0.6474


In [89]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.489690948782148, -6.515436433575198, -5.38...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.022382157662364, -6.166316830857317, -6.1...","[-6.919102780799915, -6.384539547318753, -6.21...","[0.06515761726733313, 0.2611049457780652, 0.14..."
1,DecisionTreeRegressor,"[-5.36, -6.24, -5.28, -5.2, -7.0, -4.92, -4.66...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -7.0, -6.19, -7.0, -6.85, -6.96, -5.66...","[-6.901999999999999, -6.51, -6.160000000000001...","[0.1960000000000001, 0.6683113047076191, 0.146..."
2,RandomForestRegressor,"[-6.391200000000001, -6.202499999999999, -5.75...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.687900000000002, -6.3187, -6.266252857142...","[-6.690620000000001, -6.360740000000001, -6.25...","[0.08673687566427606, 0.11374795998170702, 0.1..."
3,GradientBoostingRegressor,"[-6.414512673295754, -6.389164738165055, -5.50...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.873340505498529, -6.4579567230333055, -6....","[-6.967002521617114, -6.475432787558274, -6.22...","[0.06426075313389149, 0.06946652904834755, 0.0..."
4,AdaBoostRegressor,"[-5.691244864932149, -5.737108450084143, -5.55...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.697015517927294, -5.675631560466694, -5.9...","[-5.618869925212399, -5.565494623116413, -5.83...","[0.04049855329765304, 0.07354499110233337, 0.1..."
5,XGBRegressor,"[-7.26253, -6.066133, -5.494929, -5.455032, -5...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.92709, -6.682286, -6.191465, -6.204871, -...","[-6.8341675, -6.4556227, -6.166051, -6.2352805...","[0.11822619, 0.18383528, 0.1472393, 0.16323926..."
6,ExtraTreesRegressor,"[-6.5965, -6.273599999999999, -5.9265000000000...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.909500000000001, -6.592500000000002, -6.1...","[-6.8827799999999995, -6.623920000000001, -6.1...","[0.045455006324935786, 0.11143194156075648, 0...."
7,LinearRegression,"[-7.025208477946809, -5.518459857077767, -5.43...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.723161692910603, -6.688004959104182, -6.4...","[-6.886696873237142, -6.690804837023478, -6.57...","[0.24384188953278435, 0.1474477429221247, 0.16..."
8,KNeighborsRegressor,"[-6.63, -6.003333333333334, -6.539999999999999...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -6.63, -6.19, -6.63, -6.37666666666666...","[-6.822, -6.6033333333333335, -6.352, -6.58333...","[0.14533639140513532, 0.05333333333333314, 0.1..."
9,SVR,"[-6.956000514067994, -6.419417843305543, -5.75...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.158073224457094, -6.864262122206911, -5.9...","[-7.169346983302901, -6.805535043459789, -6.08...","[0.05972209153251348, 0.16318374599963464, 0.1..."


In [90]:
result_df.to_csv('Results/Descriptors/Results_3D_All_desc_LVR.csv')
prediction_df.to_csv('Results/Descriptors/Prediction_data_3D_All_desc_LVR.csv')

In [91]:
#2d and 3d descriptors all
df_train_2d = pd.read_csv('features/Descriptors/Train_2d_all_descriptors.csv')
df_train_2d
df_train_3d = pd.read_csv('features/Descriptors/Train_3d_all_descriptors.csv')
df_train_3d

df_2d_3d_train = df_train_2d.merge(df_train_3d, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_2d_3d_train.to_csv('features/Descriptors/Train_2d_3d_all_descriptors.csv', index=False)
df_2d_3d_train

/tmp/ipykernel_3612489/2153735606.py:2: DtypeWarning: Columns (1275,1277,1280,1285,1292,1298,1304,1354,1356,1359,1364,1371,1377,1383,1579,1580,1581,1583,1584,1590,1595,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train_2d = pd.read_csv('features/Descriptors/Train_2d_all_descriptors.csv')


,ID,SMILES,Permeability,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,915,CC[C@H](C)[C@H](NC(=O)[C@@H]1CC(=O)N[C@@H](Cc2...,-7.0,15.738544,15.738544,0.010382,-1.908222,0.047997,24.622047,1773.325,...,0.645217,0.280139,0.482152,0.434807,0.328596,85.400757,1822.029585,10310.940180,0.467826,1.245555
1,888,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1cccc(Cl)c1)N(C)...,-7.0,15.975705,15.975705,0.027671,-1.943827,0.026511,21.983740,1745.057,...,0.823799,0.133805,0.505465,0.357149,0.373757,96.526908,1405.322971,5704.929467,0.735698,1.236370
2,593,C/N=C(\NC)NCCC[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C...,-7.0,15.828473,15.828473,0.049834,-1.862294,0.021075,22.464000,1733.267,...,0.638375,0.270605,0.448344,0.374973,0.359780,77.329248,1527.741116,8875.871077,0.457562,1.183097
3,916,CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@@H]1CC(=...,-7.0,15.595105,15.595105,0.004954,-1.811756,0.069603,24.479675,1725.281,...,0.565329,0.374752,0.554262,0.543760,0.297042,87.856411,2070.065223,10766.402505,0.410122,1.395064
4,900,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)N(C)C(=O...,-7.0,15.867592,15.867592,0.029192,-1.876686,0.046796,23.089431,1723.309,...,0.834311,0.113966,0.516432,0.367614,0.451429,108.294919,1690.332479,8044.745507,0.751467,1.335474
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5563,2469,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-4.7,12.835172,12.835172,0.168936,-0.728726,0.606745,24.965517,402.539,...,0.676563,0.210855,0.446073,0.409627,0.423449,19.367569,90.986445,227.031496,0.514844,1.279149
5564,2467,CC(C)C[C@@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2cccc...,-5.6,12.673271,12.673271,0.196704,-0.722720,0.611202,25.037037,374.485,...,0.787624,0.163507,0.454126,0.429936,0.446975,20.666779,74.857439,151.076798,0.681436,1.331037
5565,2512,CC(C)C[C@H]1NC(=O)[C@@H](C)NCCCCCCNC(=O)[C@H](...,-5.5,12.572226,12.572226,0.181773,-1.016186,0.465702,27.576923,370.494,...,0.585877,0.334994,0.455378,0.451219,0.447788,17.819583,85.459813,191.155916,0.381306,1.354384
5566,2511,CC(C)[C@@H]1NC(=O)[C@@H](CO)NC(=O)[C@@H](C)NCC...,-6.3,12.364448,12.364448,0.112529,-1.084532,0.446971,28.200000,356.467,...,0.458760,0.383626,0.465648,0.486191,0.476767,14.563300,65.485651,165.726434,0.263580,1.428605


In [92]:
df_test_2d = pd.read_csv('features/Descriptors/Test_2d_all_descriptors.csv')
df_test_2d
df_test_3d = pd.read_csv('features/Descriptors/Test_3d_all_descriptors.csv')
df_test_3d

df_2d_3d_test = df_test_2d.merge(df_test_3d, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_2d_3d_test.to_csv('features/Descriptors/Test_2d_3d_all_descriptors.csv', index=False)
df_2d_3d_test

/tmp/ipykernel_3612489/2718142607.py:1: DtypeWarning: Columns (1275,1277,1292,1298,1354,1356,1371,1377,1579,1580,1581,1583,1584,1595,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test_2d = pd.read_csv('features/Descriptors/Test_2d_all_descriptors.csv')


,ID,SMILES,Permeability,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,908,CC[C@H](C)[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[...,-7.00,15.806942,15.806942,0.022547,-1.946094,0.037676,23.507937,1777.744,...,0.767875,0.179312,0.502200,0.441173,0.380319,98.965167,1838.483001,8985.839378,0.651812,1.323692
1,923,CC[C@H](C)[C@@H]1NC(=O)[C@H](C)N(C)C(=O)[C@H](...,-7.00,15.512854,15.512854,0.072929,-1.849230,0.082004,25.902439,1725.281,...,0.585474,0.346546,0.520768,0.527965,0.365485,81.328539,1761.081455,9261.968931,0.398030,1.414218
2,897,CC[C@H](C)[C@@H]1NC(=O)[C@H](Cc2ccccc2)N(C)C(=...,-7.00,16.029525,16.029525,0.042154,-1.949385,0.046040,22.827869,1701.218,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,587,CC(C)C[C@@H]1NC(=O)[C@H](Cc2c[nH]cn2)NC(=O)[C@...,-6.74,15.776936,15.776936,0.046352,-1.865645,0.035370,22.622951,1686.166,...,0.591414,0.326538,0.433409,0.441889,0.423103,80.003796,1718.149538,9911.992077,0.387121,1.298401
4,921,CC[C@H](C)[C@@H]1NC(=O)[C@H](CC(C)C)N(C)C(=O)[...,-5.54,15.432735,15.432735,0.040020,-1.845687,0.100132,26.663866,1669.173,...,0.611991,0.343470,0.568474,0.538717,0.354893,86.672490,1898.733289,8081.078328,0.433191,1.462084
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1387,2481,CC(C)C[C@H]1NC(=O)[C@H](C)NC(=O)[C@H](Cc2ccccc...,-4.50,12.958074,12.958074,0.156212,-0.733388,0.586385,24.903226,430.593,...,0.719948,0.221983,0.461292,0.478911,0.410452,21.264177,96.995284,207.488727,0.579923,1.350655
1388,2485,CC(C)C[C@H]1NC(=O)[C@H](C)NC(=O)[C@@H](Cc2cccc...,-4.80,12.958074,12.958074,0.156212,-0.733388,0.586385,24.903226,430.593,...,0.712577,0.226527,0.474584,0.463075,0.429658,22.925205,114.891434,256.251692,0.568866,1.367317
1389,5604,CC(C)CN1CC(=O)N[C@@H](Cc2ccccc2)C(=O)NCCCCC(=O...,-6.38,12.898424,12.898424,0.143657,-0.742398,0.662387,22.258065,430.549,...,0.544506,0.389352,0.501813,0.459884,0.479810,20.788934,118.318494,265.091453,0.400788,1.441507
1390,2513,C[C@H]1NCCCCCCNC(=O)[C@H](Cc2ccc(O)cc2)NC(=O)[...,-7.80,13.095577,13.095577,0.065481,-0.737917,0.561202,26.806452,430.549,...,0.554492,0.350152,0.404760,0.473007,0.444278,19.365724,105.166185,258.994679,0.356966,1.322045


In [93]:
#All 2d and 3d descriptors
df_train = pd.read_csv('features/Descriptors/Train_2d_3d_all_descriptors.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_3d_all_descriptors.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

/tmp/ipykernel_3612489/3638277591.py:2: DtypeWarning: Columns (1275,1277,1280,1285,1292,1298,1304,1354,1356,1359,1364,1371,1377,1383,1579,1580,1581,1583,1584,1590,1595,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('features/Descriptors/Train_2d_3d_all_descriptors.csv')


X_train shape:  (5568, 3529)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX


/tmp/ipykernel_3612489/3638277591.py:10: DtypeWarning: Columns (1275,1277,1292,1298,1354,1356,1371,1377,1579,1580,1581,1583,1584,1595,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test = pd.read_csv('features/Descriptors/Test_2d_3d_all_descriptors.csv')


X_test shape:  (1392, 3529)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.313976 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 627430
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 2822
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.327525 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 627479
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 2827
[LightGBM] [Info] Start training from score -5.744959
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.293270 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] T

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2118,0.3398,0.4602,0.6598,0.8134,0.7783,0.2116,0.3336,0.4600,0.6672,0.8186,0.7973
DecisionTreeRegressor,0.4510,0.4806,0.6716,0.2756,0.6467,0.6221,0.2674,0.3740,0.5171,0.5795,0.7660,0.7381
RandomForestRegressor,0.2177,0.3462,0.4666,0.6504,0.8074,0.7739,0.2202,0.3399,0.4693,0.6536,0.8095,0.7888
GradientBoostingRegressor,0.2237,0.3518,0.4730,0.6406,0.8028,0.7634,0.2228,0.3445,0.4720,0.6496,0.8099,0.7802
AdaBoostRegressor,0.3759,0.4957,0.6131,0.3963,0.6651,0.6056,0.3599,0.4815,0.5999,0.4339,0.7003,0.6673
XGBRegressor,0.2405,0.3604,0.4904,0.6138,0.7845,0.7516,0.2172,0.3362,0.4661,0.6583,0.8118,0.7893
ExtraTreesRegressor,0.2125,0.3404,0.4610,0.6587,0.8117,0.7782,0.2185,0.3367,0.4674,0.6563,0.8102,0.7916
LinearRegression,1.3376,0.7012,1.1565,-1.1484,0.3955,0.5102,0.6698,0.5304,0.8184,-0.0536,0.5335,0.6106
KNeighborsRegressor,0.2727,0.3812,0.5222,0.5621,0.7579,0.7256,0.2750,0.3730,0.5244,0.5674,0.7587,0.7518
SVR,0.2294,0.3444,0.4790,0.6315,0.7979,0.7692,0.2384,0.3487,0.4882,0.6251,0.7949,0.7737


In [94]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.636653545675242, -6.681005805477045, -6.13...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.960273881772997, -6.090355191773246, -6.8...","[-6.924825786001401, -6.42079683232817, -6.813...","[0.09496275198100144, 0.23801194391904637, 0.0..."
1,DecisionTreeRegressor,"[-7.0, -6.6, -5.85, -4.25, -5.15, -4.77, -4.57...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.96, -6.24, -5.68, -7.0, -4.43, -7.0, -6.8...","[-6.6, -6.426, -6.456, -6.698, -5.23, -6.94, -...","[0.4952575087769997, 0.8224743157083996, 0.634..."
2,RandomForestRegressor,"[-6.700400000000002, -6.703599999999999, -5.80...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.800899999999998, -6.393500000000001, -6.5...","[-6.761799999999999, -6.460120000000001, -6.66...","[0.09398099807939846, 0.07510971708108144, 0.0..."
3,GradientBoostingRegressor,"[-6.934528344099418, -6.988494123637158, -5.53...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.089487172287799, -6.091214853366252, -6.7...","[-7.121258475272957, -6.369234024996914, -6.75...","[0.07365615321158679, 0.18358114711947093, 0.0..."
4,AdaBoostRegressor,"[-5.95653429602888, -5.994149911492844, -5.630...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.214311111111114, -5.634961484305196, -5.7...","[-6.075948695603364, -5.6336000689935, -5.7853...","[0.18331898583171408, 0.03330731995173463, 0.1..."
5,XGBRegressor,"[-7.007291, -6.4003043, -6.113007, -6.189088, ...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.8437247, -6.4251924, -6.816961, -6.741024...","[-7.027953, -6.5525513, -6.8499618, -6.731916,...","[0.106291115, 0.5147628, 0.26119104, 0.3477811..."
6,ExtraTreesRegressor,"[-6.773400000000001, -6.9072, -6.1864000000000...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.983199999999999, -6.380300000000001, -6.8...","[-6.97216, -6.357420000000002, -6.797579999999...","[0.019817729436037882, 0.1894796917878004, 0.1..."
7,LinearRegression,"[-3.9, -10.0, -8.396586828531781, -7.280538717...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-3.9, -3.9, -7.98957892635093, -10.0, -7.654...","[-5.119999999999999, -5.951092739854916, -8.37...","[2.44, 2.5160897732780354, 2.3704854732731055,..."
8,KNeighborsRegressor,"[-6.63, -6.986666666666667, -5.88, -4.85333333...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.63, -6.503333333333333, -6.49333333333333...","[-6.7780000000000005, -6.552, -6.5206666666666...","[0.18126224096595522, 0.09733333333333362, 0.0..."
9,SVR,"[-6.381256413626859, -6.860513250567857, -5.75...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.077880093822507, -6.703799444410125, -6.1...","[-6.119260992553231, -6.79243689547093, -6.184...","[0.027511202384364775, 0.1655634095302695, 0.0..."


In [95]:
result_df.to_csv('Results/Descriptors/Results_2D_3D_All_desc.csv')
prediction_df.to_csv('Results/Descriptors/Prediction_data_2D_3D_All_desc.csv')

In [97]:
#All 2d and 3d descriptors const rem
df_train = pd.read_csv('features/Descriptors/Train_2d_3d_all_descriptors.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train,  const_col =  remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_3d_all_descriptors.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

/tmp/ipykernel_3612489/2958371523.py:2: DtypeWarning: Columns (1275,1277,1280,1285,1292,1298,1304,1354,1356,1359,1364,1371,1377,1383,1579,1580,1581,1583,1584,1590,1595,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('features/Descriptors/Train_2d_3d_all_descriptors.csv')


X_train shape:  (5568, 2946)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX


/tmp/ipykernel_3612489/2958371523.py:11: DtypeWarning: Columns (1275,1277,1292,1298,1354,1356,1371,1377,1579,1580,1581,1583,1584,1595,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test = pd.read_csv('features/Descriptors/Test_2d_3d_all_descriptors.csv')


X_test shape:  (1392, 2946)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.304848 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 627430
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 2822
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.294200 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 627479
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 2827
[LightGBM] [Info] Start training from score -5.744959
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.292968 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] T

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2118,0.3398,0.4602,0.6598,0.8134,0.7783,0.2116,0.3336,0.4600,0.6672,0.8186,0.7973
DecisionTreeRegressor,0.4615,0.4848,0.6794,0.2587,0.6402,0.6157,0.2732,0.3751,0.5227,0.5702,0.7605,0.7371
RandomForestRegressor,0.2177,0.3457,0.4666,0.6504,0.8074,0.7738,0.2203,0.3401,0.4694,0.6534,0.8095,0.7889
GradientBoostingRegressor,0.2235,0.3516,0.4727,0.6410,0.8030,0.7635,0.2230,0.3446,0.4722,0.6492,0.8096,0.7804
AdaBoostRegressor,0.3792,0.4995,0.6158,0.3910,0.6605,0.6008,0.3635,0.4832,0.6029,0.4282,0.6953,0.6667
XGBRegressor,0.2405,0.3604,0.4904,0.6138,0.7845,0.7516,0.2172,0.3362,0.4661,0.6583,0.8118,0.7893
ExtraTreesRegressor,0.2148,0.3410,0.4634,0.6551,0.8094,0.7777,0.2173,0.3349,0.4662,0.6582,0.8113,0.7925
LinearRegression,1.3376,0.7012,1.1565,-1.1484,0.3955,0.5102,0.6698,0.5304,0.8184,-0.0536,0.5335,0.6106
KNeighborsRegressor,0.2727,0.3812,0.5222,0.5621,0.7579,0.7256,0.2750,0.3730,0.5244,0.5674,0.7587,0.7518
SVR,0.2294,0.3444,0.4790,0.6315,0.7979,0.7692,0.2384,0.3487,0.4882,0.6251,0.7949,0.7737


In [98]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.636653545675242, -6.681005805477045, -6.13...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.960273881772997, -6.090355191773246, -6.8...","[-6.924825786001401, -6.42079683232817, -6.813...","[0.09496275198100144, 0.23801194391904637, 0.0..."
1,DecisionTreeRegressor,"[-7.0, -6.64, -5.28, -4.57, -5.07, -4.77, -4.2...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -6.09, -5.28, -7.13, -4.72, -7.0, -6.8...","[-6.708, -6.382, -6.587999999999999, -6.728, -...","[0.584, 0.8568407086500968, 0.99048271060125, ..."
2,RandomForestRegressor,"[-6.6453999999999995, -6.737700000000001, -5.7...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.795300000000001, -6.319600000000003, -6.5...","[-6.756380000000002, -6.453800000000001, -6.69...","[0.07007371547163753, 0.08721282015850663, 0.1..."
3,GradientBoostingRegressor,"[-6.934528344099418, -6.988494123637157, -5.53...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.089487172287799, -6.091214853366252, -6.7...","[-7.121258475272957, -6.3814402560647085, -6.7...","[0.0736561532115871, 0.1874576737850576, 0.060..."
4,AdaBoostRegressor,"[-5.894430951532188, -6.148989898989898, -5.59...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.033486339700903, -5.630170941288475, -5.8...","[-5.91602262984866, -5.597694021580338, -5.853...","[0.13508624560093724, 0.02999152655256714, 0.2..."
5,XGBRegressor,"[-7.007291, -6.4003043, -6.113007, -6.189088, ...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.8437247, -6.4251924, -6.816961, -6.741024...","[-7.027953, -6.5525513, -6.8499618, -6.731916,...","[0.106291115, 0.5147628, 0.26119104, 0.3477811..."
6,ExtraTreesRegressor,"[-6.7473, -6.8646, -6.111100000000001, -5.3157...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.9781, -6.256, -6.8770999999999995, -6.602...","[-6.967619999999999, -6.318216857866, -6.81933...","[0.013722011514351965, 0.16328557048888379, 0...."
7,LinearRegression,"[-3.9, -10.0, -8.396587299326768, -7.280539011...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-3.9, -3.9, -7.989581360863781, -10.0, -7.65...","[-5.119999999999999, -5.951093410735997, -8.37...","[2.44, 2.5160904320504587, 2.370485393508118, ..."
8,KNeighborsRegressor,"[-6.63, -6.986666666666667, -5.88, -4.85333333...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.63, -6.503333333333333, -6.49333333333333...","[-6.7780000000000005, -6.552, -6.5206666666666...","[0.18126224096595522, 0.09733333333333362, 0.0..."
9,SVR,"[-6.3813530796421745, -6.86069060193266, -5.75...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.077909529153203, -6.70380712097384, -6.14...","[-6.119235546264135, -6.792318236398853, -6.18...","[0.027494313000103198, 0.1655075810603596, 0.0..."


In [99]:
result_df.to_csv('Results/Descriptors/Results_2D_3D_All_desc_const_rem.csv')
prediction_df.to_csv('Results/Descriptors/Prediction_data_2D_3D_All_desc_const_rem.csv')

In [100]:
#All 2d and 3d descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_2d_3d_all_descriptors.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train,  const_col =  remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_3d_all_descriptors.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

/tmp/ipykernel_3612489/212632653.py:2: DtypeWarning: Columns (1275,1277,1280,1285,1292,1298,1304,1354,1356,1359,1364,1371,1377,1383,1579,1580,1581,1583,1584,1590,1595,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('features/Descriptors/Train_2d_3d_all_descriptors.csv')


X_train shape:  (5568, 2087)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX


/tmp/ipykernel_3612489/212632653.py:11: DtypeWarning: Columns (1275,1277,1292,1298,1354,1356,1371,1377,1579,1580,1581,1583,1584,1595,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test = pd.read_csv('features/Descriptors/Test_2d_3d_all_descriptors.csv')


X_test shape:  (1392, 2087)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.226460 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 438192
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 2059
[LightGBM] [Info] Start training from score -5.741402
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.229270 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 438246
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 2064
[LightGBM] [Info] Start training from score -5.744959
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.227605 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] T

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2155,0.3431,0.4642,0.6538,0.8094,0.7736,0.2155,0.3375,0.4642,0.6611,0.8147,0.7931
DecisionTreeRegressor,0.4447,0.4756,0.6668,0.2858,0.6499,0.6266,0.2713,0.3765,0.5209,0.5733,0.7610,0.7376
RandomForestRegressor,0.2204,0.3467,0.4694,0.6460,0.8046,0.7708,0.2207,0.3410,0.4698,0.6528,0.8091,0.7877
GradientBoostingRegressor,0.2233,0.3517,0.4725,0.6414,0.8033,0.7618,0.2261,0.3465,0.4756,0.6443,0.8064,0.7797
AdaBoostRegressor,0.3800,0.5000,0.6165,0.3896,0.6581,0.5913,0.3612,0.4824,0.6010,0.4319,0.6995,0.6625
XGBRegressor,0.2423,0.3643,0.4923,0.6108,0.7829,0.7493,0.2201,0.3399,0.4692,0.6537,0.8087,0.7828
ExtraTreesRegressor,0.2103,0.3392,0.4586,0.6622,0.8139,0.7818,0.2170,0.3360,0.4659,0.6586,0.8116,0.7927
LinearRegression,0.6341,0.4920,0.7963,-0.0184,0.5836,0.6615,0.4556,0.4351,0.6750,0.2834,0.6484,0.7086
KNeighborsRegressor,0.2804,0.3856,0.5295,0.5497,0.7503,0.7143,0.2837,0.3774,0.5326,0.5538,0.7503,0.7494
SVR,0.2386,0.3514,0.4884,0.6168,0.7889,0.7595,0.2458,0.3526,0.4958,0.6133,0.7878,0.7679


In [101]:
result_df.to_csv('Results/Descriptors/Results_2D_3D_All_desc_LVR.csv')
prediction_df.to_csv('Results/Descriptors/Prediction_data_2D_3D_All_desc_LVR.csv')

In [102]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.552857969738684, -6.8362647600530355, -5.8...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.889475710721171, -6.085882358780136, -7.0...","[-6.840696286122535, -6.3872936975972, -6.8381...","[0.16766901543006266, 0.20404468050556154, 0.1..."
1,DecisionTreeRegressor,"[-7.0, -6.6, -5.85, -4.57, -5.07, -4.77, -4.57...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -5.8, -5.54, -7.0, -4.74, -7.0, -7.0, ...","[-6.5, -6.608, -6.21, -6.5120000000000005, -5....","[0.582443130271102, 0.4998559792580259, 0.9496..."
2,RandomForestRegressor,"[-6.6451, -6.722499999999999, -5.7527999999999...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.783700000000001, -6.315900000000002, -6.6...","[-6.748800000000001, -6.493480000000001, -6.75...","[0.08533927583475247, 0.1251978657965051, 0.10..."
3,GradientBoostingRegressor,"[-6.697241870799273, -6.971083305067908, -5.64...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.857444583434824, -5.87250977751348, -6.72...","[-6.99664451103317, -6.26644871195122, -6.7871...","[0.11372666310446232, 0.219092411848478, 0.069..."
4,AdaBoostRegressor,"[-5.806066294550562, -5.849775545175726, -5.60...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-5.944678462793358, -5.5425140866271025, -5....","[-6.031907442236947, -5.616093204155526, -5.84...","[0.12523408123346289, 0.058952310923716444, 0...."
5,XGBRegressor,"[-6.5451407, -6.582066, -6.192833, -5.588872, ...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.1722603, -6.439779, -6.903651, -6.5879493...","[-6.8322706, -6.478139, -6.835491, -6.7522545,...","[0.3099406, 0.21446864, 0.1701113, 0.23630993,..."
6,ExtraTreesRegressor,"[-6.796900000000002, -6.9464999999999995, -6.0...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.959200000000001, -6.303800000000003, -6.9...","[-6.951940000000002, -6.315520000000001, -6.82...","[0.02606956846593384, 0.08109005857686798, 0.1..."
7,LinearRegression,"[-3.9, -7.666371694390051, -6.853749878676865,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-3.9, -5.765705259110035, -10.0, -6.88408269...","[-3.9, -6.384626776662566, -9.24362414223993, ...","[0.0, 0.4781208465793842, 1.0878253867439651, ..."
8,KNeighborsRegressor,"[-6.63, -6.986666666666667, -5.88, -4.85333333...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.63, -6.503333333333333, -6.49333333333333...","[-6.7780000000000005, -6.552, -6.5946666666666...","[0.18126224096595522, 0.09733333333333362, 0.1..."
9,SVR,"[-6.341097575662663, -6.893473191379597, -5.80...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.153968461460335, -6.740162928791563, -6.1...","[-6.171512829522047, -6.800461223103028, -6.18...","[0.021429582801276457, 0.17408879848052683, 0...."


In [4]:
from sklearn.model_selection import GridSearchCV
import os
import joblib
def train_and_test_predict_with_tuning(models, param_grids, X_train, y_train, X_test, y_test, save_dir):
   
    kf = KFold(n_splits=5, shuffle=True, random_state=101)
    results = {}
    predictions = []  

    for model in models:
        model_name = model.__class__.__name__
        predictions_train = []
        actual_y_train = []
        test_predictions_folds = []

        best_params = None

        # hyperparameter tuning 
        if model_name in param_grids and param_grids[model_name]:
            default_params = model.get_params()
            print(model_name, ': Default params', default_params)
            grid_search = GridSearchCV(
                estimator=model, 
                param_grid=param_grids[model_name], 
                cv=kf,
                scoring='neg_mean_squared_error', 
                n_jobs=-1)
            grid_search.fit(X_train, y_train)
            model = grid_search.best_estimator_
            best_params = grid_search.best_params_
            print(model_name)
            print(": best params",best_params)
        else:
            default_params = model.get_params()
            print(model_name, ': Default params', default_params)
            best_params = {}
            print(model_name, ':Used Default params')

        for train_index, val_index in kf.split(X_train):
            X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
            y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

            model.fit(X_train_fold, y_train_fold)

            y_pred_fold = model.predict(X_val_fold)
            predictions_train.extend(y_pred_fold)
            actual_y_train.extend(y_val_fold)

            predictions_test_fold = model.predict(X_test)
            predictions_test_fold = np.clip(predictions_test_fold, -10, -3.9)  
            test_predictions_folds.append(predictions_test_fold)

        mse_train = mean_squared_error(actual_y_train, predictions_train)
        mae_train = mean_absolute_error(actual_y_train, predictions_train)
        rmse_train = np.sqrt(mse_train)
        r2_train = r2_score(actual_y_train, predictions_train)
        pearson_train, _ = pearsonr(actual_y_train, predictions_train)
        spearman_train, _ = spearmanr(actual_y_train, predictions_train)

        predictions_test_mean = np.mean(test_predictions_folds, axis=0)
        predictions_test_std = np.std(test_predictions_folds, axis=0)

        mse_test = mean_squared_error(y_test, predictions_test_mean)
        mae_test = mean_absolute_error(y_test, predictions_test_mean)
        rmse_test = np.sqrt(mse_test)
        r2_test = r2_score(y_test, predictions_test_mean)
        print(r2_test)
        pearson_test, _ = pearsonr(y_test, predictions_test_mean)
        spearman_test, _ = spearmanr(y_test, predictions_test_mean)

        predictions.append({
            'Model': model_name,
            'Y Train pred': predictions_train,
            'Y Test actual': y_test,
            'Test Predictions Mean': predictions_test_mean,
            'Test Predictions Std': predictions_test_std,
            'Best Parameters': best_params
        })

        results[model_name] = {
            'Train MSE (5 fold cv)': f"{mse_train:.4f}",
            'Train MAE (5 fold cv)': f"{mae_train:.4f}",
            'Train RMSE (5 fold cv)': f"{rmse_train:.4f}",
            'Train R2 (5 fold cv)': f"{r2_train:.4f}",
            'Train PCC (5 fold cv)': f"{pearson_train:.4f}",
            'Train SCC (5 fold cv)': f"{spearman_train:.4f}",
            'Test MSE': f"{mse_test:.4f}",
            'Test MAE': f"{mae_test:.4f}",
            'Test RMSE': f"{rmse_test:.4f}",
            'Test R2': f"{r2_test:.4f}",
            'Test Pearson Correlation': f"{pearson_test:.4f}",
            'Test Spearman Correlation': f"{spearman_test:.4f}",
        }
        # Save the model
        model_path = os.path.join(save_dir, f"{model_name}.joblib")
        joblib.dump(model, model_path)
        print(f"Saved {model_name} model to {model_path}")

    results_df = pd.DataFrame(results).T
    predictions_df = pd.DataFrame(predictions)

    return results_df, predictions_df


In [ ]:
param_grids = {
        'ExtraTreesRegressor': {
            'n_estimators': [50, 100, 200, 400],
            'max_depth': [None,1,5, 10, 20],
            'min_samples_split': [2, 5, 10]
        },
        'LGBMRegressor': {
            'n_estimators': [50, 100, 200, 400],
            'learning_rate': [0.001, 0.01, 0.05, 0.1],
            'num_leaves': [31, 50, 100]
        },
        'DecisionTreeRegressor': {
            'max_depth': [None, 10, 20, 50, 100],
            'min_samples_split': [2, 5, 10]
        },
        'RandomForestRegressor': {
            'n_estimators': [50, 100, 200, 400],
            'max_depth': [None, 1, 5, 10, 20],
            'min_samples_split': [2, 5, 10]
        },
        'GradientBoostingRegressor': {
            'n_estimators': [50, 100, 200, 400],
            'learning_rate': [0.001, 0.01, 0.05, 0.1],
            'max_depth': [3, 5, 7, 10]
        },
        'AdaBoostRegressor': {
            'n_estimators': [50, 100, 200, 400],
            'learning_rate': [0.001, 0.01, 0.1, 1.0]
        },
        'SVR': {
            'C': [0.001, 0.1, 1, 10],
            'epsilon': [0.1, 0.2, 0.5],
            'gamma': [0.001, 0.1, 1, 10]
        },
        'KNeighborsRegressor': {
            'n_neighbors': [3, 5, 10],
            'weights': ['uniform', 'distance']
        },
        'MLPRegressor': {
            'hidden_layer_sizes': [(50,), (100,), (50, 50)],
            'learning_rate': ['constant', 'adaptive'],
            'max_iter': [100,200, 400, 500]
}
    }


In [7]:
#2d Mordred descriptors const removal
df_train = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train, const_col = remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = X_test.select_dtypes(include=['number'])
X_test = X_test.drop(const_col,axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)
models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
save_dir = 'Results/Descriptors/Models_2d_Mordred_const_rem_with_HPT/'
os.makedirs(save_dir, exist_ok=True)
result_df, prediction_df = train_and_test_predict_with_tuning(models,param_grids, X_train,y_train, X_test,  y_test, save_dir)
result_df

C:\Users\aksha\AppData\Local\Temp\ipykernel_10956\1153071293.py:2: DtypeWarning: Columns (1058,1060,1075,1081,1137,1139,1154,1160,1362,1363,1364,1366,1367,1378,1379,1380) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('Descriptors/Train_2d_Mordred_desc.csv')


X_train shape:  (5568, 1227)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX


C:\Users\aksha\AppData\Local\Temp\ipykernel_10956\1153071293.py:10: DtypeWarning: Columns (1058,1060,1081,1137,1139,1160,1362,1363,1364,1366,1367,1380) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test = pd.read_csv('Descriptors/Test_2d_Mordred_desc.csv')


X_test shape:  (1392, 1227)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
LGBMRegressor : Default params {'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 1.0, 'importance_type': 'split', 'learning_rate': 0.05, 'max_depth': -1, 'min_child_samples': 20, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 100, 'n_jobs': None, 'num_leaves': 31, 'objective': 'regression', 'random_state': 101, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'subsample': 1.0, 'subsample_for_bin': 200000, 'subsample_freq': 0, 'metric': 'rmse'}


c:\Python312\Lib\site-packages\sklearn\utils\_tags.py:354: FutureWarning: The LGBMRegressor or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.057735 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 265910
[LightGBM] [Info] Number of data points in the train set: 5568, number of used features: 1196
[LightGBM] [Info] Start training from score -5.742906
LGBMRegressor
: best params {'learning_rate': 0.05, 'n_estimators': 200, 'num_leaves': 31}
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.039435 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 265212
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 1187
[LightGBM] [Info] Start training from score -5.738310
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.048236 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 265242
[LightGBM] [Info] N

KeyboardInterrupt: 

In [ ]:
result_df.to_csv('Results/Descriptors/Results_2d_Mordred_const_rem_with_HPT.csv')
prediction_df.to_csv('Results/Descriptors/Prediction_df_2d_Mordred_const_rem_with_HPT.csv')

In [ ]:
#2d All descriptors const rem
df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train, const_col =  remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_all_descriptors.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
save_dir = 'Results/Descriptors/Models_2D_All_desc_const_rem_with_HPT/'
os.makedirs(save_dir, exist_ok=True)
result_df, prediction_df = train_and_test_predict_with_tuning(models,param_grids, X_train,y_train, X_test,  y_test, save_dir)
result_df

In [ ]:
result_df.to_csv('Results/Descriptors/Results_2D_All_desc_const_rem_with_HPT.csv')
prediction_df.to_csv('Results/Descriptors/Prediction_data_2D_All_desc_const_rem_with_HPT.csv')

In [ ]:
#2d RDKit descriptors const removal
df_train = pd.read_csv('features/Descriptors/Train_2d_RDKit_des.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train, const_col = remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_RDKit_des.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = X_test.drop(const_col,axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
save_dir = 'Results/Descriptors/Models_2d_rdkit_const_rem_with_HPT/'
os.makedirs(save_dir, exist_ok=True)
result_df, prediction_df = train_and_test_predict_with_tuning(models,param_grids, X_train,y_train, X_test,  y_test, save_dir)
result_df

In [ ]:
result_df.to_csv('Results/Descriptors/Results_2d_rdkit_const_rem_with_HPT.csv')
prediction_df.to_csv('Results/Descriptors/Prediction_data_2d_rdkit_const_rem_with_HPT.csv')

In [122]:
#2d Mordred descriptors const removal
df_train = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train, const_col = remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = X_test.select_dtypes(include=['number'])
X_test = X_test.drop(const_col,axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models_2dM = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models_2dM, X_train,y_train, X_test,  y_test)
result_df

C:\Users\aksha\AppData\Local\Temp\ipykernel_32620\728849309.py:2: DtypeWarning: Columns (1058,1060,1075,1081,1137,1139,1154,1160,1362,1363,1364,1366,1367,1378,1379,1380) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('Descriptors/Train_2d_Mordred_desc.csv')


X_train shape:  (5568, 1227)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX


C:\Users\aksha\AppData\Local\Temp\ipykernel_32620\728849309.py:10: DtypeWarning: Columns (1058,1060,1081,1137,1139,1160,1362,1363,1364,1366,1367,1380) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test = pd.read_csv('Descriptors/Test_2d_Mordred_desc.csv')


X_test shape:  (1392, 1227)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.170237 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 265212
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 1187
[LightGBM] [Info] Start training from score -5.738310
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.470950 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 265242
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 1190
[LightGBM] [Info] Start training from score -5.742699
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.203769 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] T

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2063,0.3347,0.4541,0.6688,0.8184,0.7858,0.2054,0.3276,0.4532,0.6754,0.8229,0.8032
DecisionTreeRegressor,0.3081,0.3866,0.5551,0.5053,0.7334,0.7176,0.2332,0.3453,0.4829,0.6315,0.7955,0.7797
RandomForestRegressor,0.2154,0.3436,0.4641,0.6542,0.8092,0.7796,0.2138,0.3351,0.4624,0.6621,0.8144,0.7952
GradientBoostingRegressor,0.2272,0.3549,0.4767,0.6352,0.8002,0.7646,0.2183,0.3425,0.4673,0.6550,0.8140,0.7856
AdaBoostRegressor,0.3832,0.4996,0.6190,0.3847,0.6608,0.6170,0.3681,0.4849,0.6067,0.4183,0.6886,0.6702
XGBRegressor,0.2174,0.3414,0.4662,0.6510,0.8073,0.7742,0.2127,0.3301,0.4612,0.6639,0.8150,0.7943
ExtraTreesRegressor,0.2145,0.3415,0.4632,0.6555,0.8097,0.7819,0.2230,0.3396,0.4722,0.6476,0.8049,0.7867
LinearRegression,0.6512,0.4717,0.8070,-0.0456,0.5858,0.6900,0.3377,0.3822,0.5812,0.4663,0.7177,0.7463
KNeighborsRegressor,0.2717,0.3807,0.5213,0.5637,0.7575,0.7263,0.2642,0.3711,0.5140,0.5824,0.7671,0.7575
SVR,0.2267,0.3421,0.4762,0.6359,0.8007,0.7714,0.2376,0.3499,0.4875,0.6245,0.7947,0.7756


In [123]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.803659819579023, -6.952297461961773, -6.31...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.077881107007449, -6.0194634527933095, -6....","[-7.095000868721698, -6.249811763734205, -6.66...","[0.025405667572675622, 0.18539061636084384, 0...."
1,DecisionTreeRegressor,"[-6.82, -7.0, -6.244999999999999, -5.05, -4.26...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -7.0, -6.96, -7.0, -6.24, -7.0, -6.85,...","[-6.992, -6.43, -6.720000000000001, -6.694, -6...","[0.016000000000000014, 0.6298571266565139, 0.5..."
2,RandomForestRegressor,"[-6.535537681159422, -6.766108333333335, -6.00...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.747766666666669, -6.1027166666666695, -6....","[-6.874323333333335, -6.307488457199335, -6.75...","[0.06899517986384564, 0.1466726414596661, 0.08..."
3,GradientBoostingRegressor,"[-7.172151031819288, -7.023936244369912, -5.93...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0754472513178035, -6.052434368338021, -6....","[-7.154501095613862, -6.281628977846543, -6.57...","[0.12123129267795736, 0.21266660343343977, 0.0..."
4,AdaBoostRegressor,"[-6.092088465441948, -6.055812499999991, -5.29...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.1702678571428535, -5.566031587411215, -5....","[-6.100103518051506, -5.645258461494352, -5.66...","[0.2048426940735511, 0.07647162008521954, 0.03..."
5,XGBRegressor,"[-6.8409624, -6.8683248, -6.0015554, -5.321909...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.8227315, -6.169962, -6.799593, -6.7047834...","[-7.014267, -6.187599, -6.730255, -6.7727537, ...","[0.116441816, 0.2011095, 0.11745094, 0.1260753..."
6,ExtraTreesRegressor,"[-6.5763000000000025, -6.813800000000001, -6.1...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.964599999999999, -6.627100000000001, -6.8...","[-6.955299999999999, -6.516620000000001, -6.86...","[0.022404374572836814, 0.13256841856188764, 0...."
7,LinearRegression,"[-7.248273384067716, -10.0, -6.148328486140372...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.56729127677113, -6.127714388314189, -8.63...","[-7.073458255354225, -6.3982907369437925, -8.6...","[2.7391545708959977, 0.9518729724276647, 0.050..."
8,KNeighborsRegressor,"[-7.0, -7.0, -5.88, -4.88, -4.733333333333333,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -6.746666666666667, -6.746666666666667...","[-7.0, -6.898666666666666, -6.797333333333334,...","[0.0, 0.12410748030101418, 0.10133333333333318..."
9,SVR,"[-7.169522117681083, -7.019981504744268, -5.91...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.300096341024478, -6.780839401372151, -7.0...","[-7.285910573423253, -6.740319863124401, -7.00...","[0.04465649746211758, 0.05312871296499056, 0.0..."


In [124]:
result_df.to_csv('Results/Descriptors/Results_2d_Mordred_const_rem_scaled.csv')
prediction_df.to_csv('Results/Descriptors/Prediction_df_2d_Mordred_const_rem_scaled.csv')

In [125]:
#2d Mordred descriptors LVR
df_train = pd.read_csv('features/Descriptors/Train_2d_Mordred_desc.csv')
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train, const_col = remove_low_variance_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_Mordred_desc.csv')
X_test = df_test.drop(['ID','SMILES','Permeability'],axis=1)
X_test = X_test.select_dtypes(include=['number'])
X_test = X_test.drop(const_col,axis=1)
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
results_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
results_df

C:\Users\aksha\AppData\Local\Temp\ipykernel_32620\3326927295.py:2: DtypeWarning: Columns (1058,1060,1075,1081,1137,1139,1154,1160,1362,1363,1364,1366,1367,1378,1379,1380) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('Descriptors/Train_2d_Mordred_desc.csv')


X_train shape:  (5568, 821)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX


C:\Users\aksha\AppData\Local\Temp\ipykernel_32620\3326927295.py:10: DtypeWarning: Columns (1058,1060,1081,1137,1139,1160,1362,1363,1364,1366,1367,1380) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test = pd.read_csv('Descriptors/Test_2d_Mordred_desc.csv')


X_test shape:  (1392, 821)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.025547 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 170429
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 808
[LightGBM] [Info] Start training from score -5.738310
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.028619 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 170473
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 811
[LightGBM] [Info] Start training from score -5.742699
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.026368 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Tota

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2086,0.3356,0.4568,0.6650,0.8160,0.7849,0.2074,0.3286,0.4554,0.6722,0.8211,0.8034
DecisionTreeRegressor,0.2915,0.3810,0.5399,0.5319,0.7450,0.7269,0.2351,0.3464,0.4848,0.6285,0.7937,0.7721
RandomForestRegressor,0.2170,0.3442,0.4658,0.6516,0.8075,0.7786,0.2150,0.3359,0.4637,0.6603,0.8131,0.7948
GradientBoostingRegressor,0.2301,0.3559,0.4797,0.6305,0.7971,0.7617,0.2240,0.3480,0.4732,0.6461,0.8081,0.7787
AdaBoostRegressor,0.3914,0.5039,0.6256,0.3716,0.6477,0.6129,0.3709,0.4856,0.6090,0.4139,0.6820,0.6686
XGBRegressor,0.2180,0.3432,0.4669,0.6500,0.8072,0.7747,0.2122,0.3298,0.4606,0.6647,0.8154,0.7930
ExtraTreesRegressor,0.2112,0.3394,0.4595,0.6609,0.8130,0.7840,0.2214,0.3374,0.4705,0.6502,0.8065,0.7898
LinearRegression,0.4254,0.4071,0.6522,0.3170,0.6722,0.7356,0.3181,0.3735,0.5640,0.4973,0.7294,0.7629
KNeighborsRegressor,0.2777,0.3834,0.5270,0.5541,0.7513,0.7180,0.2656,0.3714,0.5154,0.5802,0.7645,0.7563
SVR,0.2365,0.3482,0.4863,0.6203,0.7907,0.7618,0.2444,0.3529,0.4944,0.6138,0.7878,0.7692


In [126]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.590723796184461, -6.864465493660207, -6.30...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.1934338278999785, -6.071564767852124, -6....","[-7.104324347085287, -6.313665025245345, -6.70...","[0.11911803727514793, 0.17757965983696158, 0.0..."
1,DecisionTreeRegressor,"[-7.0, -7.0, -6.03, -5.2, -6.244999999999999, ...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -6.24, -7.0, -5.89, -5.949999999999999...","[-7.37, -6.49, -6.728, -6.868, -5.854000000000...","[0.7601578783384412, 0.6307455905513726, 0.524..."
2,RandomForestRegressor,"[-6.683116057605002, -6.8004, -6.1641083333333...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.867600000000001, -6.257900000000002, -6.7...","[-6.884765, -6.37418474261543, -6.71476, -6.54...","[0.040921154675790536, 0.12665785981964658, 0...."
3,GradientBoostingRegressor,"[-6.9538278086749825, -7.052640720542491, -5.8...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.229801769686274, -6.1517648848612785, -6....","[-7.191434474201043, -6.310766037726277, -6.70...","[0.11328046235770686, 0.19618053637325925, 0.0..."
4,AdaBoostRegressor,"[-6.308654545454544, -6.308654545454544, -5.55...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.239743589743589, -5.672502385440382, -5.7...","[-6.060696597590882, -5.708925619905679, -5.66...","[0.1503225840297493, 0.057164129004639755, 0.0..."
5,XGBRegressor,"[-7.2010727, -7.05254, -6.1245418, -4.817527, ...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.3523107, -6.408034, -7.0629597, -6.966821...","[-7.18943, -6.34603, -6.8971586, -6.7855177, -...","[0.1483452, 0.2780104, 0.15064599, 0.15745322,..."
6,ExtraTreesRegressor,"[-6.860600000000002, -6.872300000000001, -6.09...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.9811000000000005, -6.597400000000001, -6....","[-6.970460000000001, -6.585390000000001, -6.91...","[0.006783391482141547, 0.08001171414236755, 0...."
7,LinearRegression,"[-7.539499947810782, -7.185399541640891, -6.23...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.847063878560675, -7.483975315476073, -7.5...","[-7.553554799793129, -7.109674921969928, -7.88...","[1.4875345840168233, 0.6016532257184658, 0.296..."
8,KNeighborsRegressor,"[-7.0, -7.0, -5.88, -4.853333333333333, -4.733...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -6.746666666666667, -6.746666666666667...","[-7.0, -6.898666666666666, -6.797333333333334,...","[0.0, 0.12410748030101418, 0.10133333333333318..."
9,SVR,"[-7.1115116900074185, -7.009329529671163, -5.9...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.118067335275781, -6.790359365497927, -7.0...","[-7.115816315634726, -6.767185642475207, -7.01...","[0.03800995531330861, 0.03985679228380903, 0.0..."


In [127]:
result_df.to_csv('Results/Descriptors/Results_2d_Mordred_LVR_scaled.csv')
prediction_df.to_csv('Results/Descriptors/Prediction_df_2d_Mordred_LVR_scaled.csv')

In [6]:
#2d All descriptors const rem
df_train = pd.read_csv('features/Descriptors/Train_2d_all_descriptors.csv')
df_train = df_train.dropna()
X_train = df_train.drop(['ID','SMILES','Permeability'],axis=1)
X_train = X_train.select_dtypes(include=['number'])
X_train, const_col =  remove_constant_columns(X_train)
y_train = df_train['Permeability']
print("X_train shape: ",X_train.shape)
print("y_train shape: ",y_train.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
df_test = pd.read_csv('features/Descriptors/Test_2d_all_descriptors.csv')
df_test = df_test.dropna()
X_test = df_test[X_train.columns]
y_test = df_test['Permeability']
print("X_test shape: ",X_test.shape)
print("y_test shape: ",y_test.shape)
print("XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

models = [
    lgb.LGBMRegressor(objective='regression',metric='rmse',boosting_type='gbdt',num_leaves=31,learning_rate=0.05,random_state=101),
    DecisionTreeRegressor(random_state=101),
    RandomForestRegressor(n_jobs=-1, random_state=101),
    GradientBoostingRegressor(random_state=101),
    AdaBoostRegressor(random_state=101),
    xgb.XGBRegressor(random_state=101),
    ExtraTreesRegressor(n_jobs=-1, n_estimators=100, random_state=101),
    LinearRegression(), 
    KNeighborsRegressor(n_neighbors=3),
    SVR(),  
    MLPRegressor(random_state=101)
]
result_df, prediction_df = train_and_test_predict(models, X_train,y_train, X_test,  y_test)
result_df

C:\Users\aksha\AppData\Local\Temp\ipykernel_19600\3868225602.py:2: DtypeWarning: Columns (1275,1277,1280,1285,1292,1298,1304,1354,1356,1359,1364,1371,1377,1383,1579,1580,1581,1583,1584,1590,1595,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv('Descriptors/Train_2d_all_descriptors.csv')


X_train shape:  (5568, 2504)
y_train shape:  (5568,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX


C:\Users\aksha\AppData\Local\Temp\ipykernel_19600\3868225602.py:11: DtypeWarning: Columns (1275,1277,1292,1298,1354,1356,1371,1377,1579,1580,1581,1583,1584,1595,1596,1597) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test = pd.read_csv('Descriptors/Test_2d_all_descriptors.csv')


X_test shape:  (1392, 2504)
y_test shape:  (1392,)
XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.102172 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 514723
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 2380
[LightGBM] [Info] Start training from score -5.738310
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.084353 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 514774
[LightGBM] [Info] Number of data points in the train set: 4454, number of used features: 2385
[LightGBM] [Info] Start training from score -5.742699
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.104829 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] T

,Train MSE (5 fold cv),Train MAE (5 fold cv),Train RMSE (5 fold cv),Train R2 (5 fold cv),Train PCC (5 fold cv),Train SCC (5 fold cv),Test MSE,Test MAE,Test RMSE,Test R2,Test Pearson Correlation,Test Spearman Correlation
LGBMRegressor,0.2042,0.3325,0.4519,0.6721,0.8203,0.7881,0.2058,0.3277,0.4537,0.6748,0.8222,0.8044
DecisionTreeRegressor,0.3059,0.3856,0.5531,0.5088,0.7329,0.7127,0.2314,0.3444,0.4810,0.6344,0.7976,0.7745
RandomForestRegressor,0.2155,0.3434,0.4643,0.6539,0.8088,0.7760,0.2196,0.3395,0.4686,0.6530,0.8085,0.7895
GradientBoostingRegressor,0.2203,0.3491,0.4693,0.6463,0.8065,0.7672,0.2187,0.3408,0.4676,0.6545,0.8125,0.7834
AdaBoostRegressor,0.3778,0.4938,0.6146,0.3934,0.6626,0.6219,0.3575,0.4771,0.5979,0.4351,0.6957,0.6761
XGBRegressor,0.2166,0.3429,0.4654,0.6522,0.8080,0.7748,0.2151,0.3318,0.4638,0.6601,0.8127,0.7923
ExtraTreesRegressor,0.2092,0.3381,0.4573,0.6642,0.8150,0.7840,0.2180,0.3370,0.4669,0.6555,0.8097,0.7900
LinearRegression,1.1424,0.6212,1.0688,-0.8343,0.4362,0.5776,0.5253,0.4695,0.7248,0.1699,0.6108,0.6739
KNeighborsRegressor,0.2736,0.3818,0.5231,0.5607,0.7550,0.7190,0.2703,0.3757,0.5199,0.5728,0.7603,0.7484
SVR,0.2240,0.3392,0.4733,0.6403,0.8032,0.7743,0.2344,0.3451,0.4841,0.6296,0.7973,0.7768


In [7]:
prediction_df

,Model,Y Train pred,Y Test actual,Test prediction folds,Test Predictions Mean,Test Predictions Std
0,LGBMRegressor,"[-6.66923493793545, -6.964947952869519, -6.187...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.1131123303759125, -6.160156349224908, -6....","[-7.018860431317639, -6.49195061784413, -6.855...","[0.07600842296393379, 0.18118606790177316, 0.0..."
1,DecisionTreeRegressor,"[-7.0, -6.96, -7.0, -5.92, -6.244999999999999,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.0, -7.0, -7.0, -7.0, -4.43, -5.8, -6.85, ...","[-6.748, -6.992, -6.5760000000000005, -6.97000...","[0.5039999999999999, 0.016000000000000014, 0.5..."
2,RandomForestRegressor,"[-6.6145000000000005, -6.813439999999999, -5.9...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.801800000000001, -6.244875, -6.6953999999...","[-6.815437, -6.483283139319333, -6.69159480598...","[0.04660011669513284, 0.13993377441419044, 0.0..."
3,GradientBoostingRegressor,"[-6.981264206991823, -6.939825737406756, -5.66...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.073263222609385, -6.063939303715299, -7.1...","[-7.073427899473096, -6.26005136233825, -6.775...","[0.08591276917451098, 0.19377985690008964, 0.1..."
4,AdaBoostRegressor,"[-5.885562913907283, -6.377272837070318, -5.60...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.304420600858371, -5.684994355584168, -5.8...","[-6.114998915502312, -5.697759913121283, -5.71...","[0.16125623562845817, 0.07888993482493699, 0.0..."
5,XGBRegressor,"[-7.2001286, -6.737824, -5.4927177, -4.932216,...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-7.2036347, -6.6719604, -7.09824, -6.8739405...","[-7.0979676, -6.543934, -6.892276, -6.6681466,...","[0.14426151, 0.29652217, 0.1287557, 0.20234475..."
6,ExtraTreesRegressor,"[-6.780100000000001, -6.956799999999999, -6.22...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.956600000000002, -6.239750000000002, -6.9...","[-6.958940000000001, -6.319060000000001, -6.93...","[0.018105093206056567, 0.12052718116673893, 0...."
7,LinearRegression,"[-3.9, -10.0, -6.561595810264407, -6.924811289...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-3.9, -7.27295671930915, -10.0, -5.678827745...","[-5.119999999999999, -6.699755758944614, -9.12...","[2.44, 2.350504710954084, 0.7756174763588719, ..."
8,KNeighborsRegressor,"[-6.63, -6.986666666666667, -5.88, -4.85333333...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.63, -6.746666666666667, -7.0, -5.90333333...","[-6.7780000000000005, -6.698, -6.9120000000000...","[0.18126224096595522, 0.09733333333333362, 0.1..."
9,SVR,"[-6.314907965426872, -6.951309678541952, -5.81...",0 -7.00 1 -7.00 2 -7.00 3 ...,"[[-6.035070885358877, -6.667518105491782, -7.0...","[-6.094828523113399, -6.742352408572694, -7.04...","[0.03399947448581303, 0.13083693271290386, 0.0..."


In [8]:
result_df.to_csv('Results/Descriptors/Results_2d_all_desc_const_col_scaled.csv')
prediction_df.to_csv('Results/Descriptors/Prediction_df_2d_all_desc_const_col_scaled.csv')